# FastAPI Notes 3: Production, Projects and Interviews

Welcome to the last notebook. In Notebooks 1 and 2 you learned how to *build* FastAPI applications.
This notebook is about everything that happens **after** the code works on your laptop:
running it in production, putting it in Docker, making it fast, watching it with logs and metrics,
designing clean APIs, serving ML and LLM models, two complete portfolio projects, and a large
interview section with questions, coding tasks, debugging scenarios and a mock interview.

This is **Notebook 3 of 3**:

| Notebook | Topic | Level |
|---|---|---|
| Notebook 1 | FastAPI Fundamentals | Beginner |
| Notebook 2 | Advanced FastAPI | Intermediate to advanced |
| **Notebook 3 (this one)** | Production, Projects and Interviews | Job-ready |

**Before you start:** finish Notebooks 1 and 2. This notebook uses, without re-teaching:
TestClient and path/query/body parameters (Notebook 1), dependency injection (Notebook 2, Part 1),
settings (Notebook 2, Part 2), async (Notebook 2, Part 3), SQLAlchemy (Notebook 2, Part 4),
security and JWT (Notebook 2, Part 5), middleware and lifespan (Notebook 2, Part 6),
streaming and SSE (Notebook 2, Part 7) and testing (Notebook 2, Part 8).

## How to use this notebook

* Parts 1-6 are **production topics**. Read a section, run its cells, then do its **Practice** block before you look at the solutions.
* Parts 7-8 are **projects**. Read the code slowly, run it, then try to rebuild it yourself in a real folder
  (the "Project layout" section of each project shows the files you would create).
* Parts 9-11 are **interview preparation**. Say the answers out loud. Write the coding tasks yourself first.
* Part 12 is a **cheat sheet** for the night before an interview.
* Each Part imports what it needs. Only the setup cell below (the `input()` helper) is shared.
* Every Part ends with **How to debug** and a **Summary**.

> **Note:** a few cells write files into a folder called `fastapi_lab_3/` next to this notebook
> (a small app file, SQLite databases). The last cell of the notebook deletes it. Nothing is written anywhere else.

Labels used, as in Notebooks 1 and 2: **Note**, **Tip**, **Warning**, **Common mistake**, **Interview tip**, and **[Hard]**
for sections you can skip on a first reading.

## Table of contents of Notebook 3

| Part | Title | You will be able to |
|---|---|---|
| 1 | Running FastAPI in Production | choose Uvicorn/Gunicorn options and worker counts, put Nginx in front, add health checks, shut down gracefully, configure with environment variables |
| 2 | Docker for FastAPI | write a small, safe, cache-friendly Dockerfile and a docker-compose file with Postgres |
| 3 | Performance | find where time goes, avoid blocking the event loop, fix N+1 queries, paginate, compress, cache, load test |
| 4 | Observability | write JSON logs with request IDs, expose metrics, understand tracing (and FastAPI's built-in OpenTelemetry support), decide what to alert on |
| 5 | API Design Best Practices | name resources, pick methods and status codes, version, paginate, filter, use idempotency keys and RFC 9457 errors |
| 6 | Serving ML and LLM Models | load models once, batch predictions, stream LLM tokens with SSE, call LLM APIs with timeouts and retries, design a RAG API |
| 7 | Project 1: Task Manager API | build a production-style API with users, JWT, ownership rules, pagination, errors, request IDs and a full test suite |
| 8 | Project 2: AI/LLM Microservice | build a chat service with SSE streaming, caching, rate limiting, API keys, timeouts and tests |
| 9 | Interview Questions and Answers | answer about 100 FastAPI interview questions in clear spoken English |
| 10 | Coding Round Questions | solve 18 common "write this endpoint" tasks with tested solutions |
| 11 | Debugging Scenarios and a Mock Interview | diagnose 10 realistic production bugs and run a 45-minute mock interview |
| 12 | Cheat Sheet and Final Checklist | revise imports, patterns, status codes and a production checklist on two screens |

## Setup: sample answers for `input()`

Some programs in this notebook ask *you* for something with `input()`: the console clients of the two projects,
a small CLI for the ML model, a quiz, and a few practice questions.

Remember: **FastAPI server code never calls `input()`**. A server gets its data from requests (path, query, body,
headers). `input()` belongs in *client* programs, the ones a person types into.

"Run All" cannot stop and wait for typing. So the next cell defines `use_sample_input(...)`. When a cell calls
`use_sample_input("Asha", "25")` before its code, the next two `input()` calls return `"Asha"` and `"25"`
and print them as if you typed them. To type your own answers, skip this cell (or call `use_real_input()`).

In [1]:
# Sample answers for input(), so "Run All" works without typing.
# To type your own answers instead, do not run this cell (or restart the kernel and skip it).
import builtins
from collections import deque

_real_input = builtins.input

def use_sample_input(*answers):
    """Make the next input() calls return these answers, and print them as if they were typed."""
    queue = deque(str(a) for a in answers)
    def sample_input(prompt=""):
        answer = queue.popleft() if queue else ""
        print(f"{prompt}{answer}")
        return answer
    builtins.input = sample_input

def use_real_input():
    builtins.input = _real_input

In [2]:
# Versions used to run this notebook
import sys, fastapi, starlette, pydantic, uvicorn, httpx, sqlalchemy

print("Python   ", sys.version.split()[0])     # 3.14.x
print("FastAPI  ", fastapi.__version__)         # 0.162.0
print("Starlette", starlette.__version__)       # 1.10.0
print("Pydantic ", pydantic.__version__)        # 2.14.2
print("Uvicorn  ", uvicorn.__version__)         # 0.54.0
print("httpx    ", httpx.__version__)           # 0.28.1
print("SQLAlchemy", sqlalchemy.__version__)     # 2.1.6

Python    3.14.6
FastAPI   0.162.0
Starlette 1.10.0
Pydantic  2.14.2
Uvicorn   0.54.0
httpx     0.28.1
SQLAlchemy 2.1.6


> **Note:** the notes use the current style: `Annotated[...]` parameters, the `lifespan` context manager,
> Pydantic v2 and SQLAlchemy 2.0. Features that need a recent FastAPI are marked, for example "(FastAPI 0.162+)".
> Some facts in this notebook differ from older tutorials, and those places say so.

# Part 1: Running FastAPI in Production

**What this Part covers**

* What changes between "it runs on my laptop" and "it runs in production".
* Uvicorn and `fastapi run` options: host, port, workers, `--proxy-headers`, `--forwarded-allow-ips`.
* Gunicorn with Uvicorn workers, and how many workers to run (and why).
* A reverse proxy (Nginx) and HTTPS.
* Liveness vs readiness health checks, and graceful shutdown.
* 12-factor configuration with environment variables.
* One real-server demo: start Uvicorn from Python, call it with httpx, stop it.

**Before you start:** Notebook 1, Part 1 (running `fastapi dev`), Notebook 2, Part 2 (settings) and
Notebook 2, Part 6 (lifespan and middleware).

## 1.1 The big picture: from laptop to production

On your laptop you run `fastapi dev main.py`. One process, auto-reload, only reachable from your machine.
In production many things change:

* The app must be reachable by other computers, usually through **HTTPS** (encrypted HTTP).
* It must survive crashes and restarts, and use more than one CPU core.
* It must tell the platform whether it is alive and ready, and stop cleanly when asked.
* Its configuration (database URL, secrets) comes from the environment, not from the code.

A typical setup looks like this:

```text
   Browser / mobile app / other service
                 |
                 |  HTTPS (port 443)
                 v
   +----------------------------------+
   | Reverse proxy / load balancer    |   Nginx, Traefik, a cloud load balancer,
   | - ends TLS (HTTPS -> HTTP)       |   or a Kubernetes Ingress
   | - adds X-Forwarded-* headers     |
   +----------------+-----------------+
                    |  plain HTTP (private network)
        +-----------+-----------+
        v                       v
   +-----------+           +-----------+
   | Uvicorn   |           | Uvicorn   |    one or more server processes
   | worker 1  |           | worker 2  |    (or containers / pods)
   | your app  |           | your app  |
   +-----+-----+           +-----+-----+
         |                       |
         +-----------+-----------+
                     v
          Database, cache, LLM APIs ...
```

New words:

* **Process:** one running program with its own memory. Two processes do not share Python variables.
* **Worker:** one server process that runs a copy of your app and handles requests.
* **Reverse proxy:** a server that receives requests from the internet and forwards them to your app.
* **TLS:** the encryption behind HTTPS. "TLS termination" means the proxy decrypts HTTPS and talks plain HTTP to your app.
* **Load balancer:** spreads requests over several workers or machines.

> **Key points**
> * Production adds HTTPS, several processes, health checks, clean shutdown and environment-based configuration.
> * Your FastAPI code barely changes. What changes is *how it is run*.

---

## 1.2 Uvicorn and `fastapi run`

**Uvicorn** is an **ASGI server**: a program that listens on a network port, speaks HTTP, and calls your
FastAPI app (an ASGI application, see Notebook 1, Part 1). `fastapi run` is a thin wrapper around Uvicorn
with production-friendly defaults.

```bash
# Development: auto-reload, listens on 127.0.0.1 only
fastapi dev main.py

# Production: no reload, listens on 0.0.0.0 (all network interfaces), port 8000
fastapi run main.py
fastapi run main.py --workers 4 --port 8080

# The same with plain Uvicorn ("module:variable")
uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4
```

Options you should know (checked in Uvicorn 0.54 and fastapi-cli 0.0.32):

| Option | What it does | Default |
|---|---|---|
| `--host` | Which network interface to listen on. `0.0.0.0` = all (needed in Docker). | `fastapi run`: `0.0.0.0`; `uvicorn`: `127.0.0.1` |
| `--port` | Port number. `fastapi run` also reads the `PORT` environment variable. | 8000 |
| `--workers N` | Start N worker processes. Cannot be combined with `--reload`. | 1 |
| `--reload` | Restart when code changes. Development only (slow, uses more CPU). | off in `run` |
| `--proxy-headers` | Trust `X-Forwarded-For` / `X-Forwarded-Proto` from a proxy, so `request.client` and the URL scheme are the real ones. | on |
| `--forwarded-allow-ips` | Which proxy IPs are trusted for those headers. `*` = trust everyone. | `127.0.0.1` (or env `FORWARDED_ALLOW_IPS`) |
| `--root-path` | The app is served under a path prefix by the proxy, for example `/api`. | empty |
| `--timeout-graceful-shutdown` | Seconds to wait for running requests on shutdown. | wait forever |
| `--timeout-keep-alive` | Close idle keep-alive connections after N seconds. | 5 |
| `--limit-concurrency` | Return 503 when more than N connections/tasks are active. | none |
| `--limit-max-requests` | Restart a worker after N requests (a guard against memory leaks). | none |
| `--log-config`, `--no-access-log` | Logging setup; turn off the access log if your own middleware logs requests. | access log on |

> **Note:** with `fastapi run`, you can save the import string in `pyproject.toml` so you only type `fastapi run`:
> ```toml
> [tool.fastapi]
> entrypoint = "app.main:app"
> ```

### Why proxy headers matter

Behind a proxy, every request reaches your app **from the proxy's IP**, over plain **http**. Without proxy
headers, `request.client.host` is the proxy and `request.url.scheme` is `http`, so redirects and links
point to `http://...` and rate limiting by IP limits everybody together.

The proxy adds headers that say what it saw:

* `X-Forwarded-For: 203.0.113.7` (the real client IP)
* `X-Forwarded-Proto: https` (the original scheme)

Uvicorn's `ProxyHeadersMiddleware` reads them, **but only from trusted IPs**. Otherwise any client could
pretend to be any IP. The demo below wraps an app with that middleware and calls it with TestClient,
whose client address is the string `"testclient"`.

In [3]:
# Proxy headers: what the app sees with and without a trusted proxy
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from uvicorn.middleware.proxy_headers import ProxyHeadersMiddleware

proxy_app = FastAPI()

@proxy_app.get("/whoami")
def whoami(request: Request):
    return {"client": request.client.host, "scheme": request.url.scheme}

forwarded = {"X-Forwarded-For": "203.0.113.7", "X-Forwarded-Proto": "https"}

# 1. No middleware: the headers are ignored
print(TestClient(proxy_app).get("/whoami", headers=forwarded).json())
# {'client': 'testclient', 'scheme': 'http'}

# 2. Middleware, but the caller is not a trusted proxy: still ignored (safe default)
untrusted = ProxyHeadersMiddleware(proxy_app, trusted_hosts="127.0.0.1")
print(TestClient(untrusted).get("/whoami", headers=forwarded).json())
# {'client': 'testclient', 'scheme': 'http'}

# 3. Middleware that trusts every caller (like --forwarded-allow-ips='*')
trusting = ProxyHeadersMiddleware(proxy_app, trusted_hosts="*")
print(TestClient(trusting).get("/whoami", headers=forwarded).json())
# {'client': '203.0.113.7', 'scheme': 'https'}

{'client': 'testclient', 'scheme': 'http'}
{'client': 'testclient', 'scheme': 'http'}
{'client': '203.0.113.7', 'scheme': 'https'}


> **Warning:** `--forwarded-allow-ips='*'` is only safe when the app port is **not reachable from the internet**
> (for example, inside a private Docker network where only the proxy can reach it). Otherwise attackers can fake
> their IP with a header. Prefer listing the proxy IPs or the private network range.

> **Common mistake:** running `uvicorn main:app` inside Docker and getting "connection refused" from outside.
> Plain Uvicorn listens on `127.0.0.1` by default, which is unreachable from outside the container. Use `--host 0.0.0.0`
> (or `fastapi run`, which uses `0.0.0.0`).

> **Key points**
> * `fastapi run` = Uvicorn with production defaults (`0.0.0.0`, no reload, proxy headers on).
> * Turn on proxy headers behind a proxy, and trust only the proxy's IPs.
> * `--reload` is for development only and cannot be used with `--workers`.

### Practice 1.2: server options

**A. Predict the output**

1. You run `uvicorn main:app` in a container and map port 8000. A request from your laptop fails with
   "connection refused". Why?
2. In the demo above, what would case 2 print if `trusted_hosts="testclient"`?

**B. Write code**

3. Write a function `uvicorn_command(module, app_var="app", workers=1, port=8000, behind_proxy=False)`
   that returns the command as a list of strings, for example
   `["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]`. Add `--workers N` when N > 1, and
   `--proxy-headers --forwarded-allow-ips *` when `behind_proxy` is true.
4. Raise `ValueError` if `workers < 1`.

**C. Find and fix the bug**

5. This command is meant for production. Find two problems: `uvicorn main:app --reload --workers 4`.

### Solutions 1.2

1. Plain `uvicorn` listens on `127.0.0.1` (only inside the container). Add `--host 0.0.0.0`.
2. `{'client': '203.0.113.7', 'scheme': 'https'}`. The caller's host `"testclient"` is now trusted.
3. and 4. See the cell below.
5. `--reload` is for development only (it watches files and restarts), and it cannot be combined with `--workers`.
   It also listens on `127.0.0.1`. Fixed: `uvicorn main:app --host 0.0.0.0 --workers 4` (or `fastapi run main.py --workers 4`).

In [4]:
# Solutions 1.2
from fastapi.testclient import TestClient
from uvicorn.middleware.proxy_headers import ProxyHeadersMiddleware

def uvicorn_command(module, app_var="app", workers=1, port=8000, behind_proxy=False):
    if workers < 1:
        raise ValueError("workers must be at least 1")
    cmd = ["uvicorn", f"{module}:{app_var}", "--host", "0.0.0.0", "--port", str(port)]
    if workers > 1:
        cmd += ["--workers", str(workers)]
    if behind_proxy:
        cmd += ["--proxy-headers", "--forwarded-allow-ips", "*"]
    return cmd

print(" ".join(uvicorn_command("main", workers=4, behind_proxy=True)))
# uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4 --proxy-headers --forwarded-allow-ips *

assert uvicorn_command("main") == ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
assert "--workers" not in uvicorn_command("main", workers=1)
assert uvicorn_command("app.main", "api", 2)[1] == "app.main:api"
try:
    uvicorn_command("main", workers=0)
    raise AssertionError("expected ValueError")
except ValueError:
    pass

# Q2: trusting "testclient" makes the headers count
trusted = ProxyHeadersMiddleware(proxy_app, trusted_hosts="testclient")
r = TestClient(trusted).get("/whoami", headers=forwarded)
assert r.json() == {"client": "203.0.113.7", "scheme": "https"}
print("Practice 1.2: all checks passed")

uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4 --proxy-headers --forwarded-allow-ips *
Practice 1.2: all checks passed


---

## 1.3 Workers: how many, and why

Python has the **GIL** (global interpreter lock): inside one process, only one thread runs Python code at a time.
One Uvicorn worker is one process with **one event loop** (plus a thread pool for `def` endpoints, see
Notebook 2, Part 3). So one worker can wait on thousands of slow network calls at once, but it uses
**at most about one CPU core** for Python work.

```text
             one worker = one process = one event loop
   +--------------------------------------------------------+
   |  event loop: async endpoints, waiting on I/O           |
   |  thread pool (default 40 threads): def endpoints       |
   |  GIL: only one thread runs Python bytecode at a time   |
   +--------------------------------------------------------+
   -> great for I/O (DB, HTTP calls), limited to ~1 core for CPU work

             4 workers = 4 processes (4 copies of the app in memory)
   [worker 1] [worker 2] [worker 3] [worker 4]   -> up to ~4 cores
```

**Starting points** (then measure with a load test, Part 3):

| Situation | Starting point | Why |
|---|---|---|
| Async app, mostly I/O, on a VM | workers = number of CPU cores | each worker uses one core; the event loop handles concurrency |
| Many blocking `def` endpoints or sync libraries | more workers, e.g. `2 x cores + 1` (classic Gunicorn rule) | workers spend time waiting, extra processes keep cores busy |
| Kubernetes / ECS / Cloud Run | 1 worker per container, scale the number of containers | the platform restarts and load balances containers; simpler metrics |
| Big ML model in memory | fewer workers | every worker loads its own copy of the model (memory x N) |

Remember the trade-offs:

* More workers = more memory (each has its own copy of the app, caches and models).
* In-memory state (a dict cache, an in-memory rate limiter) is **per worker**. With 4 workers you have 4 different caches.
  Shared state belongs in Redis or the database.

> **Note:** Python 3.13+ also has an optional "free-threaded" build without the GIL. It is still new; most production
> FastAPI deployments today use the normal build and scale with processes.

### Gunicorn with Uvicorn workers

**Gunicorn** is a mature process manager from the WSGI world. It starts workers, restarts crashed ones and
supports graceful reloads. With an ASGI app you tell it to use **Uvicorn worker classes**.

```bash
pip install gunicorn uvicorn-worker
gunicorn main:app -k uvicorn_worker.UvicornWorker -w 4 -b 0.0.0.0:8000 \
    --graceful-timeout 30 --timeout 60 --max-requests 1000 --max-requests-jitter 100
```

> **Note:** older tutorials write `-k uvicorn.workers.UvicornWorker`. In Uvicorn 0.54 that module still exists
> but raises a `DeprecationWarning` that says to use the separate `uvicorn-worker` package instead.
> Also, modern `uvicorn --workers N` has its own supervisor that restarts dead workers, so many teams
> no longer need Gunicorn. Both are valid; Gunicorn gives more process-management knobs.
> Gunicorn does not run on Windows; use `uvicorn --workers` there.

> **Interview tip:** "Why is one worker a bottleneck?" Say: one worker is one process, and because of the GIL it
> runs Python on about one core. If an endpoint blocks the event loop or does CPU work, all requests in that worker
> wait. You scale with more workers or more containers, and you keep shared state outside the process.

In [5]:
# A small helper that turns the table above into code
import os

def recommend_workers(cores: int, workload: str, orchestrated: bool = False) -> int:
    """Return a starting number of workers. Always confirm with a load test."""
    if cores < 1:
        raise ValueError("cores must be >= 1")
    if orchestrated:
        return 1                      # scale containers instead of workers
    if workload == "io":
        return cores
    if workload == "blocking":
        return 2 * cores + 1
    if workload == "ml":
        return max(1, cores // 2)     # memory is usually the limit
    raise ValueError(f"unknown workload: {workload}")

print("This machine has", os.cpu_count(), "logical CPUs")   # depends on your computer
print(recommend_workers(4, "io"))                            # 4
print(recommend_workers(4, "blocking"))                      # 9
print(recommend_workers(4, "io", orchestrated=True))         # 1

This machine has 14 logical CPUs
4
9
1


The same helper as a small console tool. It asks you for the numbers with `input()`. This is a *client-side*
script for a developer, so `input()` is natural here.

In [6]:
# A tiny "deployment advisor" that asks the developer two questions
use_sample_input("8", "blocking")

cores = int(input("How many CPU cores does the server have? "))
workload = input("Workload type (io / blocking / ml): ").strip().lower()
n = recommend_workers(cores, workload)
print(f"Start with: fastapi run main.py --workers {n}")
# How many CPU cores does the server have? 8
# Workload type (io / blocking / ml): blocking
# Start with: fastapi run main.py --workers 17

How many CPU cores does the server have? 8
Workload type (io / blocking / ml): blocking
Start with: fastapi run main.py --workers 17


> **Common mistake:** putting an in-memory counter or cache in the app and running 4 workers. Each request lands on any
> worker, so you see different values. Section 1.8 shows this for real.

> **Key points**
> * One worker ~ one core for Python work. Use workers or containers to use more cores.
> * Start from the table, then measure. Memory and shared state limit the worker count.
> * Use `uvicorn --workers N`, or Gunicorn with `uvicorn_worker.UvicornWorker`.

---

## 1.4 A reverse proxy (Nginx) and HTTPS

You *can* expose Uvicorn directly, but in most production setups a reverse proxy sits in front. It:

* ends **TLS** (holds the HTTPS certificate, for example from Let's Encrypt);
* protects the app from slow clients (it buffers the request and sends it to the app at once);
* serves static files, compresses, limits request size, and spreads load across workers or machines.

A minimal Nginx site for a FastAPI app on port 8000:

```nginx
upstream fastapi_app {
    server 127.0.0.1:8000;          # or app:8000 inside docker-compose
    keepalive 32;
}

server {
    listen 80;
    server_name api.example.com;
    return 301 https://$host$request_uri;          # always redirect to HTTPS
}

server {
    listen 443 ssl;
    server_name api.example.com;
    ssl_certificate     /etc/letsencrypt/live/api.example.com/fullchain.pem;
    ssl_certificate_key /etc/letsencrypt/live/api.example.com/privkey.pem;
    add_header Strict-Transport-Security "max-age=31536000" always;   # HSTS

    client_max_body_size 10m;                      # reject huge uploads early (413)

    location / {
        proxy_pass http://fastapi_app;
        proxy_http_version 1.1;
        proxy_set_header Host $host;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        proxy_set_header Connection "";
    }

    location /ws/ {                                # WebSockets need the upgrade headers
        proxy_pass http://fastapi_app;
        proxy_http_version 1.1;
        proxy_set_header Upgrade $http_upgrade;
        proxy_set_header Connection "upgrade";
    }
}
```

And Uvicorn behind it: `uvicorn main:app --host 127.0.0.1 --port 8000 --proxy-headers --forwarded-allow-ips 127.0.0.1`.

**HSTS** (`Strict-Transport-Security`) tells browsers to use HTTPS only for this domain from now on.

**Path prefix (`root_path`).** If the proxy serves the app under `https://example.com/api/` and strips `/api`
before forwarding, start the app with `--root-path /api` (or `FastAPI(root_path="/api")`). Then `/docs` and the
OpenAPI URLs include the prefix.

> **Note:** for Server-Sent Events, Nginx must not buffer the response. FastAPI's built-in SSE support
> already sends `X-Accel-Buffering: no` for this (Part 6.3).

> **Key points**
> * The proxy handles HTTPS, buffering, body-size limits and load balancing. The app speaks plain HTTP behind it.
> * Forward `Host`, `X-Forwarded-For` and `X-Forwarded-Proto`, and turn on proxy headers in Uvicorn.
> * WebSockets need the `Upgrade`/`Connection` headers in the proxy config.

---

## 1.5 Health checks: liveness vs readiness

A **health check** is a small endpoint that a platform (Docker, Kubernetes, a load balancer) calls every few seconds.

| Check | Question it answers | If it fails | What to check inside |
|---|---|---|---|
| **Liveness** (`/healthz`) | "Is the process alive and not stuck?" | the platform **restarts** the container | almost nothing: return 200 fast |
| **Readiness** (`/readyz`) | "Can it serve traffic right now?" | the platform **stops sending traffic** (no restart) | DB reachable, model loaded, not shutting down |
| **Startup** (optional) | "Has slow startup finished?" | waits before running the other two | same as readiness, longer timeout |

> **Common mistake:** checking the database in the **liveness** probe. If the database has a short outage,
> every app container fails liveness and gets restarted at the same time, which makes the outage worse.
> Put dependencies in **readiness**, keep liveness trivial.

In [7]:
# Liveness and readiness with lifespan state
from contextlib import asynccontextmanager
from fastapi import FastAPI, Response, status
from fastapi.testclient import TestClient

class FakeDB:
    def __init__(self):
        self.up = True
    def ping(self) -> bool:
        return self.up

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.db = FakeDB()           # open connections here
    app.state.ready = True            # ready only after startup work is done
    yield
    app.state.ready = False           # shutting down: stop taking traffic

health_app = FastAPI(lifespan=lifespan)

@health_app.get("/healthz")
def liveness():
    return {"status": "alive"}

@health_app.get("/readyz")
def readiness(response: Response):
    checks = {"startup_done": health_app.state.ready, "database": health_app.state.db.ping()}
    if not all(checks.values()):
        response.status_code = status.HTTP_503_SERVICE_UNAVAILABLE
    return {"ready": all(checks.values()), "checks": checks}

with TestClient(health_app) as client:          # "with" runs the lifespan
    print(client.get("/healthz").status_code)    # 200
    print(client.get("/readyz").json())          # {'ready': True, 'checks': {'startup_done': True, 'database': True}}
    health_app.state.db.up = False               # simulate a DB outage
    r = client.get("/readyz")
    print(r.status_code, r.json()["checks"])     # 503 {'startup_done': True, 'database': False}
    print(client.get("/healthz").status_code)    # 200  (still alive: do NOT restart)

200
{'ready': True, 'checks': {'startup_done': True, 'database': True}}
503 {'startup_done': True, 'database': False}
200


In Kubernetes the probes look like this (shown for reference, not run):

```yaml
livenessProbe:
  httpGet: {path: /healthz, port: 8000}
  periodSeconds: 10
  failureThreshold: 3
readinessProbe:
  httpGet: {path: /readyz, port: 8000}
  periodSeconds: 5
  failureThreshold: 2
```

> **Key points**
> * Liveness: "am I alive?" Keep it trivial. Readiness: "can I serve?" Check dependencies.
> * Return **503** when not ready. Load balancers look at the status code, not the body.

---

## 1.6 Graceful shutdown

Deployments replace old containers with new ones all the time. **Graceful shutdown** means: when the platform asks
the process to stop (it sends the **SIGTERM** signal, a polite "please stop" message from the operating system), the server

1. stops accepting new connections,
2. lets running requests finish (up to a timeout),
3. runs the **shutdown part of your lifespan** (close DB pools, flush logs, stop background workers),
4. exits.

```text
 SIGTERM -> stop accepting -> finish in-flight requests -> lifespan code after "yield" -> exit
              (readiness should now fail so the load balancer stops sending traffic)
```

Things to set:

* Uvicorn: `--timeout-graceful-shutdown 30` (otherwise it waits for long requests forever).
* Kubernetes: `terminationGracePeriodSeconds` (default 30) must be longer than your graceful timeout.
* Long streams (SSE, WebSockets) keep connections open: give them a way to end, or they block shutdown until the timeout.
* `BackgroundTasks` run inside the worker. If the process is killed, unfinished tasks are lost. Work that must not be
  lost belongs in a real queue (Celery, RQ, Arq), see Notebook 2, Part 7.

With TestClient you can see the lifespan shutdown code run when the `with` block ends:

In [8]:
# The shutdown half of the lifespan runs on exit
from contextlib import asynccontextmanager
from fastapi import FastAPI
from fastapi.testclient import TestClient

events = []

@asynccontextmanager
async def lifespan(app: FastAPI):
    events.append("startup: open pool")
    yield
    events.append("shutdown: finish work, close pool")

shutdown_app = FastAPI(lifespan=lifespan)

@shutdown_app.get("/")
def home():
    events.append("request")
    return {"ok": True}

with TestClient(shutdown_app) as client:
    client.get("/")
print(events)   # ['startup: open pool', 'request', 'shutdown: finish work, close pool']

['startup: open pool', 'request', 'shutdown: finish work, close pool']


> **Note:** `@app.on_event("startup")` / `("shutdown")` is the **old** style. In FastAPI 0.162 it still works but is
> marked deprecated in favour of `lifespan`. You will see it in old code and interviews; write `lifespan` in new code.

> **Key points**
> * SIGTERM -> stop accepting -> drain -> lifespan shutdown -> exit.
> * Set a graceful timeout and make the platform's grace period longer.
> * Do not rely on in-process background tasks for work that must survive a restart.

---

## 1.7 12-factor configuration

The **Twelve-Factor App** is a well-known list of rules for apps that run in the cloud. The ones that matter most for a FastAPI service:

| Factor | In practice |
|---|---|
| III. Config in the environment | DB URL, secrets, feature flags come from environment variables, not from code |
| IV. Backing services as attached resources | Postgres, Redis, LLM APIs are just URLs in config; swap them without code changes |
| VI. Stateless processes | no user data in process memory or local disk; use DB / Redis / object storage |
| VII. Port binding | the app serves HTTP on a port (`--port` or `PORT`) |
| VIII. Concurrency | scale by adding processes (workers, containers) |
| IX. Disposability | start fast, shut down gracefully (sections 1.5 and 1.6) |
| X. Dev/prod parity | same Postgres version in dev and prod (docker-compose, Part 2) |
| XI. Logs as event streams | write logs to stdout; the platform collects them (Part 4) |

Notebook 2, Part 2 showed `pydantic-settings`. Here is the production angle: **the same image runs in every
environment, only the environment variables change**, and the app fails fast at startup if a required value is missing.

In [9]:
# Settings from the environment, failing fast when a required value is missing
import os
from pydantic import SecretStr, ValidationError
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="TASKAPP_")
    environment: str = "dev"
    database_url: str = "sqlite:///./dev.db"
    jwt_secret: SecretStr                      # required: no default
    workers: int = 1

def load_settings(env: dict):
    """Load settings with a temporary set of environment variables (for the demo)."""
    old = dict(os.environ)
    os.environ.update(env)
    try:
        return Settings()
    finally:
        os.environ.clear()
        os.environ.update(old)

prod = load_settings({"TASKAPP_ENVIRONMENT": "prod",
                      "TASKAPP_DATABASE_URL": "postgresql+psycopg://app@db/app",
                      "TASKAPP_JWT_SECRET": "change-me", "TASKAPP_WORKERS": "4"})
print(prod.environment, prod.workers, prod.database_url)   # prod 4 postgresql+psycopg://app@db/app
print(prod.jwt_secret)                                      # **********  (SecretStr hides the value)

try:
    load_settings({})                                       # missing TASKAPP_JWT_SECRET
except ValidationError as e:
    print(e.errors()[0]["type"], e.errors()[0]["loc"])      # missing ('jwt_secret',)

prod 4 postgresql+psycopg://app@db/app
**********
missing ('jwt_secret',)


> **Warning:** never commit `.env` files with real secrets, and never bake secrets into a Docker image.
> In production, secrets come from the platform (Kubernetes Secrets, AWS Secrets Manager, Docker secrets).

### Practice 1.5-1.7: health, shutdown and config

**A. Predict the output**

1. A liveness probe checks the database. The database is down for 2 minutes. What happens to your containers?
2. In the shutdown demo, what does `events` contain if you call `TestClient(shutdown_app).get("/")` **without** `with`?

**B. Write code**

3. Build an app `ready_app` with `/healthz` (always 200) and `/readyz` that returns 503 while a module-level
   flag `draining` is `True` (the process received SIGTERM and is draining). Test both states.
4. Write a `Settings` class with prefix `SHOP_`, a required `api_key: SecretStr` and `debug: bool = False`.
   Load it with `{"SHOP_API_KEY": "k1", "SHOP_DEBUG": "true"}` and check that `debug is True`.

**C. Find and fix the bug**

5. Why is this readiness check wrong?
   ```python
   @app.get("/readyz")
   def ready():
       if not db_ok():
           return {"ready": False}
       return {"ready": True}
   ```

### Solutions 1.5-1.7

1. All containers fail liveness and are restarted again and again, which adds load during the outage and never fixes it.
   Check the DB in readiness instead.
2. Only `'request'` is added. Without `with`, TestClient does not run the lifespan.
3. and 4. See the cell below.
5. It always returns **200**. Load balancers look at the status code, not the body. Return 503 when not ready.

In [10]:
# Solutions 1.5-1.7
import os
from fastapi import FastAPI, Response
from fastapi.testclient import TestClient
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

# Q2
events.clear()
TestClient(shutdown_app).get("/")
assert events == ["request"]

# Q3
draining = False
ready_app = FastAPI()

@ready_app.get("/healthz")
def healthz():
    return {"status": "alive"}

@ready_app.get("/readyz")
def readyz(response: Response):
    if draining:
        response.status_code = 503
        return {"ready": False, "reason": "draining"}
    return {"ready": True}

c = TestClient(ready_app)
assert c.get("/readyz").status_code == 200
draining = True
assert c.get("/readyz").status_code == 503
assert c.get("/healthz").status_code == 200
draining = False

# Q4
class ShopSettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="SHOP_")
    api_key: SecretStr
    debug: bool = False

old = dict(os.environ)
os.environ.update({"SHOP_API_KEY": "k1", "SHOP_DEBUG": "true"})
try:
    s = ShopSettings()
finally:
    os.environ.clear(); os.environ.update(old)
assert s.debug is True and s.api_key.get_secret_value() == "k1"
print("Practice 1.5-1.7: all checks passed")

Practice 1.5-1.7: all checks passed


---

## 1.8 Real-server demo: Uvicorn started from Python

Everything so far used TestClient (no network). Once, let us run a **real** server, like production does:

1. write a small app into `fastapi_lab_3/prod_demo.py`;
2. start `python -m uvicorn prod_demo:app` with `subprocess.Popen` on a free port;
3. wait until `/healthz` answers, then call it with `httpx` over the real network stack;
4. stop it in `try/finally`, asking for a graceful stop first.

The app counts requests **in memory**. We start it with **2 workers** to see the "state is per worker" problem
from section 1.3 for real.

In [11]:
# Write the app file into the lab folder
from pathlib import Path

LAB = Path("fastapi_lab_3")
LAB.mkdir(exist_ok=True)

(LAB / "prod_demo.py").write_text("""
import os
import time
from contextlib import asynccontextmanager
from fastapi import FastAPI

counter = {"hits": 0}

@asynccontextmanager
async def lifespan(app: FastAPI):
    yield
    print(f"worker {os.getpid()} shutting down after {counter['hits']} hits", flush=True)

app = FastAPI(lifespan=lifespan)

@app.get("/healthz")
def healthz():
    return {"status": "alive"}

@app.get("/hit")
async def hit():
    counter["hits"] += 1
    time.sleep(0.1)   # blocks this worker's event loop ON PURPOSE (see Part 3), so the other worker takes new calls
    return {"pid": os.getpid(), "hits_in_this_worker": counter["hits"]}
""", encoding="utf-8")
print(sorted(p.name for p in LAB.iterdir()))   # ['prod_demo.py']

['prod_demo.py']


In [12]:
# Start uvicorn with 2 workers, call it with httpx, then stop it
import signal, socket, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor
import httpx

def free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

port = free_port()
cmd = [sys.executable, "-m", "uvicorn", "prod_demo:app", "--host", "127.0.0.1",
       "--port", str(port), "--workers", "2", "--timeout-graceful-shutdown", "5", "--no-access-log"]
flags = subprocess.CREATE_NEW_PROCESS_GROUP if sys.platform == "win32" else 0
proc = subprocess.Popen(cmd, cwd=LAB, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, creationflags=flags)
try:
    base = f"http://127.0.0.1:{port}"
    for _ in range(100):                          # wait up to ~20 s for startup
        try:
            if httpx.get(base + "/healthz", timeout=1).status_code == 200:
                break
        except httpx.HTTPError:
            time.sleep(0.2)
    else:
        raise RuntimeError("server did not start")
    print("server is up")                         # server is up
    time.sleep(2)                                 # the second worker may start a moment later

    def call_hit(_):
        return httpx.get(base + "/hit", timeout=10).json()
    with ThreadPoolExecutor(max_workers=10) as pool:   # 20 calls, 10 at a time
        answers = list(pool.map(call_hit, range(20)))
    per_pid = {}
    for a in answers:
        per_pid[a["pid"]] = max(per_pid.get(a["pid"], 0), a["hits_in_this_worker"])
    print("worker processes that answered:", len(per_pid))      # 2
    print("hits counted by each worker:", sorted(per_pid.values()), "total:", sum(per_pid.values()))
    # hits counted by each worker: e.g. [9, 11] total: 20  -> no single worker knows the real total
finally:
    # Graceful stop: SIGTERM on Linux/macOS, CTRL_BREAK on Windows; hard kill only as a fallback
    try:
        proc.send_signal(signal.CTRL_BREAK_EVENT if sys.platform == "win32" else signal.SIGTERM)
        out, _ = proc.communicate(timeout=15)
    except (subprocess.TimeoutExpired, OSError):
        proc.kill()
        out, _ = proc.communicate()
    print("server stopped")
    shutdown_lines = [line for line in out.splitlines() if "shutting down after" in line]
    print("workers that ran lifespan shutdown:", len(shutdown_lines))   # 2

server is up


worker processes that answered: 2
hits counted by each worker: [8, 12] total: 20


server stopped
workers that ran lifespan shutdown: 2


What you should notice:

* The **same URL** was answered by **different processes**, each with its own `counter`. No worker knows the real
  total. That is why in-memory state is wrong once you scale.
* Which worker gets a connection is decided by the operating system. Here the endpoint blocks its event loop for 0.1 s on
  purpose, so a busy worker cannot accept new connections and the other one takes them.
* On a graceful stop, each worker ran its lifespan shutdown code (the count shows how many did). If a process is killed
  hard (`kill -9`, `proc.kill()`), the shutdown code never runs.

> **Note:** on Windows, the console signal a separate process group can receive is Ctrl+Break, and Uvicorn handles it
> like SIGTERM. That is why the cell uses `CREATE_NEW_PROCESS_GROUP` and `CTRL_BREAK_EVENT` on Windows. If the signal
> cannot be delivered in your environment, the cell falls back to `kill()` and you see 0 shutdown lines.

### Practice 1.8: the real server

**A. Predict:** you change the demo to `--workers 1`. How many different PIDs do you see? What is the highest
`hits_in_this_worker`? Roughly how long do the 20 calls take now, and why?

**B. Write code:** write a function `wait_until_up(url, timeout_s)` that polls a URL with httpx until it returns 200, and
returns the number of seconds it took, or raises `TimeoutError`. Test it against a URL that never answers
(use `httpx.MockTransport` that always raises `httpx.ConnectError`, so no real network is used) and against one that
answers on the third try.

### Solutions 1.8

**A.** One PID, and the highest `hits_in_this_worker` is 20, because one process handled every call. The 20 calls take
about 20 x 0.1 = 2 seconds, because `time.sleep` inside `async def` blocks the only event loop, so the requests run one
after another. This is the "one worker is a bottleneck" problem (Part 3.2).

**B.** The function takes an optional `client` so the test can inject a `MockTransport`.

In [13]:
# Solutions 1.8
import time
import httpx

def wait_until_up(url: str, timeout_s: float, client: httpx.Client | None = None, pause: float = 0.01) -> float:
    client = client or httpx.Client()
    start = time.perf_counter()
    while time.perf_counter() - start < timeout_s:
        try:
            if client.get(url, timeout=1).status_code == 200:
                return time.perf_counter() - start
        except httpx.HTTPError:
            pass
        time.sleep(pause)
    raise TimeoutError(f"{url} not up after {timeout_s}s")

def never_up(request):
    raise httpx.ConnectError("connection refused", request=request)

try:
    wait_until_up("http://app/healthz", 0.1, httpx.Client(transport=httpx.MockTransport(never_up)))
    raise AssertionError("expected TimeoutError")
except TimeoutError as e:
    print("TimeoutError:", e)       # TimeoutError: http://app/healthz not up after 0.1s

calls = {"n": 0}
def third_time_lucky(request):
    calls["n"] += 1
    if calls["n"] < 3:
        raise httpx.ConnectError("not yet", request=request)
    return httpx.Response(200, json={"status": "alive"})

took = wait_until_up("http://app/healthz", 2, httpx.Client(transport=httpx.MockTransport(third_time_lucky)))
assert calls["n"] == 3 and took < 2
print("Practice 1.8: all checks passed")

TimeoutError: http://app/healthz not up after 0.1s
Practice 1.8: all checks passed


---

## How to debug production startup problems

| Symptom | Likely cause | What to do |
|---|---|---|
| "Connection refused" from outside a container | listening on `127.0.0.1` | `--host 0.0.0.0` or `fastapi run` |
| `Error loading ASGI app. Could not import module "main"` | wrong working directory or import string | run from the folder that contains the package; check `module:variable` |
| `request.client.host` is always the proxy IP | proxy headers not trusted | `--proxy-headers --forwarded-allow-ips <proxy ip>` |
| Links/redirects use `http://` behind HTTPS | `X-Forwarded-Proto` missing or not trusted | set it in the proxy, trust the proxy |
| Values differ between requests | in-memory state + several workers | move state to Redis/DB |
| Deploys cut off requests | no graceful shutdown | handle SIGTERM (Uvicorn does), set timeouts, fail readiness while draining |
| Container restarted during a DB outage | DB check in liveness probe | move it to readiness |

## Summary

* Run with `fastapi run` or `uvicorn --host 0.0.0.0`, never `--reload` in production.
* One worker uses about one core. Scale with workers or containers, and keep state outside the process.
* Put a reverse proxy in front for HTTPS, buffering and limits; trust its forwarded headers.
* `/healthz` = alive (trivial), `/readyz` = can serve (checks dependencies, 503 when not).
* Graceful shutdown: SIGTERM, drain, lifespan cleanup. Configuration comes from the environment.

# Part 2: Docker for FastAPI

**What this Part covers**

* What a container image is, and why teams ship FastAPI apps as Docker images.
* A production Dockerfile, line by line: slim base image, layer caching, non-root user, multi-stage build.
* `.dockerignore`, environment variables and secrets.
* `docker-compose` with Postgres, health checks and `depends_on`.

**Before you start:** Part 1 (host, port, workers, health checks).

> **Note:** Docker is not needed to run this notebook. The Docker files are shown as code blocks to read and copy.
> The code cells in this Part check Dockerfile text with small Python functions, so you still practise.

## 2.1 Images, containers and layers

* **Image:** a read-only package with an operating system base, Python, your dependencies and your code.
* **Container:** a running instance of an image (like an object made from a class).
* **Layer:** each instruction in a Dockerfile (`FROM`, `COPY`, `RUN`) creates a layer. Docker **caches** layers:
  if an instruction and its inputs did not change, the old layer is reused and the build is fast.
* **Registry:** a store for images (Docker Hub, GitHub Container Registry, AWS ECR).

```text
   Dockerfile --docker build--> image (layers) --docker push--> registry
                                     |
                                     +--docker run--> container (your app, port 8000)

   Layer cache: a change in one layer rebuilds that layer AND every layer after it.
   [FROM python:3.14-slim]       cached
   [COPY requirements.txt]       cached (unchanged)
   [RUN pip install ...]         cached  <- slow step, so keep it BEFORE copying code
   [COPY ./app]                  rebuilt (you changed code)
```

> **Key points**
> * One image runs the same everywhere (laptop, CI, cloud): "works on my machine" problems shrink.
> * Order the Dockerfile from "changes rarely" to "changes often" to use the cache.

---

## 2.2 A production Dockerfile, line by line

```dockerfile
# syntax=docker/dockerfile:1

# ---------- stage 1: build the dependencies ----------
FROM python:3.14-slim AS builder                 # (1) small official base image
ENV PIP_NO_CACHE_DIR=1 PIP_DISABLE_PIP_VERSION_CHECK=1
WORKDIR /build
COPY requirements.txt .                          # (2) copy ONLY the dependency list first
RUN python -m venv /opt/venv \
 && /opt/venv/bin/pip install -r requirements.txt   # (3) cached until requirements.txt changes

# ---------- stage 2: the small runtime image ----------
FROM python:3.14-slim AS runtime                 # (4) fresh image: no build tools, no pip cache
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PATH="/opt/venv/bin:$PATH"                   # (5) logs appear immediately; use the venv
RUN useradd --create-home --uid 10001 appuser    # (6) a non-root user
WORKDIR /app
COPY --from=builder /opt/venv /opt/venv          # (7) take only the installed packages
COPY --chown=appuser:appuser ./app ./app         # (8) code last: it changes most often
USER appuser                                     # (9) never run as root
EXPOSE 8000                                      # (10) documentation: the app listens on 8000
HEALTHCHECK --interval=30s --timeout=3s --start-period=10s --retries=3 \
  CMD python -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/healthz', timeout=2)"
CMD ["fastapi", "run", "app/main.py", "--port", "8000", "--proxy-headers"]   # (11) exec form
```

What each numbered line does and why:

1. **`python:3.14-slim`**: Debian-based, small, has a normal C library so most wheels install. `alpine` is smaller
   but uses musl, so many Python packages must compile from source; it is often slower to build and not smaller in the end.
2. and 3. **Layer caching:** copy `requirements.txt` alone and install it. Editing your code does not re-run `pip install`.
4. **Multi-stage build:** the final image starts fresh and copies only what is needed, so compilers and caches from the
   builder stage are left behind. Smaller image, smaller attack surface.
5. `PYTHONUNBUFFERED=1` sends prints/logs straight to stdout (Part 1.7, factor XI). `PYTHONDONTWRITEBYTECODE=1` skips `.pyc` files.
6. and 9. **Non-root user:** if an attacker breaks into the app, they are not root inside the container.
7. Only the virtual environment is copied from the builder.
8. Code is copied last and owned by the app user.
10. `EXPOSE` documents the port; `-p 8000:8000` on `docker run` actually publishes it.
11. **Exec form** (`["fastapi", "run", ...]`) makes the server PID 1, so it receives SIGTERM directly and shuts down
    gracefully (Part 1.6). The shell form (`CMD fastapi run ...`) wraps it in `/bin/sh`, which may not forward the signal.

Build and run:

```bash
docker build -t taskapi:1.0 .
docker run --rm -p 8000:8000 -e TASKAPP_JWT_SECRET=dev-only taskapi:1.0
docker run --rm -p 8000:8000 --env-file .env.prod taskapi:1.0
```

> **Note:** one worker per container is common when an orchestrator (Kubernetes, ECS) runs many containers
> (Part 1.3). On a single VM you may add `--workers 4` to the `CMD`.

> **Tip:** `uv` is a fast Python package manager. Many teams now use `uv sync --frozen` with a `uv.lock` file instead of
> `pip install -r requirements.txt`. The layer-caching idea is the same: copy the lock file first, install, then copy code.

### `.dockerignore`

`COPY . .` sends the whole folder (the "build context") to Docker. Without a `.dockerignore`, you may copy your
virtual environment, `.git`, test databases and **secret `.env` files** into the image.

```gitignore
# .dockerignore
.git
.venv
venv
__pycache__/
*.pyc
.pytest_cache/
.env
.env.*
*.db
tests/
notebooks/
Dockerfile
docker-compose*.yml
```

> **Common mistake:** `ENV OPENAI_API_KEY=sk-...` or `COPY .env .` in the Dockerfile. Anyone who can pull the image
> can read it (`docker history`, `docker inspect`). Pass secrets at run time (`-e`, `--env-file`, orchestrator secrets),
> or use BuildKit secrets for build-time needs (`RUN --mount=type=secret,id=pip_token ...`).

> **Key points**
> * Slim base, dependencies before code, multi-stage, non-root user, exec-form `CMD`, a `HEALTHCHECK`.
> * `.dockerignore` keeps the image small and keeps secrets out.

In [14]:
# A tiny Dockerfile "linter": checks a few rules from this section
def lint_dockerfile(text: str) -> list[str]:
    lines = [ln.strip() for ln in text.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    upper = [ln.upper() for ln in lines]
    problems = []
    if not any(ln.startswith("USER ") and "ROOT" not in ln for ln in upper):
        problems.append("runs as root: add a non-root USER")
    if any(ln.startswith("FROM ") and ln.endswith(":LATEST") for ln in upper):
        problems.append("unpinned base image tag 'latest'")
    if any(ln.startswith("ENV ") and any(w in ln for w in ("KEY=", "SECRET=", "PASSWORD=")) for ln in upper):
        problems.append("secret in ENV")
    copy_all = next((i for i, ln in enumerate(upper) if ln.startswith("COPY . ")), None)
    pip = next((i for i, ln in enumerate(upper) if "PIP INSTALL" in ln), None)
    if copy_all is not None and pip is not None and copy_all < pip:
        problems.append("code copied before pip install: breaks layer caching")
    cmd = next((ln for ln in lines if ln.upper().startswith("CMD")), "")
    if cmd and not cmd[3:].strip().startswith("["):
        problems.append("CMD in shell form: use exec form so SIGTERM reaches the server")
    if any("--RELOAD" in ln for ln in upper):
        problems.append("--reload in a production image")
    return problems

bad = """
FROM python:latest
ENV API_KEY=sk-123
COPY . .
RUN pip install -r requirements.txt
CMD uvicorn main:app --reload
"""
for p in lint_dockerfile(bad):
    print("-", p)
# - runs as root: add a non-root USER
# - unpinned base image tag 'latest'
# - secret in ENV
# - code copied before pip install: breaks layer caching
# - CMD in shell form: use exec form so SIGTERM reaches the server
# - --reload in a production image

- runs as root: add a non-root USER
- unpinned base image tag 'latest'
- secret in ENV
- code copied before pip install: breaks layer caching
- CMD in shell form: use exec form so SIGTERM reaches the server
- --reload in a production image


---

## 2.3 docker-compose with Postgres

**docker-compose** starts several containers together from one YAML file: here, the API and a Postgres database.
Containers in the same compose project share a private network and reach each other **by service name**
(`db`, not `localhost`).

```yaml
# docker-compose.yml
services:
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      TASKAPP_ENVIRONMENT: dev
      TASKAPP_DATABASE_URL: postgresql+psycopg://app:app@db:5432/app   # "db" = service name
    env_file:
      - .env                       # secrets for local dev only, never committed
    depends_on:
      db:
        condition: service_healthy # wait until Postgres answers, not just "started"
    restart: unless-stopped

  db:
    image: postgres:17
    environment:
      POSTGRES_USER: app
      POSTGRES_PASSWORD: app       # dev only; use secrets in production
      POSTGRES_DB: app
    volumes:
      - pgdata:/var/lib/postgresql/data   # data survives "docker compose down"
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U app -d app"]
      interval: 5s
      timeout: 3s
      retries: 10

volumes:
  pgdata:
```

```bash
docker compose up --build          # build the API image and start both services
docker compose logs -f api         # follow the API logs
docker compose exec db psql -U app # open a SQL shell in the db container
docker compose down                # stop (add -v to also delete the data volume)
```

**Database migrations.** Do not call `Base.metadata.create_all()` in production. Use **Alembic** migrations and run them
as a separate step before the new version starts (a one-off container, an init container, or a CI/CD step):

```bash
docker compose run --rm api alembic upgrade head
```

> **Common mistake:** `DATABASE_URL=...@localhost:5432/...` inside the API container. Inside a container, `localhost` is the
> container itself. Use the service name `db`.

> **Common mistake:** `depends_on: [db]` without a health condition. Compose then only waits until the db container has
> *started*, not until Postgres accepts connections, so the API crashes on its first query. Use `condition: service_healthy`
> (and still add retries in the app, because databases can restart later too).

### Secrets in production

| Where | How |
|---|---|
| Local dev | `.env` file (in `.gitignore` and `.dockerignore`) |
| Docker Swarm / Compose | `secrets:` mounted as files under `/run/secrets/...` |
| Kubernetes | `Secret` objects mapped to env vars or files |
| Cloud | AWS Secrets Manager, GCP Secret Manager, Azure Key Vault |

`pydantic-settings` can read secret files too: `SettingsConfigDict(secrets_dir="/run/secrets")`.

> **Key points**
> * Compose = several containers + a private network; use service names as host names.
> * Use health checks and `condition: service_healthy`; keep data in named volumes.
> * Run Alembic migrations as a separate step. Secrets come from the platform.

### Practice 2: Docker

**A. Predict**

1. You edit one line in `app/main.py` and rebuild the Dockerfile from section 2.2. Which steps are re-run?
2. Your API container logs `connection refused` for `localhost:5432` although Postgres is running in compose. Why?

**B. Write code**

3. Run `lint_dockerfile` on the good Dockerfile from section 2.2 and check that it reports no problems.
4. Write `parse_env_file(text) -> dict` for `.env` files: skip blank lines and `#` comments, split on the **first** `=`,
   strip spaces and surrounding quotes. `DATABASE_URL=postgresql://u:p=x@db/app` must keep the `=` inside the value.
5. Write `compose_db_url(user, password, host, port, db)` that returns a SQLAlchemy URL for psycopg, with the password
   URL-escaped (use `urllib.parse.quote`) so a password like `p@ss:word` works.

**C. Find and fix the bug**

6. ```dockerfile
   FROM python:3.14-slim
   WORKDIR /app
   COPY . .
   RUN pip install -r requirements.txt
   CMD fastapi run main.py
   ```
   List the problems and write the fixed version.

### Solutions 2

1. Only `COPY ./app ./app` in the runtime stage and the steps after it. The `pip install` layer is cached because
   `requirements.txt` did not change.
2. Inside a container, `localhost` is the container itself. Use the compose service name: `db:5432`.
3. to 5. See the cell below.
6. Problems: it copies everything before installing (any code change re-runs `pip install`), runs as root,
   uses the shell form of `CMD` (SIGTERM may not reach the server), and has no `.dockerignore`/health check.
   Fixed:
   ```dockerfile
   FROM python:3.14-slim
   ENV PYTHONUNBUFFERED=1
   WORKDIR /app
   COPY requirements.txt .
   RUN pip install --no-cache-dir -r requirements.txt
   RUN useradd --create-home appuser
   COPY --chown=appuser:appuser . .
   USER appuser
   CMD ["fastapi", "run", "main.py", "--port", "8000"]
   ```

In [15]:
# Solutions 2
from urllib.parse import quote

good = """
# syntax=docker/dockerfile:1
FROM python:3.14-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN python -m venv /opt/venv && /opt/venv/bin/pip install -r requirements.txt
FROM python:3.14-slim AS runtime
ENV PYTHONUNBUFFERED=1 PATH="/opt/venv/bin:$PATH"
RUN useradd --create-home --uid 10001 appuser
WORKDIR /app
COPY --from=builder /opt/venv /opt/venv
COPY --chown=appuser:appuser ./app ./app
USER appuser
CMD ["fastapi", "run", "app/main.py", "--port", "8000", "--proxy-headers"]
"""
assert lint_dockerfile(good) == []      # Q3

def parse_env_file(text: str) -> dict:  # Q4
    result = {}
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        key, _, value = line.partition("=")
        result[key.strip()] = value.strip().strip('"').strip("'")
    return result

env = parse_env_file("""
# local settings
DATABASE_URL=postgresql://u:p=x@db/app
JWT_SECRET = "dev-secret"

DEBUG='true'
""")
assert env == {"DATABASE_URL": "postgresql://u:p=x@db/app", "JWT_SECRET": "dev-secret", "DEBUG": "true"}

def compose_db_url(user, password, host, port, db):  # Q5
    return f"postgresql+psycopg://{quote(user, safe='')}:{quote(password, safe='')}@{host}:{port}/{db}"

url = compose_db_url("app", "p@ss:word", "db", 5432, "app")
print(url)   # postgresql+psycopg://app:p%40ss%3Aword@db:5432/app
assert "p%40ss%3Aword@db" in url

fixed6 = """
FROM python:3.14-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
RUN useradd --create-home appuser
COPY --chown=appuser:appuser . .
USER appuser
CMD ["fastapi", "run", "main.py", "--port", "8000"]
"""
assert lint_dockerfile(fixed6) == []
print("Practice 2: all checks passed")

postgresql+psycopg://app:p%40ss%3Aword@db:5432/app
Practice 2: all checks passed


---

## How to debug Docker problems

| Symptom | Likely cause | What to do |
|---|---|---|
| Every build re-installs all packages | code copied before `pip install` | copy `requirements.txt` first |
| Image is 1+ GB | full base image, build tools, `.venv` copied | `-slim`, multi-stage, `.dockerignore` |
| Cannot reach the app on `localhost:8000` | listening on 127.0.0.1 or port not published | `--host 0.0.0.0` (or `fastapi run`), `-p 8000:8000` |
| Logs appear late or never | Python output buffering | `PYTHONUNBUFFERED=1` |
| Container takes 10 s to stop, then is killed | shell-form `CMD`, SIGTERM not forwarded | exec form `CMD [...]` |
| API cannot reach DB in compose | `localhost` used, or DB not ready | use the service name; `condition: service_healthy` |
| Permission denied writing files | non-root user cannot write to `/app` | write to a volume or a folder owned by the user |

Useful commands: `docker logs <container>`, `docker exec -it <container> sh`, `docker image ls`, `docker history <image>`.

## Summary

* An image packages OS + Python + dependencies + code; containers are running images.
* Good Dockerfile: slim base, dependencies before code, multi-stage, non-root, exec-form `CMD`, health check.
* `.dockerignore` keeps `.env`, `.git` and venvs out. Secrets are injected at run time.
* Compose wires the API and Postgres on a private network; use service names and health conditions.

# Part 3: Performance

**What this Part covers**

* Where the time of a request really goes, and how to measure it.
* Async correctness: the one mistake that makes a FastAPI app slow for everyone.
* Database performance: connection pooling, pagination and the N+1 query problem.
* Response size and serialization (and what this FastAPI version recommends instead of `ORJSONResponse`).
* Caching layers, from HTTP `ETag` to Redis.
* Load testing with Locust or k6, and a tiny load test inside the notebook.

**Before you start:** Notebook 2, Part 3 (async in depth) and Part 4 (SQLAlchemy).

> **Note:** timings in this Part depend on your computer. The comments show typical values; the *ratios* are what matter.

## 3.1 Where does the time go?

A request passes through several layers. In most web APIs, the time is spent **waiting** (database, other services),
not running Python code.

```text
 client --network--> proxy --> Uvicorn --> middleware --> dependencies --> endpoint --> serialize --> back
                                                              |               |
                                                              v               v
                                                        auth, DB session   DB queries, HTTP calls, model inference
 Typical share of time in a CRUD API:   network 10-30% | Python/framework 5-15% | DB and external calls 50-80%
```

**Measure before you optimise.** Simple tools:

* A timing middleware that adds a `Server-Timing` or `X-Process-Time` header (Notebook 2, Part 6 built one).
* Logs with durations per request (Part 4) and per DB query (`echo=True` in SQLAlchemy, or slow-query logs).
* Profilers: `pyinstrument` (shows a call tree for one request), `py-spy top --pid <PID>` (samples a running server without code changes).
* Metrics: p50/p95/p99 latency per route (Part 4.2).

```bash
pip install pyinstrument py-spy
py-spy top --pid 12345              # what is the server doing right now?
py-spy record -o profile.svg --pid 12345 --duration 30   # flame graph
```

> **Interview tip:** when asked "the API is slow, what do you do?", start with *measure*: which endpoint, p95 or average,
> DB time vs app time. Then fix the biggest part first. Guessing is a red flag.

---

## 3.2 Async correctness: do not block the event loop

Notebook 2, Part 3 explained the rule:

* `async def` endpoints run **on the event loop**. Anything slow inside them must be **awaited** (async DB driver, `httpx.AsyncClient`, `asyncio.sleep`).
* `def` endpoints run **in a thread pool**, so blocking calls (`time.sleep`, `requests.get`, a sync DB driver) are fine there.
* A blocking call inside `async def` freezes **every** request of that worker.

Let us *measure* it. We send 5 requests **at the same time** to three endpoints that each "wait" 0.2 s.
`httpx.ASGITransport` lets an `AsyncClient` call the app in memory, and top-level `await` works in Jupyter.

In [16]:
# Measure: blocking vs non-blocking waits under 5 concurrent requests
import asyncio, time
import httpx
from fastapi import FastAPI

perf_app = FastAPI()

@perf_app.get("/async-blocking")
async def async_blocking():
    time.sleep(0.2)                 # WRONG: blocks the event loop
    return {"ok": True}

@perf_app.get("/async-awaiting")
async def async_awaiting():
    await asyncio.sleep(0.2)        # right: the loop serves others while waiting
    return {"ok": True}

@perf_app.get("/sync-def")
def sync_def():
    time.sleep(0.2)                 # fine: runs in the thread pool
    return {"ok": True}

async def timed_burst(path: str, n: int = 5) -> float:
    transport = httpx.ASGITransport(app=perf_app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
        start = time.perf_counter()
        responses = await asyncio.gather(*(client.get(path) for _ in range(n)))
        assert all(r.status_code == 200 for r in responses)
        return time.perf_counter() - start

results = {}
for path in ["/async-blocking", "/async-awaiting", "/sync-def"]:
    results[path] = await timed_burst(path)
    print(f"{path:16} {results[path]:.2f}s")
# /async-blocking  1.00s   <- 5 x 0.2 s, one after another
# /async-awaiting  0.20s   <- all 5 wait together
# /sync-def        0.20s   <- 5 threads wait together

/async-blocking  1.01s


/async-awaiting  0.24s


/sync-def        0.21s


The blocking version is 5 times slower with only 5 users. With 100 users it is 100 times slower, and health checks
start to time out too.

### How to fix blocking code you cannot change

If a library has no async version (an old SDK, a CPU-heavy function), you have three options:

1. Make the endpoint `def` (FastAPI runs it in the thread pool).
2. Keep `async def` and push the blocking part to a thread: `await run_in_threadpool(func, arg)` (from `fastapi.concurrency`)
   or `await anyio.to_thread.run_sync(func, arg)`.
3. For **CPU-heavy** work (image processing, big model inference in pure Python), threads do not help much because of the GIL.
   Use a process pool, a separate worker service, or a task queue (Part 6.2).

The thread pool is **not unlimited**. AnyIO's default limit is 40 threads per worker. If 40 slow `def` endpoints are
running, request 41 waits.

In [17]:
# Option 2: keep async def, move the blocking call to a thread
import anyio
from fastapi.concurrency import run_in_threadpool

def legacy_sdk_call(x: int) -> int:     # imagine a blocking library call
    time.sleep(0.2)
    return x * 2

@perf_app.get("/async-offloaded")
async def async_offloaded():
    value = await run_in_threadpool(legacy_sdk_call, 21)
    return {"value": value}

print(f"/async-offloaded {await timed_burst('/async-offloaded'):.2f}s")   # /async-offloaded 0.20s
print("thread pool size:", anyio.to_thread.current_default_thread_limiter().total_tokens)   # thread pool size: 40

/async-offloaded 0.21s
thread pool size: 40


> **Common mistake:** "async makes everything faster". It does not. Async lets one worker *wait* on many things at once.
> It does nothing for CPU work, and a single blocking call inside `async def` makes things *slower* than plain `def`.

> **Key points**
> * Inside `async def`: only awaitable I/O. Blocking calls go to `def` endpoints or `run_in_threadpool`.
> * CPU-heavy work needs processes or a separate service, not threads.
> * The default thread pool has 40 threads per worker.

### Practice 3.2: async correctness

**A. Predict the output**

1. 10 concurrent requests hit `/async-blocking` (0.2 s each). About how long does the burst take?
2. 10 concurrent requests hit `/sync-def`. About how long? What if it were 100 requests?

**B. Write code**

3. Add an endpoint `/fixed` to a **new** app `fix_app` that calls `legacy_sdk_call(5)` without blocking the loop, keeping `async def`.
   Measure 5 concurrent calls with an `AsyncClient` + `ASGITransport` and assert the burst takes less than 0.6 s.
4. Write `async def fetch_all(urls, client)` that fetches all URLs **concurrently** with `asyncio.gather` and returns the status codes.
   Test it with an `httpx.AsyncClient(transport=httpx.MockTransport(handler))` whose handler returns 200 for `/ok` and 404 otherwise.

**C. Find and fix the bug**

5. ```python
   @app.get("/weather")
   async def weather(city: str):
       r = requests.get(f"https://api.example.com/{city}")   # requests is a blocking library
       return r.json()
   ```

### Solutions 3.2

1. About 2 s: the 10 calls run one after another (10 x 0.2 s).
2. About 0.2 s, because 10 threads wait together. With 100 requests, only 40 run at a time (thread pool limit), so about
   3 rounds: roughly 0.6 s.
3. and 4. See the cell below.
5. `requests.get` blocks the event loop. Use `httpx.AsyncClient` with `await` (best: one shared client created in the lifespan),
   or make the endpoint `def`:
   ```python
   @app.get("/weather")
   async def weather(city: str, request: Request):
       r = await request.app.state.http.get(f"https://api.example.com/{city}")
       return r.json()
   ```

In [18]:
# Solutions 3.2
import asyncio, time
import httpx
from fastapi import FastAPI
from fastapi.concurrency import run_in_threadpool

fix_app = FastAPI()

@fix_app.get("/fixed")
async def fixed():
    return {"value": await run_in_threadpool(legacy_sdk_call, 5)}

async with httpx.AsyncClient(transport=httpx.ASGITransport(app=fix_app), base_url="http://t") as client:
    start = time.perf_counter()
    rs = await asyncio.gather(*(client.get("/fixed") for _ in range(5)))
    took = time.perf_counter() - start
assert all(r.json() == {"value": 10} for r in rs)
assert took < 0.6, took

async def fetch_all(urls, client):
    responses = await asyncio.gather(*(client.get(u) for u in urls))
    return [r.status_code for r in responses]

def handler(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200 if request.url.path == "/ok" else 404)

async with httpx.AsyncClient(transport=httpx.MockTransport(handler), base_url="http://api") as client:
    codes = await fetch_all(["/ok", "/missing", "/ok"], client)
assert codes == [200, 404, 200]
print("Practice 3.2: all checks passed")

Practice 3.2: all checks passed


---

## 3.3 Database performance: pooling, pagination, N+1

### Connection pooling

Opening a database connection is slow (network round trips, authentication, TLS). A **connection pool** keeps a few
connections open and lends them to requests. SQLAlchemy's engine has a pool built in:

```python
engine = create_engine(
    settings.database_url,
    pool_size=10,         # connections kept open per worker process
    max_overflow=20,      # extra connections allowed during spikes
    pool_timeout=30,      # seconds to wait for a free connection before an error
    pool_pre_ping=True,   # check a connection before use (handles DB restarts)
    pool_recycle=1800,    # replace connections older than 30 minutes
)
```

* Create the engine **once** per process (module level or lifespan), never per request.
* The pool is **per worker**. 8 workers x (10 + 20) = up to 240 connections. Postgres often allows only 100 by default.
  Size pools so `workers x (pool_size + max_overflow)` stays below the database limit, or put **PgBouncer** in front.
* The same idea applies to HTTP: create one `httpx.AsyncClient` in the lifespan and reuse it (it keeps connections alive).

### The N+1 query problem

**N+1** means: 1 query to load a list, then 1 more query **for each item** to load related data. With 100 authors you make
101 queries. It often hides in lazy-loaded relationships. Let us count the queries.

In [19]:
# Count SQL queries to see the N+1 problem and its fix
from sqlalchemy import ForeignKey, create_engine, event, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, selectinload

class PerfBase(DeclarativeBase):
    pass

class Author(PerfBase):
    __tablename__ = "authors"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    books: Mapped[list["Book"]] = relationship(back_populates="author")

class Book(PerfBase):
    __tablename__ = "books"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str]
    author_id: Mapped[int] = mapped_column(ForeignKey("authors.id"))
    author: Mapped[Author] = relationship(back_populates="books")

perf_engine = create_engine("sqlite://")          # in-memory database
PerfBase.metadata.create_all(perf_engine)

query_log = []
@event.listens_for(perf_engine, "before_cursor_execute")
def count_query(conn, cursor, statement, params, context, executemany):
    query_log.append(statement)

with Session(perf_engine) as s:
    for i in range(20):
        s.add(Author(name=f"Author {i}", books=[Book(title=f"Book {i}-{j}") for j in range(3)]))
    s.commit()

def books_per_author(load_options=()):
    query_log.clear()
    with Session(perf_engine) as s:
        authors = s.scalars(select(Author).options(*load_options)).all()
        result = {a.name: len(a.books) for a in authors}     # touching a.books triggers lazy loads
    return result, len(query_log)

_, n_lazy = books_per_author()
_, n_eager = books_per_author([selectinload(Author.books)])
print("lazy loading queries:", n_lazy)       # lazy loading queries: 21   (1 + 20)
print("selectinload queries:", n_eager)      # selectinload queries: 2    (authors, then all their books)

lazy loading queries: 21
selectinload queries: 2


`selectinload` loads all related books with **one** extra query (`WHERE author_id IN (...)`). `joinedload` uses a JOIN instead,
which is good for many-to-one relations (book -> author). In async SQLAlchemy, lazy loading raises an error instead of
silently running queries, which is why async code must load relations explicitly.

> **Tip:** set `lazy="raise"` on relationships in performance-critical apps. Any accidental lazy load then fails loudly in tests.

### Pagination for performance

Never return "all rows". Two styles (API design details in Part 5.3):

* **Offset:** `LIMIT 20 OFFSET 10000`. Simple, supports "jump to page 50", but the DB still walks past 10,000 rows, and rows can be
  skipped or repeated when data changes between pages.
* **Keyset / cursor:** `WHERE id > :last_id ORDER BY id LIMIT 20`. Uses the index, stays fast on page 10,000, stable under inserts.
  Cannot jump to an arbitrary page.

Always have an **index** on the column you sort and filter by.

In [20]:
# Offset vs keyset pagination on the same data
def page_offset(session, page: int, size: int):
    return session.scalars(select(Book.id).order_by(Book.id).limit(size).offset((page - 1) * size)).all()

def page_keyset(session, after_id: int, size: int):
    return session.scalars(select(Book.id).where(Book.id > after_id).order_by(Book.id).limit(size)).all()

with Session(perf_engine) as s:
    print(page_offset(s, page=2, size=5))          # [6, 7, 8, 9, 10]
    first = page_keyset(s, after_id=0, size=5)
    second = page_keyset(s, after_id=first[-1], size=5)
    print(first, second)                           # [1, 2, 3, 4, 5] [6, 7, 8, 9, 10]

[6, 7, 8, 9, 10]
[1, 2, 3, 4, 5] [6, 7, 8, 9, 10]


> **Key points**
> * One engine per process; size pools against the DB connection limit; `pool_pre_ping=True`.
> * Watch the query count. Fix N+1 with `selectinload` / `joinedload`.
> * Paginate everything; prefer keyset pagination for large or fast-changing tables.

### Practice 3.3: database performance

**A. Predict**

1. 6 workers, `pool_size=10`, `max_overflow=5`. What is the maximum number of DB connections?
2. With lazy loading, how many queries does `books_per_author()` run for 50 authors?

**B. Write code**

3. Write `titles_with_author_names()` that returns `["Book 0-0 by Author 0", ...]` for all books, using
   **at most 2 queries** (hint: `select(Book).options(joinedload(Book.author))` runs only 1). Assert on `len(query_log)`.
4. Write a keyset helper `iterate_all_ids(session, size)` that yields every book id in pages of `size` until there are no more rows.
   Check that it returns all 60 ids in order.

**C. Find and fix the bug**

5. ```python
   def get_db():
       engine = create_engine(DATABASE_URL)
       with Session(engine) as session:
           yield session
   ```

### Solutions 3.3

1. 6 x (10 + 5) = 90 connections.
2. 51 queries: one for the authors and one per author.
3. and 4. See the cell below.
5. It creates a new engine (and a new pool) **on every request**, so pooling never works and connections pile up.
   Create the engine once at module level (or in the lifespan) and only create sessions per request.

In [21]:
# Solutions 3.3
from sqlalchemy.orm import joinedload

def titles_with_author_names():
    query_log.clear()
    with Session(perf_engine) as s:
        books = s.scalars(select(Book).options(joinedload(Book.author)).order_by(Book.id)).all()
        return [f"{b.title} by {b.author.name}" for b in books]

titles = titles_with_author_names()
print(titles[0], "| queries:", len(query_log))    # Book 0-0 by Author 0 | queries: 1
assert titles[0] == "Book 0-0 by Author 0" and len(titles) == 60
assert len(query_log) <= 2

def iterate_all_ids(session, size):
    last = 0
    while True:
        ids = page_keyset(session, after_id=last, size=size)
        if not ids:
            return
        yield from ids
        last = ids[-1]

with Session(perf_engine) as s:
    all_ids = list(iterate_all_ids(s, 7))
assert all_ids == list(range(1, 61))
print("Practice 3.3: all checks passed")

Book 0-0 by Author 0 | queries: 1
Practice 3.3: all checks passed


---

## 3.4 Response size and serialization

**Send less.** Large responses cost serialization time, bandwidth and client parsing time.

* Return only the fields the client needs (a small `response_model`, see Notebook 1, Part 6).
* Paginate lists (3.3, 5.3).
* Compress large text responses with **gzip** (or let the proxy do it).

**Serialize efficiently.** In older tutorials you see `default_response_class=ORJSONResponse` for speed. In FastAPI 0.162,
`ORJSONResponse` and `UJSONResponse` are **deprecated**. The deprecation message explains why: when an endpoint has a
**return type annotation or a `response_model`**, FastAPI now serializes the data **directly to JSON bytes with Pydantic**
(in Rust), which is fast and needs no custom response class. So the modern advice is simply: *declare the return type*.

In [22]:
# Declaring a return type lets Pydantic serialize straight to JSON bytes
import time
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

class Row(BaseModel):
    id: int
    name: str
    score: float
    tags: list[str]

rows = [Row(id=i, name=f"item {i}", score=i / 3, tags=["a", "b", "c"]) for i in range(5000)]
ser_app = FastAPI()

@ser_app.get("/typed")
def typed() -> list[Row]:                 # fast path: Pydantic -> JSON bytes
    return rows

@ser_app.get("/untyped")
def untyped():                            # no type: jsonable_encoder walks every object first
    return rows

client = TestClient(ser_app)
assert client.get("/typed").json() == client.get("/untyped").json()   # same JSON either way

def best_of(path, runs=5):
    times = []
    for _ in range(runs):
        t = time.perf_counter(); client.get(path); times.append(time.perf_counter() - t)
    return min(times)

t_typed, t_untyped = best_of("/typed"), best_of("/untyped")
print(f"typed:   {t_typed * 1000:.0f} ms")       # typed:   e.g. 10 ms
print(f"untyped: {t_untyped * 1000:.0f} ms")     # untyped: e.g. 70 ms (several times slower)

typed:   11 ms
untyped: 108 ms


In [23]:
# Compression: GZipMiddleware compresses responses above a size threshold
from fastapi.middleware.gzip import GZipMiddleware

gzip_app = FastAPI()
gzip_app.add_middleware(GZipMiddleware, minimum_size=1000)   # bytes; tiny responses are not worth it

@gzip_app.get("/rows")
def all_rows() -> list[Row]:
    return rows[:500]

r = TestClient(gzip_app).get("/rows", headers={"Accept-Encoding": "gzip"})
print(r.headers["content-encoding"])                         # gzip
print("compressed bytes on the wire:", r.headers["content-length"])     # e.g. 4500
print("JSON bytes after decompression:", len(r.content))                # e.g. 33000

gzip
compressed bytes on the wire: 3948
JSON bytes after decompression: 35916


> **Note:** gzip costs CPU on the server. Many teams let the reverse proxy or CDN compress instead. Never compress responses that
> mix secrets with attacker-controlled text over HTTPS without thinking about it (BREACH-style attacks); for normal JSON APIs it is fine.

> **Key points**
> * Smaller payloads: tight response models, pagination, compression for large text.
> * In FastAPI 0.162, declare return types; `ORJSONResponse` is deprecated because Pydantic already serializes to JSON bytes quickly.

---

## 3.5 Caching layers

A **cache** keeps a copy of a result so the next request does not repeat the work. Caches exist at several layers:

```text
  client/browser cache  ->  CDN / proxy cache  ->  app cache (memory or Redis)  ->  DB (its own buffer cache)
  (Cache-Control, ETag)     (public responses)     (computed results, lookups)
  nearest to the user, cheapest  ------------------------------------------->  most expensive to reach
```

| Layer | Good for | Watch out for |
|---|---|---|
| HTTP caching (`Cache-Control`, `ETag`) | public or per-user GET responses | never cache private data as `public` |
| In-process dict / TTL cache | small, hot, read-only data | per worker, lost on restart, memory growth |
| Redis | shared across workers and servers | network hop, invalidation, stampedes |

**ETag** = a fingerprint of the response. The client sends it back in `If-None-Match`; if nothing changed, the server answers
**304 Not Modified** with an empty body, saving bandwidth.

In [24]:
# HTTP caching with ETag and 304 Not Modified
import hashlib, json
from typing import Annotated
from fastapi import FastAPI, Header, Response
from fastapi.testclient import TestClient

etag_app = FastAPI()
catalog = {"version": 1, "items": ["pen", "book"]}

@etag_app.get("/catalog")
def get_catalog(response: Response, if_none_match: Annotated[str | None, Header()] = None):
    body = json.dumps(catalog, sort_keys=True).encode()
    etag = '"' + hashlib.sha256(body).hexdigest()[:16] + '"'
    if if_none_match == etag:
        return Response(status_code=304, headers={"ETag": etag})
    response.headers["ETag"] = etag
    response.headers["Cache-Control"] = "public, max-age=60"
    return catalog

client = TestClient(etag_app)
first = client.get("/catalog")
print(first.status_code, first.headers["etag"])                         # 200 "<16 hex chars>"
second = client.get("/catalog", headers={"If-None-Match": first.headers["etag"]})
print(second.status_code, len(second.content))                          # 304 0
catalog["items"].append("bag")                                          # data changes -> new ETag
third = client.get("/catalog", headers={"If-None-Match": first.headers["etag"]})
print(third.status_code)                                                # 200

200 "74f2c670e630f1ed"
304 0
200


**App-level cache-aside** (the most common pattern): look in the cache; on a miss, compute, store with a **TTL**
(time to live), return. With Redis it looks like this (not run here, no Redis server):

```python
import json
import redis.asyncio as redis

r = redis.from_url("redis://localhost:6379/0")      # create once, in the lifespan

async def get_product(product_id: int) -> dict:
    key = f"product:{product_id}"
    if (cached := await r.get(key)) is not None:
        return json.loads(cached)                        # hit
    product = await load_product_from_db(product_id)     # miss
    await r.set(key, json.dumps(product), ex=300)        # expire after 5 minutes
    return product
```

The two hard problems:

* **Invalidation:** when data changes, delete the key (`await r.delete(key)`) or accept staleness for the TTL.
* **Stampede:** a hot key expires and 500 requests miss at once and all hit the DB. Fixes: a lock so one request
  recomputes, random jitter on TTLs, or refreshing early.

Part 8 builds an in-memory TTL cache for LLM answers; Notebook 2, Part 10 covered caching basics.

### Practice 3.5: caching

**A. Predict:** a client sends `If-None-Match` with an old ETag after the data changed. What status does it get?

**B. Write code:** write a class `TTLCache` with `get(key)`, `set(key, value)` and a `ttl` in seconds, using an
injectable clock function (`clock=time.monotonic`) so tests do not need to sleep. Then use it in a FastAPI endpoint
`/square/{n}` that counts how many times it really computed. Two calls -> 1 computation; after the TTL -> 2.

**C. Find the bug:** `@lru_cache` on an endpoint that returns the current user's orders. What goes wrong?

### Solutions 3.5

**A.** 200 with the new body and a new ETag.

**B.** See the cell below. The fake clock is a list we change by hand.

**C.** `lru_cache` keys only on the function arguments and never expires. If the user is not part of the arguments,
user A can receive user B's orders; even if it is, new orders never appear and memory grows. Cache per user with a TTL,
and invalidate on writes.

In [25]:
# Solutions 3.5
import time
from fastapi import FastAPI
from fastapi.testclient import TestClient

class TTLCache:
    def __init__(self, ttl: float, clock=time.monotonic):
        self.ttl, self.clock, self._data = ttl, clock, {}

    def get(self, key):
        item = self._data.get(key)
        if item is None:
            return None
        value, expires = item
        if self.clock() >= expires:
            del self._data[key]
            return None
        return value

    def set(self, key, value):
        self._data[key] = (value, self.clock() + self.ttl)

now = [0.0]
cache = TTLCache(ttl=10, clock=lambda: now[0])
computed = {"n": 0}
cache_app = FastAPI()

@cache_app.get("/square/{n}")
def square(n: int):
    if (hit := cache.get(n)) is not None:
        return {"value": hit, "cached": True}
    computed["n"] += 1
    cache.set(n, n * n)
    return {"value": n * n, "cached": False}

c = TestClient(cache_app)
assert c.get("/square/4").json() == {"value": 16, "cached": False}
assert c.get("/square/4").json() == {"value": 16, "cached": True}
assert computed["n"] == 1
now[0] = 11                                  # time passes beyond the TTL
assert c.get("/square/4").json()["cached"] is False and computed["n"] == 2
print("Practice 3.5: all checks passed")

Practice 3.5: all checks passed


---

## 3.6 Load testing

A **load test** sends many requests, like real users, and measures **throughput** (requests per second, RPS),
**latency percentiles** and the **error rate**.

* **p50** (median): half the requests were faster. **p95/p99**: 95%/99% were faster. Users notice the slow tail,
  so set targets on p95/p99, not on the average.
* Test against a production-like setup (same workers, real DB, behind the proxy), never against `--reload`.
* Increase load step by step and watch where latency jumps: that is your capacity.

**Locust** (Python) and **k6** (JavaScript scripts, Go engine) are popular; `hey` and `oha` are quick one-liners.

```python
# locustfile.py
from locust import HttpUser, task, between

class ApiUser(HttpUser):
    wait_time = between(0.5, 2)

    def on_start(self):
        r = self.client.post("/auth/login", data={"username": "asha", "password": "secret123"})
        self.client.headers["Authorization"] = f"Bearer {r.json()['access_token']}"

    @task(3)
    def list_tasks(self):
        self.client.get("/tasks?limit=20")

    @task(1)
    def create_task(self):
        self.client.post("/tasks", json={"title": "load test"})
```

```bash
locust -f locustfile.py --host http://localhost:8000 --users 200 --spawn-rate 20 --run-time 2m --headless
```

```javascript
// k6 script: k6 run load.js
import http from "k6/http";
import { check } from "k6";
export const options = { vus: 100, duration: "1m", thresholds: { http_req_duration: ["p(95)<300"] } };
export default function () {
  const r = http.get("http://localhost:8000/healthz");
  check(r, { "status is 200": (res) => res.status === 200 });
}
```

```bash
oha -z 30s -c 50 http://localhost:8000/healthz      # 50 connections for 30 seconds
```

Inside the notebook we can build a **tiny** load tester with `asyncio` to understand the numbers (it is not a replacement
for a real tool, because client and app share one process).

In [26]:
# A tiny in-process load test: concurrency, RPS and percentiles
import asyncio, random, statistics, time
import httpx
from fastapi import FastAPI

load_app = FastAPI()

@load_app.get("/work")
async def work():
    await asyncio.sleep(random.uniform(0.005, 0.02))   # pretend DB time 5-20 ms
    if random.random() < 0.01:
        await asyncio.sleep(0.1)                       # 1% slow requests: the "tail"
    return {"ok": True}

def percentile(values, p):
    ordered = sorted(values)
    index = max(0, min(len(ordered) - 1, round(p / 100 * len(ordered)) - 1))
    return ordered[index]

async def load_test(app, path, total=400, concurrency=50):
    latencies, errors = [], 0
    sem = asyncio.Semaphore(concurrency)
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://t") as client:
        async def one():
            nonlocal errors
            async with sem:
                t = time.perf_counter()
                r = await client.get(path)
                latencies.append(time.perf_counter() - t)
                errors += r.status_code >= 500
        start = time.perf_counter()
        await asyncio.gather(*(one() for _ in range(total)))
        elapsed = time.perf_counter() - start
    return {"rps": round(total / elapsed), "p50_ms": round(percentile(latencies, 50) * 1000),
            "p95_ms": round(percentile(latencies, 95) * 1000), "p99_ms": round(percentile(latencies, 99) * 1000),
            "errors": errors}

random.seed(1)
print(await load_test(load_app, "/work"))
# e.g. {'rps': 2000, 'p50_ms': 15, 'p95_ms': 25, 'p99_ms': 30, 'errors': 0}
# (with only 4 slow requests out of 400, the slow tail shows up in p99 or max on some runs, not all)

{'rps': 1081, 'p50_ms': 36, 'p95_ms': 51, 'p99_ms': 56, 'errors': 0}


### Practice 3.6: load testing

**A. Predict:** latencies (ms) are `[10]*98 + [500, 900]`. What are the mean and the p95? Which one describes users better?

**B. Write code**

1. Using `percentile` above, assert that p50 of `list(range(1, 101))` is 50 and p99 is 99.
2. Write `summarize(latencies_ms)` returning `{"mean": ..., "p95": ..., "max": ...}` (rounded to 1 decimal) and check it on the list from A.
3. Run `load_test` on `perf_app` path `/async-blocking` with `total=10, concurrency=10`. Print the result. Assert that the
   throughput is at most 6 requests per second (10 requests that block the loop for 0.2 s each need about 2 s).
   Is the reported p50 latency honest? Why not?

**C. Explain:** why is a load test against `fastapi dev` misleading?

### Solutions 3.6

**A.** Mean = (980 + 500 + 900) / 100 = 23.8 ms; p95 = 10 ms (only the top 2% are slow). The mean is pulled up by two
outliers, while p95 shows that most users are fast; p99/max show the bad tail. Look at several percentiles, not just the average.

**B.** See the cell below.

**B3.** The p50 latency looks fine (about 200 ms), but that is a lie of the in-process test: the *client* runs in the same
blocked event loop, so it cannot even start the next request (and its timer) until the previous one finishes. A real load
tool in another process would see latencies up to 2 s. Throughput (about 5 RPS) tells the truth. This is why real load tests
run the client in a separate process or machine.

**C.** `fastapi dev` runs one process with auto-reload and file watching, on `127.0.0.1`, without your production workers,
proxy or database setup. The numbers say nothing about production capacity.

In [27]:
# Solutions 3.6
import statistics

assert percentile(list(range(1, 101)), 50) == 50
assert percentile(list(range(1, 101)), 99) == 99

def summarize(latencies_ms):
    return {"mean": round(statistics.mean(latencies_ms), 1),
            "p95": round(percentile(latencies_ms, 95), 1),
            "max": round(max(latencies_ms), 1)}

s = summarize([10] * 98 + [500, 900])
print(s)                            # {'mean': 23.8, 'p95': 10, 'max': 900}
assert s == {"mean": 23.8, "p95": 10, "max": 900}

res = await load_test(perf_app, "/async-blocking", total=10, concurrency=10)
print(res)                           # e.g. {'rps': 5, 'p50_ms': 201, 'p95_ms': 202, 'p99_ms': 202, 'errors': 0}
assert res["rps"] <= 6
print("Practice 3.6: all checks passed")

{'mean': 23.8, 'p95': 10, 'max': 900}


{'rps': 5, 'p50_ms': 202, 'p95_ms': 203, 'p99_ms': 203, 'errors': 0}
Practice 3.6: all checks passed


---

## How to debug performance problems

| Symptom | Likely cause | What to check |
|---|---|---|
| Everything slows down together, even `/healthz` | a blocking call inside `async def` | `py-spy dump`, search for `time.sleep`, `requests.`, sync DB calls in async code |
| Slow only under load, fine alone | thread pool or DB pool exhausted | pool size, `QueuePool limit ... reached` errors, thread pool limit (40) |
| One list endpoint is slow | N+1 queries, no pagination, missing index | count queries, `EXPLAIN` the SQL |
| High CPU in the app | big JSON responses, untyped returns, heavy Python loops | profile with pyinstrument; declare return types; paginate |
| p99 much higher than p50 | occasional slow dependency, GC pauses, cold caches | per-dependency timings, timeouts, tracing (Part 4) |
| "Too many connections" from Postgres | workers x pool size too high | reduce pool sizes or add PgBouncer |

## Summary

* Measure first; most time is usually spent waiting on the DB and other services.
* Never block the event loop; offload blocking code to threads and CPU work to processes or other services.
* Reuse pooled connections; fix N+1; paginate; index.
* Send less data, declare return types (Pydantic serializes to JSON bytes; `ORJSONResponse` is deprecated in 0.162).
* Cache at the right layer with a TTL and an invalidation plan; load test with Locust/k6 and judge by p95/p99.

# Part 4: Observability

**What this Part covers**

* The three signals: logs, metrics and traces (plus error tracking).
* Structured JSON logging with a request ID that follows each request.
* Metrics in the Prometheus style and a `/metrics` endpoint.
* Distributed tracing with OpenTelemetry, and FastAPI's **built-in** OpenTelemetry support (`fastapi.telemetry`, FastAPI 0.162).
* Error tracking with Sentry.
* What to alert on, and what not to.

**Before you start:** Notebook 2, Part 6 (middleware) and Part 9 (error handling and logging).

## 4.1 Logs, metrics, traces

**Observability** means: from the outside (logs, numbers, traces) you can understand what the system is doing inside,
especially when something goes wrong at 3 a.m.

| Signal | What it is | Answers | Example tools |
|---|---|---|---|
| **Logs** | timestamped events, one line each | "What happened to request X?" | stdout -> Loki, ELK, CloudWatch |
| **Metrics** | numbers aggregated over time | "How many errors per minute? Is p95 rising?" | Prometheus + Grafana, Datadog |
| **Traces** | the path of one request through services, with timings | "Which call made this request slow?" | OpenTelemetry -> Jaeger, Tempo, Honeycomb |
| **Error tracking** | exceptions grouped with stack traces | "Which bug is hurting the most users?" | Sentry |

```text
            request id / trace id ties them together
  logs:    {"request_id": "ab12", "msg": "order created", "duration_ms": 41}
  metrics: http_requests_total{route="/orders", status="201"} 1532
  traces:  POST /orders [41ms] -> dependencies [3ms] -> endpoint [35ms] -> INSERT orders [20ms]
```

> **Key points**
> * Logs tell the story of one request; metrics show trends; traces show where time went across services.
> * A shared ID (request ID or trace ID) lets you jump from an alert to the exact logs and trace.

---

## 4.2 Structured JSON logging with request IDs

Plain text logs (`INFO order created for 42`) are hard to search. **Structured logs** are JSON objects with fixed fields,
so a log system can filter `status=500 AND route="/orders"`.

A **request ID** is a unique ID for one request. The middleware below:

1. reads `X-Request-ID` from the incoming request (a proxy or another service may have set it), or creates one;
2. stores it in a **context variable** (`contextvars.ContextVar`), which is like a global variable but separate for each
   request, even when many requests run concurrently on one event loop;
3. adds it to every log line through a logging filter, and returns it in the response header.

In [28]:
# JSON logs with a request ID, captured into a string so we can look at them
import io, json, logging, time, uuid
from contextvars import ContextVar
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

request_id_var: ContextVar[str] = ContextVar("request_id", default="-")

class RequestIdFilter(logging.Filter):
    def filter(self, record):
        record.request_id = request_id_var.get()
        return True

class JsonFormatter(logging.Formatter):
    def format(self, record):
        data = {"level": record.levelname, "logger": record.name, "msg": record.getMessage(),
                "request_id": getattr(record, "request_id", "-")}
        data.update(getattr(record, "extra_fields", {}))
        if record.exc_info:
            data["exc"] = self.formatException(record.exc_info).splitlines()[-1]
        return json.dumps(data)

log_stream = io.StringIO()                          # in production: sys.stdout
handler = logging.StreamHandler(log_stream)
handler.setFormatter(JsonFormatter())
handler.addFilter(RequestIdFilter())
log = logging.getLogger("obs_demo")
log.handlers = [handler]
log.setLevel(logging.INFO)
log.propagate = False

obs_app = FastAPI()

@obs_app.middleware("http")
async def request_context(request: Request, call_next):
    rid = request.headers.get("X-Request-ID") or uuid.uuid4().hex[:12]
    token = request_id_var.set(rid)
    start = time.perf_counter()
    try:
        response = await call_next(request)
        route = request.scope.get("route")
        log.info("request finished", extra={"extra_fields": {
            "method": request.method, "route": route.path if route else request.url.path,
            "status": response.status_code, "duration_ms": round((time.perf_counter() - start) * 1000, 1)}})
        response.headers["X-Request-ID"] = rid
        return response
    finally:
        request_id_var.reset(token)

@obs_app.get("/orders/{order_id}")
def get_order(order_id: int):
    log.info("loading order", extra={"extra_fields": {"order_id": order_id}})
    return {"id": order_id}

client = TestClient(obs_app)
r = client.get("/orders/7", headers={"X-Request-ID": "req-abc"})
print(r.headers["x-request-id"])                    # req-abc
client.get("/orders/8")                             # no header: a new ID is generated
for line in log_stream.getvalue().splitlines():
    print(line)
# {"level": "INFO", "logger": "obs_demo", "msg": "loading order", "request_id": "req-abc", "order_id": 7}
# {"level": "INFO", "logger": "obs_demo", "msg": "request finished", "request_id": "req-abc", "method": "GET", "route": "/orders/{order_id}", "status": 200, "duration_ms": ...}
# {"level": "INFO", ... "request_id": "<12 random hex chars>", "order_id": 8}
# ...

req-abc
{"level": "INFO", "logger": "obs_demo", "msg": "loading order", "request_id": "req-abc", "order_id": 7}
{"level": "INFO", "logger": "obs_demo", "msg": "request finished", "request_id": "req-abc", "method": "GET", "route": "/orders/{order_id}", "status": 200, "duration_ms": 1.4}
{"level": "INFO", "logger": "obs_demo", "msg": "loading order", "request_id": "74e795dadf73", "order_id": 8}
{"level": "INFO", "logger": "obs_demo", "msg": "request finished", "request_id": "74e795dadf73", "method": "GET", "route": "/orders/{order_id}", "status": 200, "duration_ms": 1.8}


Notice:

* The endpoint did not pass the request ID around: the context variable carried it.
* The log uses the **route template** `/orders/{order_id}`, not `/orders/7`, so logs and metrics group nicely.
* Pass the request ID on to other services (`headers={"X-Request-ID": rid}`) so their logs share it.

**What not to log:** passwords, tokens, API keys, full card numbers, health data and other personal data (PII).
Log IDs instead of names and emails. Healthcare and finance apps often have legal rules about this.

> **Tip:** popular libraries do this for you: `structlog`, `python-json-logger`, or `loguru`. The ideas stay the same:
> JSON to stdout, one line per event, request ID on every line.

> **Interview tip:** "How do you debug a production issue reported by a user?" Ask for the request ID (return it in an
> `X-Request-ID` header and in error bodies), search the logs for it, open the matching trace, find the failing step.

> **Key points**
> * Log JSON to stdout, one event per line, with level, message, request ID and useful fields.
> * Use a `ContextVar` + middleware so every log line gets the request ID automatically.
> * Never log secrets or personal data.

### Practice 4.2: structured logging

**A. Predict**

1. Why use a `ContextVar` for the request ID instead of a normal global variable?
2. A client sends `X-Request-ID: abc`. What header comes back?

**B. Write code**

3. Write a function `redact(data: dict) -> dict` that replaces the values of keys `password`, `token`, `authorization`
   and `api_key` (any letter case) with `"***"`, also inside nested dicts. Check it.
4. Add an endpoint `/fail` to a new app `fail_app` that raises `ValueError("boom")`. Unexpected errors must be logged with
   `log.exception(...)` (with the request ID on the log line) and return 500 with
   `{"detail": "Internal server error", "request_id": <id>}` and the `X-Request-ID` header.
   Hint: first try `@fail_app.exception_handler(Exception)` and look at the request ID you get. Then explain what you see.

**C. Find and fix the bug**

5. ```python
   @app.middleware("http")
   async def add_id(request, call_next):
       request_id_var.set(uuid.uuid4().hex)
       return await call_next(request)
   ```
   The logs show the right IDs, but a reviewer says two things are missing. What are they?

### Solutions 4.2

1. Many requests run at the same time on one event loop. A global would be overwritten by whichever request ran last.
   A `ContextVar` keeps a separate value per request (per asyncio task).
2. `X-Request-ID: abc`: the middleware reuses the incoming ID.
3. See the cell below.
4. A handler for `Exception` (or status 500) is special in Starlette: it runs in `ServerErrorMiddleware`, the **outermost**
   layer, *outside* your `@app.middleware("http")`. By the time it runs, your middleware has already reset the context variable,
   so the handler sees `"-"` and the response gets no `X-Request-ID` header. The fix used below: catch unexpected exceptions
   **inside** the request-ID middleware, log them there, and build the 500 response there. (Handlers for `HTTPException` and
   your own exception classes run *inside* the middleware, so they do see the ID.)
   With `raise_server_exceptions=False`, TestClient returns the 500 response instead of re-raising the error.
5. It ignores an incoming `X-Request-ID` (so IDs from the proxy or the caller are lost), and it never returns the ID in the
   response header. (Resetting the variable with the token is also good practice.)

In [29]:
# Solutions 4.2
import uuid
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient

SENSITIVE = {"password", "token", "authorization", "api_key"}

def redact(data: dict) -> dict:
    clean = {}
    for key, value in data.items():
        if key.lower() in SENSITIVE:
            clean[key] = "***"
        elif isinstance(value, dict):
            clean[key] = redact(value)
        else:
            clean[key] = value
    return clean

assert redact({"user": "asha", "Password": "x", "meta": {"API_KEY": "k", "n": 1}}) == \
    {"user": "asha", "Password": "***", "meta": {"API_KEY": "***", "n": 1}}

# First attempt: an exception handler for Exception -> the request ID is lost
first_try = FastAPI()

@first_try.middleware("http")
async def ctx_first(request: Request, call_next):
    token = request_id_var.set(request.headers.get("X-Request-ID") or uuid.uuid4().hex[:12])
    try:
        return await call_next(request)
    finally:
        request_id_var.reset(token)

@first_try.exception_handler(Exception)
async def unhandled(request: Request, exc: Exception):
    return JSONResponse({"detail": "Internal server error", "request_id": request_id_var.get()}, status_code=500)

@first_try.get("/fail")
def fail_first():
    raise ValueError("boom")

r = TestClient(first_try, raise_server_exceptions=False).get("/fail", headers={"X-Request-ID": "req-500"})
print(r.json(), r.headers.get("x-request-id"))   # {'detail': 'Internal server error', 'request_id': '-'} None

# Fix: handle unexpected errors inside the request-ID middleware
fail_app = FastAPI()

@fail_app.middleware("http")
async def ctx(request: Request, call_next):
    rid = request.headers.get("X-Request-ID") or uuid.uuid4().hex[:12]
    token = request_id_var.set(rid)
    try:
        response = await call_next(request)
    except Exception:
        log.exception("unhandled error")
        response = JSONResponse({"detail": "Internal server error", "request_id": rid}, status_code=500)
    finally:
        request_id_var.reset(token)
    response.headers["X-Request-ID"] = rid
    return response

@fail_app.get("/fail")
def fail():
    raise ValueError("boom")

log_stream.truncate(0); log_stream.seek(0)
r = TestClient(fail_app, raise_server_exceptions=False).get("/fail", headers={"X-Request-ID": "req-500"})
print(r.status_code, r.json())          # 500 {'detail': 'Internal server error', 'request_id': 'req-500'}
assert r.status_code == 500 and r.json()["request_id"] == "req-500"
assert r.headers["x-request-id"] == "req-500"
last = json.loads(log_stream.getvalue().splitlines()[-1])
print(last["exc"])                      # ValueError: boom
assert last["request_id"] == "req-500"
print("Practice 4.2: all checks passed")

{'detail': 'Internal server error', 'request_id': '-'} None
500 {'detail': 'Internal server error', 'request_id': 'req-500'}
ValueError: boom
Practice 4.2: all checks passed


---

## 4.3 Metrics: Prometheus concepts and `/metrics`

**Prometheus** is a popular open-source metrics system. It **pulls** ("scrapes") a `/metrics` URL from each app every
15-30 seconds and stores the numbers as time series. Grafana draws dashboards from them.

Metric types:

| Type | Meaning | Example |
|---|---|---|
| **Counter** | only goes up | `http_requests_total` |
| **Gauge** | goes up and down | `http_requests_in_progress`, queue length |
| **Histogram** | counts values into buckets, for percentiles | `http_request_duration_seconds` |

**Labels** split a metric: `http_requests_total{method="GET", route="/orders/{order_id}", status="200"}`.
Every unique label combination is a separate time series.

> **Warning:** never use unbounded values as labels (user IDs, raw paths like `/orders/7`, error messages).
> This "high cardinality" creates millions of series and can take down Prometheus. Use the route template.

The **RED method** for every service: **R**ate (requests/s), **E**rrors (failed requests/s), **D**uration (latency histogram).

In a real app you use the `prometheus_client` library (or `prometheus-fastapi-instrumentator`):

```python
from prometheus_client import Counter, Histogram, make_asgi_app

REQUESTS = Counter("http_requests_total", "Requests", ["method", "route", "status"])
LATENCY = Histogram("http_request_duration_seconds", "Latency", ["route"])
app.mount("/metrics", make_asgi_app())          # Prometheus scrapes this URL
```

With several workers, each process has its own counters; `prometheus_client` has a "multiprocess mode" for that, or you
run one worker per container and let Prometheus scrape each container.

To understand what the library does, here is a small hand-written version that produces the Prometheus text format:

In [30]:
# A minimal metrics registry + middleware + /metrics endpoint (Prometheus text format)
import bisect, time
from collections import defaultdict
from fastapi import FastAPI, Request
from fastapi.responses import PlainTextResponse
from fastapi.testclient import TestClient

BUCKETS = [0.005, 0.01, 0.05, 0.1, 0.5, 1.0, float("inf")]

class Metrics:
    def __init__(self):
        self.requests = defaultdict(int)                       # (method, route, status) -> count
        self.duration = defaultdict(lambda: [0] * len(BUCKETS))  # route -> bucket counts
        self.in_progress = 0

    def observe(self, method, route, status, seconds):
        self.requests[(method, route, str(status))] += 1
        counts = self.duration[route]
        for i in range(bisect.bisect_left(BUCKETS, seconds), len(BUCKETS)):
            counts[i] += 1                                       # histogram buckets are cumulative

    def render(self) -> str:
        lines = ["# TYPE http_requests_total counter"]
        for (m, r, s), n in sorted(self.requests.items()):
            lines.append(f'http_requests_total{{method="{m}",route="{r}",status="{s}"}} {n}')
        lines.append("# TYPE http_request_duration_seconds histogram")
        for route, counts in sorted(self.duration.items()):
            for le, n in zip(BUCKETS, counts):
                le_txt = "+Inf" if le == float("inf") else le
                lines.append(f'http_request_duration_seconds_bucket{{route="{route}",le="{le_txt}"}} {n}')
        lines.append("# TYPE http_requests_in_progress gauge")
        lines.append(f"http_requests_in_progress {self.in_progress}")
        return "\n".join(lines) + "\n"

metrics = Metrics()
metrics_app = FastAPI()

@metrics_app.middleware("http")
async def measure(request: Request, call_next):
    if request.url.path == "/metrics":
        return await call_next(request)                     # do not measure the scraper itself
    metrics.in_progress += 1
    start = time.perf_counter()
    status = 500
    try:
        response = await call_next(request)
        status = response.status_code
        return response
    finally:
        metrics.in_progress -= 1
        route = request.scope.get("route")
        metrics.observe(request.method, route.path if route else "unmatched", status, time.perf_counter() - start)

@metrics_app.get("/items/{item_id}")
def item(item_id: int):
    return {"id": item_id}

@metrics_app.get("/metrics", response_class=PlainTextResponse)
def prometheus_metrics():
    return metrics.render()

client = TestClient(metrics_app)
for i in [1, 2, 3]:
    client.get(f"/items/{i}")
client.get("/items/abc")                                   # 422
client.get("/nope")                                        # 404, no route
text = client.get("/metrics").text
print("\n".join(line for line in text.splitlines() if "requests_total" in line))
# # TYPE http_requests_total counter
# http_requests_total{method="GET",route="/items/{item_id}",status="200"} 3
# http_requests_total{method="GET",route="/items/{item_id}",status="422"} 1
# http_requests_total{method="GET",route="unmatched",status="404"} 1

# TYPE http_requests_total counter
http_requests_total{method="GET",route="/items/{item_id}",status="200"} 3
http_requests_total{method="GET",route="/items/{item_id}",status="422"} 1
http_requests_total{method="GET",route="unmatched",status="404"} 1


Three requests to three different URLs became **one** series, because we used the route template. Unknown paths are grouped as
`unmatched` so a scanner hitting random URLs cannot create new series.

Typical Prometheus queries (PromQL) on top of such metrics:

```text
sum(rate(http_requests_total[5m]))                                        # requests per second
sum(rate(http_requests_total{status=~"5.."}[5m])) / sum(rate(http_requests_total[5m]))   # error ratio
histogram_quantile(0.95, sum by (le) (rate(http_request_duration_seconds_bucket[5m])))  # p95 latency
```

> **Note:** protect `/metrics`: serve it on an internal port or network, or require a token. It can reveal routes and traffic.

> **Key points**
> * Counters, gauges, histograms; labels with **bounded** values only (route template, method, status class).
> * RED: rate, errors, duration for every service.
> * Use `prometheus_client` (or an instrumentator package) in real apps; mind multiple workers.

### Practice 4.3: metrics

**A. Predict:** someone changes the middleware to use `request.url.path` as the route label. After 1 million requests to
`/items/1` ... `/items/1000000`, how many series exist for `http_requests_total`?

**B. Write code**

1. Using `metrics.requests`, write `error_ratio(metrics) -> float`: the share of requests with status `5xx`.
   Then make `metrics_app` return a 500 once (add a `/crash` route that raises, and call it with
   `TestClient(metrics_app, raise_server_exceptions=False)`) and check the ratio is above 0.
2. Write `count_le(metrics, route, seconds)` that returns how many requests to `route` took at most `seconds`
   (use the bucket whose upper bound is `seconds`). Check that all 4 `/items/{item_id}` requests took at most 1 second.

**C. Explain:** why is `user_id` a bad label, but `status="500"` a good one?

### Solutions 4.3

**A.** About 1 million series (one per URL): a cardinality explosion.

**B.** See the cell below.

**C.** `user_id` is unbounded: every new user creates new series. `status` has a small, fixed set of values.

In [31]:
# Solutions 4.3
from fastapi.testclient import TestClient

def error_ratio(m) -> float:
    total = sum(m.requests.values())
    errors = sum(n for (_, _, status), n in m.requests.items() if status.startswith("5"))
    return errors / total if total else 0.0

@metrics_app.get("/crash")
def crash():
    raise RuntimeError("bug")

assert error_ratio(metrics) == 0.0
TestClient(metrics_app, raise_server_exceptions=False).get("/crash")
print(round(error_ratio(metrics), 3))     # 0.167  (1 of 6 requests)
assert error_ratio(metrics) > 0

def count_le(m, route, seconds):
    return m.duration[route][BUCKETS.index(seconds)]

assert count_le(metrics, "/items/{item_id}", 1.0) == 4
print("Practice 4.3: all checks passed")

0.167
Practice 4.3: all checks passed


---

## 4.4 Tracing with OpenTelemetry (and FastAPI's built-in support)

A **trace** is the full journey of one request. It is made of **spans**: timed steps with a name and attributes
(`GET /orders/{id}` -> `SELECT orders` -> `POST payment-service/charge`). **Context propagation** passes the trace ID to
other services in the `traceparent` HTTP header, so one trace can cross many services.

**OpenTelemetry (OTel)** is the vendor-neutral standard for traces, metrics and logs. You instrument once and send data
(usually with the **OTLP** protocol) to Jaeger, Grafana Tempo, Honeycomb, Datadog and others.

```text
 trace 4bf92f... ------------------------------------------------------------ 120 ms
   span  GET /orders/{order_id}                         [==========================]
   span    fastapi.dependencies                          [=]
   span    fastapi.endpoint                                [==================]
   span      SELECT orders  (SQLAlchemy instrumentation)     [========]
   span      POST payments  (httpx instrumentation)                [=======]
   span    fastapi.serialization                                            [=]
```

### `fastapi.telemetry` in FastAPI 0.162

Older tutorials add tracing with the separate package `opentelemetry-instrumentation-fastapi`
(`FastAPIInstrumentor.instrument_app(app)`). FastAPI 0.162 (a recent addition) has **native OpenTelemetry support** built in.
What the installed source shows:

* `FastAPI(telemetry={...})` takes an optional `TelemetryConfig` dict: `tracer_provider`, `meter_provider`, `logger_provider`,
  `tracing`, `metrics`, `logs`, `operation_spans`, `auto_configure`, `exclude` (all optional).
* It is **active only when an OpenTelemetry provider is configured** (globally, or passed in the config). With no provider set
  up, it does nothing, so normal apps pay no cost.
* When active it creates a server span per request named like `GET /items/{item_id}` with standard attributes
  (`http.route`, `http.response.status_code`, ...), child spans `fastapi.dependencies`, `fastapi.endpoint`,
  `fastapi.serialization` (and background tasks), the metrics `http.server.request.duration` and `http.server.active_requests`,
  and log records for validation failures and unhandled exceptions.
* `auto_configure` (or the environment variable `FASTAPI_OTEL_AUTO_CONFIGURE=true`) adds OTLP exporters from the standard
  `OTEL_EXPORTER_OTLP_ENDPOINT` variables.
* If the old `opentelemetry-instrumentation-fastapi` middleware is detected, FastAPI avoids duplicating its work.
* `fastapi.telemetry.get_telemetry_data()` lets custom OpenTelemetry processors read request data (request, body, errors) while
  the request is running.

Because the API is new, check the FastAPI release notes before relying on details. Here it is in action with an in-memory exporter
(the OpenTelemetry SDK is installed with `fastapi[standard]` in this environment):

In [32]:
# FastAPI's native OpenTelemetry spans, captured in memory
from fastapi import FastAPI
from fastapi.testclient import TestClient
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()
provider = TracerProvider()                        # we pass it in; we do NOT set a global provider
provider.add_span_processor(SimpleSpanProcessor(exporter))

traced_app = FastAPI(telemetry={"tracer_provider": provider, "metrics": False, "logs": False})

@traced_app.get("/items/{item_id}")
def read_item(item_id: int):
    return {"id": item_id}

TestClient(traced_app).get("/items/3")
spans = exporter.get_finished_spans()
for span in spans:
    print(span.name)
# fastapi.dependencies
# fastapi.endpoint
# fastapi.serialization
# GET /items/{item_id}
server = spans[-1]
print(server.attributes["http.route"], server.attributes["http.response.status_code"])   # /items/{item_id} 200
print(all(s.context.trace_id == server.context.trace_id for s in spans))                 # True (one trace)

fastapi.dependencies
fastapi.endpoint
fastapi.serialization
GET /items/{item_id}
/items/{item_id} 200
True


In production you would configure a provider once at startup and export with OTLP, for example:

```bash
export FASTAPI_OTEL_AUTO_CONFIGURE=true
export OTEL_SERVICE_NAME=task-api
export OTEL_EXPORTER_OTLP_ENDPOINT=http://otel-collector:4318
fastapi run app/main.py
```

Add instrumentations for the libraries you call (`opentelemetry-instrumentation-sqlalchemy`, `-httpx`) so DB queries and
outgoing HTTP calls become child spans, and **sample** traces (for example keep 10%) on busy services to control cost.

> **Tip:** put the trace ID in your JSON logs (from `opentelemetry.trace.get_current_span().get_span_context().trace_id`)
> so you can jump from a log line to its trace.

> **Key points**
> * Trace = tree of spans for one request, across services via `traceparent`.
> * OpenTelemetry is the standard; FastAPI 0.162 has native support via `FastAPI(telemetry=...)`, active when a provider exists.
> * Instrument DB and HTTP clients too; sample in production.

---

## 4.5 Error tracking with Sentry

**Sentry** collects unhandled exceptions with stack traces, request data and release versions, groups duplicates, and alerts
you. Its Python SDK detects FastAPI/Starlette automatically.

```python
import sentry_sdk

sentry_sdk.init(
    dsn=settings.sentry_dsn,          # from the environment; no DSN = disabled
    environment=settings.environment, # "prod", "staging"
    release="task-api@1.4.2",         # ties errors to a deploy
    traces_sample_rate=0.1,           # 10% performance traces
    send_default_pii=False,           # do not send user data unless you decide to
)
app = FastAPI()
```

* Call `init` **before** creating the app.
* Errors you handle yourself (`HTTPException(404)`) are not reported; unhandled exceptions (500s) are.
* Scrub sensitive data (`before_send` hook) in healthcare and finance apps.

> **Key points:** Sentry groups exceptions so you fix the most common bug first; tag errors with environment and release.

---

## 4.6 What to alert on

An **alert** wakes a person up, so every alert must be **urgent, actionable and real**. Good practice:

* Alert on **symptoms users feel**, not on every cause: high error rate, high p95/p99 latency, the service being down.
  Causes (CPU at 80%, one pod restarted) go on dashboards.
* Base alerts on **SLOs** (service level objectives), for example "99.9% of requests succeed and p95 < 300 ms over 30 days".
* Use a time window (`for: 5m`) so one spike does not page anyone.

| Alert | Example rule | Why |
|---|---|---|
| Error rate | 5xx ratio > 2% for 5 minutes | users see failures |
| Latency | p95 > 500 ms for 10 minutes | users see slowness |
| Availability | `/readyz` failing on all replicas | outage |
| Saturation | DB pool waits, queue length growing for 15 minutes | outage soon |
| Dead man's switch | no successful background job in 1 hour | silent failure |
| Certificates | TLS certificate expires in < 14 days | avoid an easy outage |

```yaml
# Prometheus alert rule (reference)
- alert: HighErrorRate
  expr: sum(rate(http_requests_total{status=~"5.."}[5m])) / sum(rate(http_requests_total[5m])) > 0.02
  for: 5m
  labels: {severity: page}
  annotations: {summary: "More than 2% of requests fail"}
```

### Practice 4.4-4.6: tracing and alerts

**A. Predict**

1. You create `FastAPI()` without `telemetry=` and without configuring any OpenTelemetry provider. Are spans created?
2. Should "CPU above 85% for 1 minute" page the on-call engineer at night?

**B. Write code**

3. Create a new app with the same in-memory exporter setup, plus `exclude` in the telemetry config so that `/healthz` is not traced
   (the function receives the ASGI scope; return `True` to skip). Call `/healthz` and `/work` and check that only `/work` produced spans.
4. Write `should_alert(samples, threshold, for_minutes)` where `samples` is a list of per-minute error ratios. Return `True` only if the
   **last** `for_minutes` samples are all above the threshold. Check: `[0.01, 0.05, 0.06, 0.07]` with `(0.02, 3)` is `True`;
   `[0.05, 0.01, 0.06]` with `(0.02, 2)` is `False`.

**C. Explain:** why exclude health checks from tracing?

### Solutions 4.4-4.6

1. No. Native telemetry is active only when a provider is configured (or passed in).
2. No. High CPU alone is a cause, not a user-facing symptom; put it on a dashboard. Page on error rate or latency.
3. and 4. See the cell below.

**C.** Health checks run every few seconds on every replica. Tracing them creates a flood of useless spans and costs money.

In [33]:
# Solutions 4.4-4.6
from fastapi import FastAPI
from fastapi.testclient import TestClient
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exp2 = InMemorySpanExporter()
prov2 = TracerProvider()
prov2.add_span_processor(SimpleSpanProcessor(exp2))

quiet_app = FastAPI(telemetry={"tracer_provider": prov2, "metrics": False, "logs": False,
                               "exclude": lambda scope: scope.get("path") == "/healthz"})

@quiet_app.get("/healthz")
def hz():
    return {"ok": True}

@quiet_app.get("/work")
def do_work():
    return {"done": True}

c = TestClient(quiet_app)
c.get("/healthz"); c.get("/work")
names = [s.name for s in exp2.get_finished_spans()]
print(names)     # ['fastapi.dependencies', 'fastapi.endpoint', 'fastapi.serialization', 'GET /work']
assert "GET /work" in names and not any("healthz" in n for n in names)

def should_alert(samples, threshold, for_minutes):
    recent = samples[-for_minutes:]
    return len(recent) == for_minutes and all(x > threshold for x in recent)

assert should_alert([0.01, 0.05, 0.06, 0.07], 0.02, 3) is True
assert should_alert([0.05, 0.01, 0.06], 0.02, 2) is False
assert should_alert([0.5], 0.02, 3) is False          # not enough data yet
print("Practice 4.4-4.6: all checks passed")

['fastapi.dependencies', 'fastapi.endpoint', 'fastapi.serialization', 'GET /work']
Practice 4.4-4.6: all checks passed


---

## How to debug with observability

| Question | Where to look |
|---|---|
| "User X got an error at 10:42" | search logs by request ID / user ID, open the trace |
| "The API got slower after the deploy" | latency histogram by route and release; compare traces before/after |
| "Errors spiked" | Sentry issue list (grouped), error-rate metric by route |
| "Only some requests are slow" | p99 vs p50; traces of slow requests; which child span is long |
| "Logs are missing the request ID" | the log call happens outside the middleware's context (startup, background thread) |
| "Prometheus is slow/huge" | high-cardinality labels; check the number of series per metric |

## Summary

* Logs (what happened), metrics (trends), traces (where time went), error tracking (which bug hurts most).
* JSON logs to stdout with a request ID from a `ContextVar`; no secrets in logs.
* Prometheus metrics with bounded labels; RED for every service; protect `/metrics`.
* OpenTelemetry for traces; FastAPI 0.162 has built-in support via `FastAPI(telemetry=...)` when a provider is configured.
* Alert on user-facing symptoms against SLOs, with time windows.

# Part 5: API Design Best Practices

**What this Part covers**

* REST resource naming, and the right HTTP method and status code for each action.
* PATCH vs PUT.
* Pagination (offset vs cursor), filtering and sorting.
* Idempotency keys for safe retries.
* One consistent error format: RFC 9457 "problem details".
* Versioning (`/v1`), deprecation, and generating clients from the OpenAPI schema.

**Before you start:** Notebook 1 (path/query/body parameters, response models, errors, OpenAPI) and
Notebook 2, Part 2 (APIRouter).

Good API design matters in interviews: "design the endpoints for X" is a very common question.

## 5.1 Resources, methods and status codes

**REST** (an architectural style for web APIs) models your API as **resources** (nouns) that clients act on with HTTP **methods** (verbs).

Naming rules that most teams follow:

* Plural nouns: `/books`, `/books/42`, `/users/7/orders`. Not verbs: avoid `/getBooks`, `/createBook`.
* Lowercase, words joined with hyphens: `/order-items`. Keep JSON field names consistent (`snake_case` or `camelCase`, pick one).
* Nest only one level for ownership (`/users/7/orders`); otherwise use filters (`/orders?user_id=7`).
* Actions that are not CRUD can be sub-resources: `POST /orders/42/cancel` or `POST /orders/42/cancellation`.

| Action | Method + path | Success status | Common errors |
|---|---|---|---|
| List | `GET /books` | 200 | 400/422 bad filters |
| Read one | `GET /books/{id}` | 200 | 404 |
| Create | `POST /books` | **201 Created** + `Location: /books/{id}` | 409 duplicate, 422 invalid |
| Replace | `PUT /books/{id}` | 200 (or 204) | 404, 422 |
| Partial update | `PATCH /books/{id}` | 200 | 404, 422 |
| Delete | `DELETE /books/{id}` | **204 No Content** | 404 (some APIs return 204 again: delete is idempotent) |
| Start long work | `POST /reports` | **202 Accepted** + a URL to poll | 429 too many |

Other codes to know: **400** bad request (malformed), **401** not authenticated (no/invalid credentials), **403** authenticated but
not allowed, **409** conflict with current state, **412** precondition failed (ETag mismatch), **413** body too large,
**415** unsupported media type, **422** validation failed, **429** too many requests, **503** temporarily unavailable.

**Safe** methods (GET, HEAD, OPTIONS) do not change data. **Idempotent** methods (GET, PUT, DELETE) give the same final state if
repeated. **POST and PATCH are not idempotent** by default; section 5.4 fixes that for POST.

In [34]:
# A small, well-shaped resource API: correct methods, statuses and Location header
from typing import Annotated
from fastapi import APIRouter, FastAPI, HTTPException, Response, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class BookIn(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    author: str = Field(min_length=1)
    year: int | None = None

class BookOut(BookIn):
    id: int

books_db: dict[int, BookOut] = {}
router = APIRouter(prefix="/v1/books", tags=["books"])

@router.post("", status_code=status.HTTP_201_CREATED)
def create_book(book: BookIn, response: Response) -> BookOut:
    if any(b.title == book.title and b.author == book.author for b in books_db.values()):
        raise HTTPException(status.HTTP_409_CONFLICT, "This book already exists")
    new = BookOut(id=len(books_db) + 1, **book.model_dump())
    books_db[new.id] = new
    response.headers["Location"] = f"/v1/books/{new.id}"
    return new

@router.get("/{book_id}")
def read_book(book_id: int) -> BookOut:
    if book_id not in books_db:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "Book not found")
    return books_db[book_id]

@router.delete("/{book_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_book(book_id: int) -> None:
    if books_db.pop(book_id, None) is None:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "Book not found")

design_app = FastAPI()
design_app.include_router(router)
client = TestClient(design_app)

r = client.post("/v1/books", json={"title": "Clean Code", "author": "Robert Martin", "year": 2008})
print(r.status_code, r.headers["location"], r.json()["id"])     # 201 /v1/books/1 1
print(client.post("/v1/books", json={"title": "Clean Code", "author": "Robert Martin"}).status_code)   # 409
print(client.delete("/v1/books/1").status_code)                  # 204
print(client.get("/v1/books/1").status_code)                     # 404

201 /v1/books/1 1
409
204
404


---

## 5.2 PATCH vs PUT

* **PUT** replaces the whole resource. The client sends **all** fields; missing optional fields are reset to their defaults.
* **PATCH** changes only the fields that were sent.

In FastAPI, a PATCH model makes every field optional, and `model_dump(exclude_unset=True)` gives only the fields the client
actually sent (so "not sent" and "sent as null" are different).

In [35]:
# PUT replaces, PATCH merges only the fields that were sent
class BookPatch(BaseModel):
    title: str | None = Field(default=None, min_length=1)
    author: str | None = None
    year: int | None = None

@router.put("/{book_id}")
def replace_book(book_id: int, book: BookIn) -> BookOut:
    if book_id not in books_db:
        raise HTTPException(404, "Book not found")
    books_db[book_id] = BookOut(id=book_id, **book.model_dump())
    return books_db[book_id]

@router.patch("/{book_id}")
def update_book(book_id: int, patch: BookPatch) -> BookOut:
    if book_id not in books_db:
        raise HTTPException(404, "Book not found")
    changes = patch.model_dump(exclude_unset=True)       # only what the client sent
    books_db[book_id] = books_db[book_id].model_copy(update=changes)
    return books_db[book_id]

# No need to include the router again: in FastAPI 0.162 an included router stays "live" (see the note below)
books_db.clear()
client.post("/v1/books", json={"title": "Dune", "author": "Frank Herbert", "year": 1965})

print(client.patch("/v1/books/1", json={"year": 1966}).json())
# {'title': 'Dune', 'author': 'Frank Herbert', 'year': 1966, 'id': 1}
print(client.put("/v1/books/1", json={"title": "Dune", "author": "F. Herbert"}).json())
# {'title': 'Dune', 'author': 'F. Herbert', 'year': None, 'id': 1}   <- PUT reset year (not sent)
print(client.patch("/v1/books/1", json={"year": None}).json()["year"])   # None (sent explicitly as null)

{'title': 'Dune', 'author': 'Frank Herbert', 'year': 1966, 'id': 1}
{'title': 'Dune', 'author': 'F. Herbert', 'year': None, 'id': 1}
None


> **Note (FastAPI 0.162):** `include_router` now keeps a live link to the router. Routes added to `router` *after*
> `design_app.include_router(router)` still work, which is why the cell above did not include it again. In older versions the routes
> were copied at include time, and late routes were silently missing. Because of this, `app.routes` now contains an internal
> "included router" entry instead of every route; to list all paths, read `app.openapi()["paths"]`.

> **Common mistake:** using `patch.model_dump()` (without `exclude_unset=True`) in a PATCH endpoint. Every field the client did not
> send becomes `None` and overwrites real data.

> **Interview tip:** "Is PATCH idempotent?" Not necessarily. `{"year": 1966}` is idempotent, but a patch like "add 1 to stock"
> is not. PUT is idempotent by definition.

> **Key points**
> * Nouns for paths, methods for actions; 201 + `Location` for create, 204 for delete, 409 for conflicts.
> * PUT = full replace; PATCH = partial, with `exclude_unset=True`.

### Practice 5.1-5.2: resources and methods

**A. Predict**

1. What status should `POST /v1/books` return on success? What header should come with it?
2. A client sends `PATCH /v1/books/1` with `{}` (empty JSON). What changes?
3. Choose a path and method: "a user cancels order 42".

**B. Write code**

4. Add `GET /v1/books` (a new router `list_router` is fine) that returns all books sorted by `id`, and check it returns a list.
5. A client app asks the user for a book with `input()` and creates it. Write `create_from_console(client)` that asks for title,
   author and year (empty year = no year), POSTs it, and returns the response. Use the sample answers.

**C. Find and fix the bug**

6. ```python
   @app.post("/createBook")
   def create(book: BookIn):
       ...
       return {"status": "created"}, 200
   ```

### Solutions 5.1-5.2

1. `201 Created`, with `Location: /v1/books/<id>`.
2. Nothing: `exclude_unset=True` gives an empty dict, so the book stays the same (200 with the current data).
3. `POST /v1/orders/42/cancel` (an action), or `PATCH /v1/orders/42` with `{"status": "cancelled"}` (a state change).
4. and 5. See the cell below.
6. A verb in the path (`/createBook` -> `POST /books`), status 200 instead of 201, and returning a tuple does not set the status in
   FastAPI (it is serialized as a JSON list). Use `status_code=201` in the decorator and return the created object.

In [36]:
# Solutions 5.1-5.2
from fastapi import APIRouter, FastAPI
from fastapi.testclient import TestClient

list_router = APIRouter(prefix="/v1/books")

@list_router.get("")
def list_books() -> list[BookOut]:
    return [books_db[k] for k in sorted(books_db)]

design_app = FastAPI()
design_app.include_router(list_router)
design_app.include_router(router)
client = TestClient(design_app)
assert isinstance(client.get("/v1/books").json(), list)

def create_from_console(client):
    title = input("Title: ").strip()
    author = input("Author: ").strip()
    year_text = input("Year (empty if unknown): ").strip()
    payload = {"title": title, "author": author, "year": int(year_text) if year_text else None}
    r = client.post("/v1/books", json=payload)
    print(r.status_code, r.json())
    return r

use_sample_input("The Pragmatic Programmer", "Hunt and Thomas", "1999")
r = create_from_console(client)
# Title: The Pragmatic Programmer
# Author: Hunt and Thomas
# Year (empty if unknown): 1999
# 201 {'title': 'The Pragmatic Programmer', 'author': 'Hunt and Thomas', 'year': 1999, 'id': 2}
assert r.status_code == 201 and r.json()["year"] == 1999
titles = [b["title"] for b in client.get("/v1/books").json()]
assert "The Pragmatic Programmer" in titles

# Q2: an empty PATCH changes nothing
before = client.get("/v1/books/1").json()
assert client.patch("/v1/books/1", json={}).json() == before
print("Practice 5.1-5.2: all checks passed")

Title: The Pragmatic Programmer
Author: Hunt and Thomas
Year (empty if unknown): 1999
201 {'title': 'The Pragmatic Programmer', 'author': 'Hunt and Thomas', 'year': 1999, 'id': 2}
Practice 5.1-5.2: all checks passed


---

## 5.3 Pagination, filtering and sorting

**Offset pagination:** `GET /books?limit=20&offset=40`. Easy, supports page numbers, but slow for deep pages and unstable when rows are
inserted (Part 3.3).

**Cursor pagination:** `GET /books?limit=20&cursor=eyJpZCI6NDB9`. The **cursor** is an opaque string (often base64 of the last
item's sort key). The response includes `next_cursor`; the client sends it back for the next page. Fast and stable, but no
"jump to page 7". Feeds, logs and big tables usually use cursors.

Response shape (pick one and use it everywhere):

```json
{"items": [...], "next_cursor": "eyJpZCI6NjB9", "has_more": true}
```

**Filtering and sorting:**

* Filters are query parameters: `?author=Herbert&year_from=1960`.
* Sorting: `?sort=-year` (minus = descending). **Allow only known fields** (a `Literal` type), never pass user input into SQL `ORDER BY`.
* Always cap `limit` (for example `le=100`) so nobody can request a million rows.

In [37]:
# Cursor pagination + filtering + safe sorting
import base64, json
from typing import Annotated, Literal
from fastapi import FastAPI, HTTPException, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel

catalog = [{"id": i, "title": f"Book {i}", "author": ["Herbert", "Austen", "Orwell"][i % 3], "year": 1900 + i * 3}
           for i in range(1, 31)]

class Page(BaseModel):
    items: list[dict]
    next_cursor: str | None
    has_more: bool

def encode_cursor(last_id: int) -> str:
    return base64.urlsafe_b64encode(json.dumps({"id": last_id}).encode()).decode()

def decode_cursor(cursor: str) -> int:
    try:
        return json.loads(base64.urlsafe_b64decode(cursor.encode()))["id"]
    except Exception:
        raise HTTPException(400, "Invalid cursor")

page_app = FastAPI()

@page_app.get("/v1/books")
def list_books_page(
    limit: Annotated[int, Query(ge=1, le=100)] = 10,
    cursor: str | None = None,
    author: str | None = None,
    sort: Literal["id", "year", "-year"] = "id",
) -> Page:
    rows = [b for b in catalog if author is None or b["author"] == author]
    if sort != "id":
        rows.sort(key=lambda b: b["year"], reverse=sort.startswith("-"))
        return Page(items=rows[:limit], next_cursor=None, has_more=len(rows) > limit)  # cursor only for id order here
    after = decode_cursor(cursor) if cursor else 0
    rows = [b for b in rows if b["id"] > after]
    items = rows[:limit]
    has_more = len(rows) > limit
    return Page(items=items, next_cursor=encode_cursor(items[-1]["id"]) if has_more else None, has_more=has_more)

client = TestClient(page_app)
p1 = client.get("/v1/books", params={"limit": 4, "author": "Austen"}).json()
print([b["id"] for b in p1["items"]], p1["has_more"])          # [1, 4, 7, 10] True
p2 = client.get("/v1/books", params={"limit": 4, "author": "Austen", "cursor": p1["next_cursor"]}).json()
print([b["id"] for b in p2["items"]])                          # [13, 16, 19, 22]
print(client.get("/v1/books", params={"sort": "title; DROP TABLE"}).status_code)   # 422 (not an allowed value)
print(client.get("/v1/books", params={"limit": 1000}).status_code)                 # 422 (limit capped at 100)
print(client.get("/v1/books", params={"cursor": "garbage"}).status_code)           # 400

[1, 4, 7, 10]

 True
[13, 16, 19, 22]
422
422
400


> **Note:** in a real database, cursor pagination with a non-unique sort key (like `year`) must include a tie-breaker:
> sort by `(year, id)` and put both values into the cursor.

> **Key points**
> * Offset for small tables and page numbers; cursor for feeds and big tables.
> * Return a consistent envelope (`items`, `next_cursor`, `has_more`).
> * Allow-list sort fields, cap `limit`, validate cursors.

### Practice 5.3: pagination

**A. Predict:** why is the cursor base64-encoded JSON instead of just `?after_id=13`?

**B. Write code**

1. Write `fetch_all(client, author)` that follows `next_cursor` until `has_more` is false and returns all ids. Check it returns all 10
   Austen books (ids 1, 4, ..., 28).
2. Add a `year_from: int | None` filter to a copy of the endpoint (a new app is fine) and check that `year_from=1960` returns only books
   with `year >= 1960`.

**C. Find the bug:** `rows.sort(key=lambda b: b[sort])` where `sort` is any string from the query. What can go wrong?

### Solutions 5.3

**A.** An opaque cursor lets the server change what is inside (add a tie-breaker, a sort direction, a timestamp) without breaking clients.
Clients must not build cursors themselves.

**B.** See the cell below.

**C.** Unknown keys raise `KeyError` (a 500 error), and in SQL the same idea becomes an injection risk. Use a `Literal` allow-list.

In [38]:
# Solutions 5.3
def fetch_all(client, author):
    ids, cursor = [], None
    while True:
        params = {"limit": 3, "author": author}
        if cursor:
            params["cursor"] = cursor
        page = client.get("/v1/books", params=params).json()
        ids += [b["id"] for b in page["items"]]
        if not page["has_more"]:
            return ids
        cursor = page["next_cursor"]

assert fetch_all(client, "Austen") == list(range(1, 31, 3))

year_app = FastAPI()

@year_app.get("/v1/books")
def books_since(year_from: int | None = None, limit: Annotated[int, Query(ge=1, le=100)] = 100) -> list[dict]:
    return [b for b in catalog if year_from is None or b["year"] >= year_from][:limit]

rows = TestClient(year_app).get("/v1/books", params={"year_from": 1960}).json()
assert rows and all(b["year"] >= 1960 for b in rows)
print("Practice 5.3: all checks passed")

Practice 5.3: all checks passed


---

## 5.4 Idempotency keys

Networks fail. A mobile app sends `POST /payments`, the connection drops, and the app does not know whether the payment happened.
If it retries, the customer may pay twice.

An **idempotency key** fixes this (Stripe made the pattern popular): the client generates a unique key (a UUID) per *logical*
operation and sends it in an `Idempotency-Key` header. The server stores the key with the response:

```text
 POST /payments  Idempotency-Key: k-123  {"amount": 50}
   key unknown          -> process, store (k-123, hash(body), response), return 201
   same key, same body  -> do NOT process again, return the stored response (replay)
   same key, other body -> 422: the key was reused for a different request
```

In production the store is the database or Redis, with an expiry (for example 24 hours), and you must handle two identical requests
arriving **at the same time** (a unique constraint or a lock on the key).

In [39]:
# Idempotent POST with an Idempotency-Key header
import hashlib, json, uuid
from typing import Annotated
from fastapi import FastAPI, Header, HTTPException, Response
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class PaymentIn(BaseModel):
    amount: int = Field(gt=0)
    currency: str = "INR"

idem_store: dict[str, tuple[str, int, dict]] = {}   # key -> (body hash, status, body)
payments: list[dict] = []
pay_app = FastAPI()

@pay_app.post("/v1/payments", status_code=201)
def create_payment(payment: PaymentIn, response: Response,
                   idempotency_key: Annotated[str, Header(min_length=8, max_length=100)]):
    body_hash = hashlib.sha256(payment.model_dump_json().encode()).hexdigest()
    if idempotency_key in idem_store:
        stored_hash, stored_status, stored_body = idem_store[idempotency_key]
        if stored_hash != body_hash:
            raise HTTPException(422, "Idempotency-Key was already used with a different request")
        response.status_code = stored_status
        response.headers["Idempotent-Replayed"] = "true"
        return stored_body
    record = {"id": len(payments) + 1, **payment.model_dump()}
    payments.append(record)                                   # the side effect happens once
    idem_store[idempotency_key] = (body_hash, 201, record)
    return record

client = TestClient(pay_app)
key = str(uuid.uuid4())
first = client.post("/v1/payments", json={"amount": 50}, headers={"Idempotency-Key": key})
retry = client.post("/v1/payments", json={"amount": 50}, headers={"Idempotency-Key": key})
print(first.status_code, retry.status_code, retry.headers.get("idempotent-replayed"))   # 201 201 true
print(first.json() == retry.json(), len(payments))                                      # True 1
print(client.post("/v1/payments", json={"amount": 99}, headers={"Idempotency-Key": key}).status_code)   # 422
print(client.post("/v1/payments", json={"amount": 50}).status_code)                     # 422 (header missing)

201 201 true
True 1
422
422


> **Key points**
> * The client creates one key per logical operation and reuses it on retries.
> * Same key + same body -> replay the stored response; same key + different body -> reject.
> * Store keys durably with an expiry and protect against concurrent duplicates.

---

## 5.5 One error format: RFC 9457 problem details

Clients hate APIs where every endpoint returns errors differently. **RFC 9457** ("Problem Details for HTTP APIs", which replaced
RFC 7807) defines a standard JSON error body with media type `application/problem+json`:

```json
{
  "type": "https://api.example.com/problems/book-not-found",
  "title": "Not Found",
  "status": 404,
  "detail": "Book 42 does not exist",
  "instance": "/v1/books/42"
}
```

* `type`: a URI that identifies the kind of problem (`about:blank` when you have nothing more specific).
* `title`: a short, human summary of the type. `detail`: this occurrence. `instance`: which request/resource.
* You can add **extension fields**, for example `errors` (validation details) or `request_id`.

FastAPI's default error bodies are `{"detail": ...}`. To use problem details everywhere, register handlers for
`HTTPException` (Starlette's base class, so 404s for unknown routes are included too) and `RequestValidationError`:

In [40]:
# Problem details (RFC 9457) for every error
from http import HTTPStatus
from fastapi import FastAPI, HTTPException, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from starlette.exceptions import HTTPException as StarletteHTTPException

def problem(status: int, detail, request: Request, type_: str = "about:blank", **extra) -> JSONResponse:
    body = {"type": type_, "title": HTTPStatus(status).phrase, "status": status,
            "detail": detail, "instance": request.url.path, **extra}
    return JSONResponse(body, status_code=status, media_type="application/problem+json")

def add_problem_handlers(app: FastAPI) -> None:
    @app.exception_handler(StarletteHTTPException)
    async def http_problem(request: Request, exc: StarletteHTTPException):
        response = problem(exc.status_code, exc.detail, request)
        if exc.headers:
            response.headers.update(exc.headers)          # keep e.g. WWW-Authenticate, Retry-After
        return response

    @app.exception_handler(RequestValidationError)
    async def validation_problem(request: Request, exc: RequestValidationError):
        errors = [{"loc": list(e["loc"]), "msg": e["msg"], "type": e["type"]} for e in exc.errors()]
        return problem(422, "The request is not valid", request, errors=errors)

problem_app = FastAPI()
add_problem_handlers(problem_app)

@problem_app.get("/v1/books/{book_id}")
def get_book(book_id: int):
    raise HTTPException(404, f"Book {book_id} does not exist")

client = TestClient(problem_app)
r = client.get("/v1/books/42")
print(r.headers["content-type"])     # application/problem+json
print(r.json())
# {'type': 'about:blank', 'title': 'Not Found', 'status': 404, 'detail': 'Book 42 does not exist', 'instance': '/v1/books/42'}
print(client.get("/v1/books/abc").json()["errors"][0])
# {'loc': ['path', 'book_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'type': 'int_parsing'}
print(client.get("/no-such-route").json()["title"])   # Not Found

application/problem+json
{'type': 'about:blank', 'title': 'Not Found', 'status': 404, 'detail': 'Book 42 does not exist', 'instance': '/v1/books/42'}
{'loc': ['path', 'book_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'type': 'int_parsing'}
Not Found


> **Common mistake:** returning `200` with `{"success": false, "error": "..."}`. Clients, proxies, monitoring and retries all depend on
> the status code. Errors must use 4xx/5xx codes.

> **Note:** do not copy the raw `input` values of validation errors into responses blindly: they can echo back passwords or other
> sensitive data. The handler above keeps only `loc`, `msg` and `type`.

---

## 5.6 Versioning and deprecation

You will need to change the API in ways that break old clients (rename a field, change a type). **Versioning** lets old and new
clients live side by side.

| Strategy | Example | Pros | Cons |
|---|---|---|---|
| URL path | `/v1/books`, `/v2/books` | obvious, easy to route and cache, shows in docs | URLs change between versions |
| Header | `Api-Version: 2` | clean URLs | harder to test in a browser, easy to forget |
| Media type | `Accept: application/vnd.shop.v2+json` | very "RESTful" | complex for clients |
| Date-based | `Stripe-Version: 2024-06-20` | fine-grained | needs a lot of infrastructure |

Path versioning with one `APIRouter` per version is the most common choice in FastAPI projects.
Only **breaking** changes need a new version; adding an optional field or a new endpoint does not.

**Deprecation:** mark old endpoints in OpenAPI (`deprecated=True`), announce a date, and send headers so clients can detect it:
`Deprecation` (RFC 9745) and `Sunset` (RFC 8594, the date it will stop working), plus a `Link` to the new version.

In [41]:
# Two versions side by side; v1 is deprecated and says so
from fastapi import APIRouter, FastAPI, Response
from fastapi.testclient import TestClient

v1 = APIRouter(prefix="/v1", tags=["v1"])
v2 = APIRouter(prefix="/v2", tags=["v2"])

@v1.get("/users/{user_id}", deprecated=True)
def get_user_v1(user_id: int, response: Response):
    response.headers["Deprecation"] = "@1767225600"                 # 2026-01-01 as a Unix timestamp (RFC 9745 format)
    response.headers["Sunset"] = "Wed, 30 Jun 2027 23:59:59 GMT"
    response.headers["Link"] = f'</v2/users/{user_id}>; rel="successor-version"'
    return {"id": user_id, "name": "Asha Rao"}                      # v1: one name field

@v2.get("/users/{user_id}")
def get_user_v2(user_id: int):
    return {"id": user_id, "first_name": "Asha", "last_name": "Rao"}   # v2: breaking change

versioned_app = FastAPI(title="Users API")
versioned_app.include_router(v1)
versioned_app.include_router(v2)
client = TestClient(versioned_app)

r1 = client.get("/v1/users/7")
print(r1.json(), r1.headers["sunset"])          # {'id': 7, 'name': 'Asha Rao'} Wed, 30 Jun 2027 23:59:59 GMT
print(client.get("/v2/users/7").json())         # {'id': 7, 'first_name': 'Asha', 'last_name': 'Rao'}
print(versioned_app.openapi()["paths"]["/v1/users/{user_id}"]["get"].get("deprecated"))   # True

{'id': 7, 'name': 'Asha Rao'} Wed, 30 Jun 2027 23:59:59 GMT
{'id': 7, 'first_name': 'Asha', 'last_name': 'Rao'}
True


---

## 5.7 Generated clients from OpenAPI

FastAPI publishes an **OpenAPI** schema at `/openapi.json` (Notebook 1, Part 10). Tools read it and **generate client code**
(TypeScript, Python, Kotlin...), so frontend and mobile teams get typed functions instead of hand-written HTTP calls.

```bash
# TypeScript client for a React frontend
npx @hey-api/openapi-ts -i http://localhost:8000/openapi.json -o src/client

# Python client
pip install openapi-python-client
openapi-python-client generate --url http://localhost:8000/openapi.json

# Many languages
npx @openapitools/openapi-generator-cli generate -i openapi.json -g kotlin -o ./kotlin-client
```

Generated function names come from each route's **operationId**. FastAPI's default ids are long
(`get_user_v2_v2_users__user_id__get`). A custom `generate_unique_id_function` gives clean names like `users-get_user`:

In [42]:
# Clean operation ids for nicer generated clients
from fastapi import APIRouter, FastAPI
from fastapi.routing import APIRoute

def tag_and_name(route: APIRoute) -> str:
    return f"{route.tags[0]}-{route.name}" if route.tags else route.name

client_app = FastAPI(generate_unique_id_function=tag_and_name)
users = APIRouter(prefix="/v1/users", tags=["users"])

@users.get("/{user_id}")
def get_user(user_id: int):
    return {"id": user_id}

@users.post("")
def create_user(name: str):
    return {"name": name}

client_app.include_router(users)
for path, ops in client_app.openapi()["paths"].items():
    for method, op in ops.items():
        print(method.upper(), path, "->", op["operationId"])
# GET /v1/users/{user_id} -> users-get_user
# POST /v1/users -> users-create_user
print(versioned_app.openapi()["paths"]["/v2/users/{user_id}"]["get"]["operationId"])   # get_user_v2_v2_users__user_id__get

GET

 /v1/users/{user_id} -> users-get_user
POST /v1/users -> users-create_user
get_user_v2_v2_users__user_id__get


> **Tip:** keep the schema stable in CI: export `app.openapi()` to a file and fail the build when a change breaks clients
> (tools such as `oasdiff` compare two schemas).

> **Key points**
> * Problem details (RFC 9457) give every error the same shape and media type.
> * Version only for breaking changes; path versioning with routers is simple. Deprecate with `deprecated=True` + headers + a date.
> * Clean `operationId`s make generated clients pleasant.

### Practice 5.4-5.7: idempotency, errors, versions

**A. Predict**

1. In the payments demo, a client retries with the same key but `"currency": "USD"`. What status?
2. With the problem handlers, what is the `title` of a 409 error?

**B. Write code**

3. Add the problem handlers to `pay_app`'s logic: build a new app `pay2_app` with `add_problem_handlers` and the same payment endpoint
   (you can call `create_payment` logic again or copy it). Check that the 422 for a reused key has `content-type: application/problem+json`
   and `status == 422`.
4. Write a small console helper that asks the user for an amount with `input()`, creates **one** idempotency key, and sends the
   payment twice (simulating a retry after a timeout). Check that only one payment was stored.
5. Write a test that every documented path of `versioned_app` starts with `/v1/` or `/v2/` (hint: `versioned_app.openapi()["paths"]`).

**C. Explain:** "We added an optional field `isbn` to the book response. Do we need `/v2`?"

### Solutions 5.4-5.7

1. 422: the body hash differs, so the key was reused for a different request.
2. `"Conflict"` (from `HTTPStatus(409).phrase`).
3. to 5. See the cell below.

**C.** Usually no. Adding an optional field is backward compatible; well-written clients ignore unknown fields. Removing or renaming a field,
changing a type, or making an optional input required are breaking changes.

In [43]:
# Solutions 5.4-5.7
import uuid
from fastapi import FastAPI
from fastapi.testclient import TestClient

pay2_app = FastAPI()
add_problem_handlers(pay2_app)
pay2_app.post("/v1/payments", status_code=201)(create_payment)     # register the same function on the new app
c2 = TestClient(pay2_app)
k = str(uuid.uuid4())
c2.post("/v1/payments", json={"amount": 10}, headers={"Idempotency-Key": k})
bad = c2.post("/v1/payments", json={"amount": 11}, headers={"Idempotency-Key": k})
assert bad.headers["content-type"] == "application/problem+json"
assert bad.json()["status"] == 422 and bad.json()["title"] == "Unprocessable Content"

def pay_with_retry(client):
    amount = int(input("Amount to pay (INR): "))
    key = str(uuid.uuid4())                      # one key for this logical payment
    responses = [client.post("/v1/payments", json={"amount": amount}, headers={"Idempotency-Key": key})
                 for _ in range(2)]              # second call = retry after a timeout
    print("statuses:", [r.status_code for r in responses])
    return responses

before = len(payments)
use_sample_input("250")
rs = pay_with_retry(c2)
# Amount to pay (INR): 250
# statuses: [201, 201]
assert len(payments) == before + 1 and rs[0].json() == rs[1].json()

paths = list(versioned_app.openapi()["paths"])     # app.routes holds an "included router" entry in 0.162
print(paths)                                         # ['/v1/users/{user_id}', '/v2/users/{user_id}']
assert paths and all(p.startswith(("/v1/", "/v2/")) for p in paths)
print("Practice 5.4-5.7: all checks passed")

Amount to pay (INR): 250
statuses: [201, 201]
['/v1/users/{user_id}', '/v2/users/{user_id}']
Practice 5.4-5.7: all checks passed


---

## How to debug API design problems

| Symptom | Likely cause | Fix |
|---|---|---|
| Duplicate orders/payments after network errors | POST retried without idempotency | idempotency keys |
| PATCH wipes fields | `model_dump()` without `exclude_unset=True` | use `exclude_unset=True` |
| Clients break after a "small" change | renamed/removed field without a new version | additive changes only; new version for breaking ones |
| Frontend parses errors in 5 different ways | mixed error bodies | one problem-details handler set |
| Deep pages time out | offset pagination on a big table | cursor/keyset pagination + index |
| 500 on `?sort=foo` | unvalidated sort parameter | `Literal` allow-list |
| Ugly generated client method names | default operation ids | `generate_unique_id_function` |

## Summary

* Plural nouns, HTTP methods as verbs; 201 + `Location`, 204, 409, 422, 429 used correctly.
* PUT replaces, PATCH merges (`exclude_unset=True`).
* Offset vs cursor pagination; allow-listed filters and sorting; capped limits.
* Idempotency keys make POST retries safe.
* RFC 9457 problem details for all errors; `/v1` routers; deprecate with headers and dates; clean operation ids for generated clients.

# Part 6: Serving ML and LLM Models with FastAPI

**What this Part covers**

* Loading a model **once** in the lifespan, and prediction endpoints with Pydantic input and output.
* CPU-bound inference, thread pools, and **micro-batching** many requests into one model call.
* Streaming LLM tokens to the client with **Server-Sent Events**, using FastAPI's built-in `fastapi.sse` module.
* Calling an external LLM API with timeouts and retries (simulated with `httpx.MockTransport`, no real API).
* The outline of a **RAG** API: upload, background processing, and questions answered with citations.

**Before you start:** Notebook 2, Part 3 (async), Part 6 (lifespan) and Part 7 (background tasks, streaming, SSE).

> **Note:** no real ML library or LLM is used. The "models" are tiny Python classes, so the cells run anywhere and fast.
> The serving patterns are exactly the same for scikit-learn, PyTorch or an LLM API.

## 6.1 Load the model once, predict with typed input and output

Loading a model can take seconds and hundreds of MB. Doing it **per request** is the most common beginner mistake. Load it once at
startup in the **lifespan**, store it on `app.state`, and reach it through a dependency (so tests can replace it).

```text
 startup (lifespan)          request                                   shutdown
 load model -> app.state  -> Depends(get_model) -> model.predict() ->  free resources
```

In [44]:
# A tiny "sentiment model" served the production way
import time
from contextlib import asynccontextmanager
from typing import Annotated, Literal
from fastapi import Depends, FastAPI, Request
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class SentimentModel:
    version = "sentiment-0.3"
    POSITIVE = {"good", "great", "love", "excellent", "fast", "helpful"}
    NEGATIVE = {"bad", "slow", "hate", "terrible", "broken", "rude"}

    def __init__(self):
        time.sleep(0.2)                                  # pretend loading weights from disk takes time
        self.loaded_at = time.time()

    def predict_one(self, text: str) -> tuple[str, float]:
        words = text.lower().replace(",", " ").replace(".", " ").split()
        score = sum(w in self.POSITIVE for w in words) - sum(w in self.NEGATIVE for w in words)
        label = "positive" if score > 0 else "negative" if score < 0 else "neutral"
        confidence = min(0.5 + 0.2 * abs(score), 0.99)
        return label, round(confidence, 2)

    def predict_batch(self, texts: list[str]) -> list[tuple[str, float]]:
        return [self.predict_one(t) for t in texts]      # real models are much faster per item in a batch

class PredictIn(BaseModel):
    text: str = Field(min_length=1, max_length=2000, examples=["The delivery was fast and the staff were helpful"])

class PredictOut(BaseModel):
    label: Literal["positive", "negative", "neutral"]
    confidence: float = Field(ge=0, le=1)
    model_version: str

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.model = SentimentModel()                   # loaded ONCE per worker process
    yield
    del app.state.model

def get_model(request: Request) -> SentimentModel:
    return request.app.state.model

ml_app = FastAPI(title="Sentiment API", lifespan=lifespan)

@ml_app.post("/v1/predict")
def predict(body: PredictIn, model: Annotated[SentimentModel, Depends(get_model)]) -> PredictOut:
    label, confidence = model.predict_one(body.text)     # def endpoint: runs in the thread pool
    return PredictOut(label=label, confidence=confidence, model_version=model.version)

with TestClient(ml_app) as client:
    t = time.perf_counter()
    for _ in range(20):
        r = client.post("/v1/predict", json={"text": "Great product, fast delivery"})
    print(r.json())                                      # {'label': 'positive', 'confidence': 0.9, 'model_version': 'sentiment-0.3'}
    print(f"20 predictions in {time.perf_counter() - t:.2f}s (model loaded once)")   # well under 0.2 s per call
    print(client.post("/v1/predict", json={"text": ""}).status_code)                 # 422 (min_length=1)

{'label': 'positive', 'confidence': 0.9, 'model_version': 'sentiment-0.3'}
20 predictions in 0.08s (model loaded once)
422


Why this shape works well:

* **Pydantic input** rejects bad data before it reaches the model (empty text, too long, wrong types) with a 422.
* **Pydantic output** documents the response in OpenAPI and guarantees the shape (for example `label` can only be three values).
* Returning `model_version` makes predictions traceable when you deploy a new model.
* The dependency `get_model` lets tests swap in a fake model with `app.dependency_overrides` (Notebook 2, Part 8).

**Where to run inference** (Part 3.2):

| Model call | Endpoint style |
|---|---|
| Fast, releases the GIL (NumPy, PyTorch, ONNX Runtime do heavy work in C) | `def` endpoint (thread pool) is usually fine |
| Pure-Python, CPU-heavy | process pool, or a separate model server (TorchServe, Triton, BentoML, Ray Serve) |
| Remote API (OpenAI, Groq, Anthropic, a model server) | `async def` + `httpx.AsyncClient` (6.4) |
| Minutes long (training, big reports) | a task queue (Celery/RQ/Arq) + `202 Accepted` + status endpoint |

> **Warning:** each worker loads its own copy of the model. A 2 GB model x 4 workers = 8 GB of RAM. That is why ML services often run
> fewer workers (Part 1.3) or put the model in a separate server.

### A small command-line client for the model

A data analyst wants to try the model from a terminal. This **client** asks for a review with `input()` and calls the API.
(The API itself never calls `input()`.)

In [45]:
# CLI client: type reviews until you enter an empty line
def review_cli(client):
    while True:
        text = input("Review (empty to quit): ").strip()
        if not text:
            print("Bye!")
            break
        r = client.post("/v1/predict", json={"text": text})
        if r.status_code != 200:
            print("Error:", r.status_code, r.json())
            continue
        out = r.json()
        print(f"  -> {out['label']} ({out['confidence']:.0%})")

use_sample_input("The support team was rude and the app is broken",
                 "Excellent quality, I love it",
                 "It arrived on Tuesday",
                 "")
with TestClient(ml_app) as client:
    review_cli(client)
# Review (empty to quit): The support team was rude and the app is broken
#   -> negative (90%)
# Review (empty to quit): Excellent quality, I love it
#   -> positive (90%)
# Review (empty to quit): It arrived on Tuesday
#   -> neutral (50%)
# Review (empty to quit):
# Bye!

Review (empty to quit): The support team was rude and the app is broken
  -> negative (90%)
Review (empty to quit): Excellent quality, I love it
  -> positive (90%)
Review (empty to quit): It arrived on Tuesday
  -> neutral (50%)
Review (empty to quit): 
Bye!


---

## 6.2 CPU-bound work, thread pools and micro-batching

Many models are much faster on a **batch** of inputs than on one input at a time (GPUs especially). But HTTP requests arrive one
by one. **Micro-batching** collects requests that arrive within a few milliseconds and runs them as one batch:

```text
 req A --+                                   +--> answer A
 req B --+--> queue --> wait max 10 ms or 16 items --> model.predict_batch([A,B,C]) --+--> answer B
 req C --+                                   +--> answer C
```

Each request awaits an `asyncio.Future` that the batch worker fills in. The model call itself is pushed to a thread with
`asyncio.to_thread`, so the event loop stays free.

In [46]:
# Micro-batching: many concurrent requests -> few model calls
import asyncio
import httpx
from fastapi import FastAPI

class MicroBatcher:
    def __init__(self, model, max_batch: int = 16, max_wait_s: float = 0.01):
        self.model, self.max_batch, self.max_wait_s = model, max_batch, max_wait_s
        self.queue: asyncio.Queue = asyncio.Queue()
        self.batch_sizes: list[int] = []
        self._worker: asyncio.Task | None = None

    async def predict(self, text: str):
        if self._worker is None:                              # start lazily (in a real app: in the lifespan)
            self._worker = asyncio.create_task(self._run())
        future = asyncio.get_running_loop().create_future()
        await self.queue.put((text, future))
        return await future

    async def _run(self):
        while True:
            batch = [await self.queue.get()]                  # wait for the first item
            deadline = asyncio.get_running_loop().time() + self.max_wait_s
            while len(batch) < self.max_batch:
                timeout = deadline - asyncio.get_running_loop().time()
                if timeout <= 0:
                    break
                try:
                    batch.append(await asyncio.wait_for(self.queue.get(), timeout))
                except TimeoutError:
                    break
            texts = [text for text, _ in batch]
            self.batch_sizes.append(len(batch))
            try:
                results = await asyncio.to_thread(self.model.predict_batch, texts)
                for (_, fut), result in zip(batch, results):
                    fut.set_result(result)
            except Exception as exc:                          # never leave callers waiting forever
                for _, fut in batch:
                    fut.set_exception(exc)

    async def stop(self):
        if self._worker:
            self._worker.cancel()
            self._worker = None

batcher = MicroBatcher(SentimentModel(), max_batch=16)
batch_app = FastAPI()

@batch_app.post("/v1/predict")
async def predict_batched(body: PredictIn) -> PredictOut:
    label, confidence = await batcher.predict(body.text)
    return PredictOut(label=label, confidence=confidence, model_version=SentimentModel.version)

texts = ["good", "bad", "great service", "slow and rude"] * 10      # 40 concurrent requests
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=batch_app), base_url="http://t") as ac:
    responses = await asyncio.gather(*(ac.post("/v1/predict", json={"text": t}) for t in texts))
await batcher.stop()

print([r.json()["label"] for r in responses[:4]])     # ['positive', 'negative', 'positive', 'negative']
print("requests:", len(responses), "model calls:", len(batcher.batch_sizes))   # requests: 40 model calls: 3
print("batch sizes:", batcher.batch_sizes)            # e.g. [16, 16, 8]

['positive', 'negative', 'positive', 'negative']
requests: 40 model calls: 3
batch sizes: [16, 16, 8]


Trade-offs of batching: higher throughput, but each request may wait up to `max_wait_s` longer. Tune `max_batch` and `max_wait_s`
with a load test. Dedicated model servers (Triton, TorchServe, vLLM for LLMs) implement batching for you.

> **Key points**
> * Load once in the lifespan; access through a dependency; typed input and output; return the model version.
> * Blocking or CPU inference: `def` endpoint, `asyncio.to_thread` / `run_in_threadpool`, or a process pool / model server.
> * Micro-batching trades a few milliseconds of latency for much higher throughput.

### Practice 6.1-6.2: model serving

**A. Predict**

1. What goes wrong if `predict()` does `model = SentimentModel()` inside the endpoint?
2. In the batcher, what happens to waiting requests if `predict_batch` raises an exception?

**B. Write code**

3. Write a test that replaces the model of `ml_app` with a fake that always returns `("neutral", 0.5)` using
   `ml_app.dependency_overrides[get_model]`, and check the response. Remove the override afterwards.
4. Add a `/v1/predict/batch` endpoint to a new app `bulk_app` that accepts `{"texts": [...]}` (1 to 32 items) and returns a list of
   `PredictOut`. Check that 33 items give 422.
5. A CLI asks the user how many reviews they want to score with `input()`, then asks for each review, and sends them **in one batch**
   call to `/v1/predict/batch`. Write it and test it with sample answers.

**C. Find the bug**

6. ```python
   @app.post("/predict")
   async def predict(body: PredictIn):
       return heavy_pure_python_model.predict(body.text)    # takes 300 ms of CPU
   ```

### Solutions 6.1-6.2

1. The model loads on every request: each call takes the full loading time and memory spikes. Load it once in the lifespan.
2. The `except` block sets the exception on every future, so each waiting request fails (500) instead of hanging forever.
3. to 5. See the cell below.
6. CPU-heavy work in `async def` blocks the event loop for 300 ms per request; every other request waits. Use `def`,
   `await run_in_threadpool(...)`, or (for pure-Python CPU work) a process pool or separate model service.

In [47]:
# Solutions 6.1-6.2
from typing import Annotated
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class FakeModel:
    version = "fake-1"
    def predict_one(self, text):
        return ("neutral", 0.5)

ml_app.dependency_overrides[get_model] = lambda: FakeModel()
try:
    with TestClient(ml_app) as c:
        assert c.post("/v1/predict", json={"text": "anything"}).json() == \
            {"label": "neutral", "confidence": 0.5, "model_version": "fake-1"}
finally:
    ml_app.dependency_overrides.clear()

class BatchIn(BaseModel):
    texts: Annotated[list[Annotated[str, Field(min_length=1, max_length=2000)]], Field(min_length=1, max_length=32)]

bulk_model = SentimentModel()
bulk_app = FastAPI()

@bulk_app.post("/v1/predict/batch")
def predict_many(body: BatchIn) -> list[PredictOut]:
    return [PredictOut(label=l, confidence=c, model_version=bulk_model.version)
            for l, c in bulk_model.predict_batch(body.texts)]

bc = TestClient(bulk_app)
assert [p["label"] for p in bc.post("/v1/predict/batch", json={"texts": ["good", "bad"]}).json()] == ["positive", "negative"]
assert bc.post("/v1/predict/batch", json={"texts": ["x"] * 33}).status_code == 422
assert bc.post("/v1/predict/batch", json={"texts": []}).status_code == 422

def batch_cli(client):
    n = int(input("How many reviews? "))
    texts = [input(f"Review {i + 1}: ") for i in range(n)]
    results = client.post("/v1/predict/batch", json={"texts": texts}).json()
    for text, res in zip(texts, results):
        print(f"  {res['label']:8} | {text}")
    return results

use_sample_input("2", "Helpful staff and great food", "Terrible wait time")
out = batch_cli(bc)
# How many reviews? 2
# Review 1: Helpful staff and great food
# Review 2: Terrible wait time
#   positive | Helpful staff and great food
#   negative | Terrible wait time
assert [r["label"] for r in out] == ["positive", "negative"]
print("Practice 6.1-6.2: all checks passed")

How many reviews? 2
Review 1: Helpful staff and great food
Review 2: Terrible wait time
  positive | Helpful staff and great food
  negative | Terrible wait time
Practice 6.1-6.2: all checks passed


---

## 6.3 Streaming LLM tokens with Server-Sent Events

An LLM produces its answer **token by token** (a token is a word piece). Waiting 10 seconds for the full answer feels slow;
streaming the tokens as they are generated feels instant. **Server-Sent Events (SSE)** is a simple standard for this: one long HTTP
response with `Content-Type: text/event-stream`, made of small text events:

```text
event: token
data: {"text": "Hel"}

event: token
data: {"text": "lo"}

event: done
data: [DONE]
```

Notebook 2, Part 7 introduced streaming. FastAPI 0.162 has a built-in **`fastapi.sse`** module (a recent addition, so older
tutorials use `StreamingResponse` or the `sse-starlette` package):

* `response_class=EventSourceResponse` on a path operation whose function **yields** values.
* Yield `ServerSentEvent(data=..., event=..., id=..., retry=...)` for full control, or yield plain objects (dicts, Pydantic models),
  which are JSON-encoded into `data:`. `data` is **always JSON** (a string `"hi"` becomes `data: "hi"`); use `raw_data=` to send
  text as-is.
* It works with **any method**, including `POST` (useful for chat: the prompt goes in the body).
* FastAPI adds `Cache-Control: no-cache` and `X-Accel-Buffering: no` (so Nginx does not buffer), and sends a `: ping` comment
  every 15 seconds when nothing else is sent, so proxies do not close an idle stream.

In [48]:
# SSE token streaming with the built-in fastapi.sse module and a fake LLM
import asyncio
from collections.abc import AsyncIterable
from fastapi import FastAPI
from fastapi.sse import EventSourceResponse, ServerSentEvent
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

async def fake_llm_tokens(prompt: str):
    """Pretend to be an LLM: yield the answer in small pieces."""
    answer = f"You asked about {prompt.strip().rstrip('?')}. Here is a short answer."
    for word in answer.split(" "):
        await asyncio.sleep(0)                 # a real model would take ~20-50 ms per token
        yield word + " "

class ChatIn(BaseModel):
    prompt: str = Field(min_length=1, max_length=4000)

sse_app = FastAPI()

@sse_app.post("/v1/chat/stream", response_class=EventSourceResponse)
async def chat_stream(body: ChatIn) -> AsyncIterable[ServerSentEvent]:
    count = 0
    async for token in fake_llm_tokens(body.prompt):
        count += 1
        yield ServerSentEvent(data={"text": token}, event="token", id=str(count))
    yield ServerSentEvent(data={"tokens": count}, event="usage")
    yield ServerSentEvent(raw_data="[DONE]", event="done")

client = TestClient(sse_app)
with client.stream("POST", "/v1/chat/stream", json={"prompt": "FastAPI workers?"}) as r:
    print(r.headers["content-type"])           # text/event-stream; charset=utf-8
    print(r.headers["x-accel-buffering"])      # no
    lines = list(r.iter_lines())
print(lines[:3])                               # ['event: token', 'data: {"text": "You "}', 'id: 1']

text/event-stream; charset=utf-8
no
['event: token', 'data: {"text": "You "}', 'id: 1']


A client reads the stream line by line and handles each event. Here is a small parser that rebuilds the events from
`iter_lines()`. A browser would use `fetch()` with a stream reader (or `EventSource` for GET endpoints).

In [49]:
# Parse SSE lines into events and rebuild the answer
import json

def parse_sse(lines):
    event = {}
    for line in lines:
        if line == "":                          # a blank line ends one event
            if event:
                yield event
            event = {}
        elif line.startswith(":"):
            continue                            # a comment, e.g. the ": ping" keep-alive
        else:
            field, _, value = line.partition(":")
            event[field] = value.removeprefix(" ")
    if event:
        yield event

def stream_answer(client, prompt):
    text, usage = "", None
    with client.stream("POST", "/v1/chat/stream", json={"prompt": prompt}) as r:
        r.raise_for_status()
        for ev in parse_sse(r.iter_lines()):
            if ev.get("event") == "token":
                piece = json.loads(ev["data"])["text"]
                text += piece
                print(piece, end="", flush=True)   # appears word by word in a real terminal
            elif ev.get("event") == "usage":
                usage = json.loads(ev["data"])
            elif ev.get("event") == "done":
                print()
    return text.strip(), usage

answer, usage = stream_answer(client, "What is ASGI?")
# You asked about What is ASGI. Here is a short answer.
print(usage)        # {'tokens': 11}

You 

asked 

about 

What 

is 

ASGI. 

... (2 outputs left out)

a 

short 

answer. 


{'tokens': 11}


**Also new: JSON Lines streaming.** If a path operation **yields** values but has no `response_class`, FastAPI 0.162 streams them as
**JSON Lines** (`application/jsonl`, one JSON object per line). That is handy for streaming results to other services.

**Client disconnects.** If the user closes the tab, stop generating (an LLM call costs money). For async generators, the server
cancels the generator when the client goes away; for long loops you can also check `await request.is_disconnected()`.

> **Common mistake:** returning `StreamingResponse(generator())` with `media_type="text/plain"` and formatting `data:` lines by hand.
> It works, but it is easy to break the format (multi-line data, missing blank lines). Use `EventSourceResponse` + `ServerSentEvent`.

> **Key points**
> * SSE = one long `text/event-stream` response of `event:`/`data:` blocks; perfect for LLM tokens (server to client only).
> * FastAPI 0.162: `response_class=EventSourceResponse`, yield `ServerSentEvent`; `data` is JSON, `raw_data` is text.
> * Use WebSockets instead only when the client must also send messages during the stream.

In [50]:
# Bonus: a generator endpoint without response_class streams JSON Lines
from fastapi import FastAPI
from fastapi.testclient import TestClient

jsonl_app = FastAPI()

@jsonl_app.get("/v1/scores")
async def stream_scores():
    for i in range(3):
        yield {"row": i, "score": i * 0.5}

r = TestClient(jsonl_app).get("/v1/scores")
print(r.headers["content-type"])     # application/jsonl
print(r.text)
# {"row": 0, "score": 0.0}
# {"row": 1, "score": 0.5}
# {"row": 2, "score": 1.0}

application/jsonl
{"row": 0, "score": 0.0}
{"row": 1, "score": 0.5}
{"row": 2, "score": 1.0}



---

## 6.4 Calling an external LLM API: timeouts and retries

Most apps call a hosted LLM (OpenAI, Anthropic, Groq, Gemini...) over HTTP. Remote calls fail in many ways: timeouts, `429 Too Many
Requests`, `500/502/503` from overloaded servers. A production client needs:

* **Timeouts** on every call (`httpx` has a 5 s default; LLM calls often need 30-60 s for long answers, but the *connect* timeout should be short).
* **Retries** only for temporary errors (timeouts, connection errors, 429, 5xx), with **exponential backoff + jitter**
  (wait 0.5 s, 1 s, 2 s... plus a random part, so many clients do not retry at the same moment), honouring `Retry-After`.
* **No retries** for 400/401/403/422: they will fail again.
* A **total time budget**, so a user request does not wait forever.
* One shared `httpx.AsyncClient` created in the lifespan (connection pooling).

> **Note:** `httpx.AsyncHTTPTransport(retries=3)` only retries **connection** failures (cannot connect), not timeouts or 5xx
> responses. Application-level retries like below are still needed. Libraries such as `tenacity` or `stamina` provide retry
> decorators; official LLM SDKs also have built-in retries.

We simulate the remote API with `httpx.MockTransport`: it fails twice (a timeout, then a 503) and then answers.

In [51]:
# A resilient LLM client: timeouts, retries with backoff + jitter, Retry-After, error mapping
import asyncio, random
import httpx

class LLMUnavailable(Exception):
    """The LLM could not answer in time (map to 503/504 in the API)."""

RETRY_STATUS = {429, 500, 502, 503, 504}

async def call_llm(client: httpx.AsyncClient, prompt: str, *, attempts: int = 4,
                   base_delay: float = 0.05, sleep=asyncio.sleep) -> str:
    for attempt in range(1, attempts + 1):
        try:
            r = await client.post("/v1/chat/completions", json={"model": "small", "prompt": prompt})
            if r.status_code in RETRY_STATUS:
                raise httpx.HTTPStatusError(f"retryable {r.status_code}", request=r.request, response=r)
            r.raise_for_status()                             # 4xx (not 429): raise at once, no retry
            return r.json()["text"]
        except (httpx.TimeoutException, httpx.ConnectError, httpx.HTTPStatusError) as exc:
            status = exc.response.status_code if isinstance(exc, httpx.HTTPStatusError) else None
            if status is not None and status not in RETRY_STATUS:
                raise
            if attempt == attempts:
                raise LLMUnavailable(f"gave up after {attempts} attempts: {exc!r}") from exc
            retry_after = exc.response.headers.get("Retry-After") if status else None
            delay = float(retry_after) if retry_after else base_delay * 2 ** (attempt - 1) * (1 + random.random())
            print(f"  attempt {attempt} failed ({status or type(exc).__name__}), retrying in {delay:.2f}s")
            await sleep(delay)

script = iter(["timeout", 503, 200])                         # what the fake API does on each call
def fake_llm_api(request: httpx.Request) -> httpx.Response:
    step = next(script)
    if step == "timeout":
        raise httpx.ReadTimeout("model is slow", request=request)
    if step == 503:
        return httpx.Response(503, headers={"Retry-After": "0.1"}, json={"error": "overloaded"})
    return httpx.Response(200, json={"text": "Paris is the capital of France."})

timeout = httpx.Timeout(30.0, connect=2.0)                   # 30 s to read, only 2 s to connect
async with httpx.AsyncClient(transport=httpx.MockTransport(fake_llm_api),
                             base_url="https://llm.example.com", timeout=timeout) as llm:
    random.seed(0)
    print(await call_llm(llm, "Capital of France?"))
#   attempt 1 failed (ReadTimeout), retrying in 0.09s
#   attempt 2 failed (503), retrying in 0.10s
# Paris is the capital of France.

  attempt 1 failed (ReadTimeout), retrying in 0.09s


  attempt 2 failed (503), retrying in 0.10s


Paris is the capital of France.


In the API, wrap the call with an overall time budget and map failures to clear status codes:

```python
@app.post("/v1/answer")
async def answer(body: ChatIn, request: Request):
    try:
        async with asyncio.timeout(20):                       # total budget for this request (Python 3.11+)
            text = await call_llm(request.app.state.llm_http, body.prompt)
    except TimeoutError:
        raise HTTPException(504, "The language model took too long")
    except LLMUnavailable:
        raise HTTPException(503, "The language model is unavailable, please retry", headers={"Retry-After": "5"})
    return {"answer": text}
```

A **circuit breaker** goes one step further: after many failures in a row, it stops calling the LLM for a short time and fails fast,
which protects both your app and the provider.

### Practice 6.3-6.4: streaming and resilient calls

**A. Predict**

1. With `ServerSentEvent(data="hello")`, what exactly is on the `data:` line?
2. The fake API returns `401` on the first call. How many attempts does `call_llm` make?

**B. Write code**

3. Add an endpoint `/v1/count` (GET) to a new app that streams `ServerSentEvent(data=i, event="tick")` for i = 1..3 and then a `done`
   event. Use `parse_sse` to check you get 4 events and the last one is `done`.
4. Test `call_llm` against an API that **always** returns 503, with `attempts=3` and a fake `sleep` that records delays instead of
   waiting. Check that it raises `LLMUnavailable` and slept exactly twice.
5. Test that a `400` response is **not** retried (the handler is called once) and raises `httpx.HTTPStatusError`.

**C. Explain:** why add random jitter to the backoff delay?

### Solutions 6.3-6.4

1. `data: "hello"` with quotes, because `data` is always JSON-encoded. Use `raw_data="hello"` for `data: hello`.
2. One. 401 is not retryable, so `raise_for_status()` raises immediately.
3. to 5. See the cell below.

**C.** Without jitter, thousands of clients that failed at the same moment retry at the same moment again ("thundering herd"),
overloading the server once more. Jitter spreads the retries out.

In [52]:
# Solutions 6.3-6.4
import httpx
from fastapi import FastAPI
from fastapi.sse import EventSourceResponse, ServerSentEvent
from fastapi.testclient import TestClient

count_app = FastAPI()

@count_app.get("/v1/count", response_class=EventSourceResponse)
async def count():
    for i in range(1, 4):
        yield ServerSentEvent(data=i, event="tick")
    yield ServerSentEvent(raw_data="bye", event="done")

with TestClient(count_app).stream("GET", "/v1/count") as r:
    events = list(parse_sse(r.iter_lines()))
assert len(events) == 4 and events[-1] == {"event": "done", "data": "bye"}
assert events[0] == {"event": "tick", "data": "1"}

slept = []
async def fake_sleep(seconds):
    slept.append(seconds)

always_503 = httpx.MockTransport(lambda req: httpx.Response(503))
async with httpx.AsyncClient(transport=always_503, base_url="https://llm") as llm:
    try:
        await call_llm(llm, "hi", attempts=3, sleep=fake_sleep)
        raise AssertionError("expected LLMUnavailable")
    except LLMUnavailable as e:
        print("LLMUnavailable:", str(e)[:40])
assert len(slept) == 2

calls = {"n": 0}
def bad_request(req):
    calls["n"] += 1
    return httpx.Response(400, json={"error": "bad prompt"})

async with httpx.AsyncClient(transport=httpx.MockTransport(bad_request), base_url="https://llm") as llm:
    try:
        await call_llm(llm, "hi", sleep=fake_sleep)
        raise AssertionError("expected HTTPStatusError")
    except httpx.HTTPStatusError as e:
        assert e.response.status_code == 400
assert calls["n"] == 1
print("Practice 6.3-6.4: all checks passed")

  attempt 1 failed (503), retrying in 0.09s
  attempt 2 failed (503), retrying in 0.14s
LLMUnavailable: gave up after 3 attempts: HTTPStatusErro
Practice 6.3-6.4: all checks passed


---

## 6.5 The outline of a RAG API

**RAG (Retrieval-Augmented Generation)** = before asking the LLM, **retrieve** relevant pieces of your own documents and put them in
the prompt, so the answer is grounded in your data and can **cite** its sources.

```text
 1. UPLOAD     POST /documents (file)  ->  202 Accepted {id, status: "processing"}
                     |
 2. PROCESS    background task: extract text -> split into chunks -> embed each chunk -> store in a vector index
                     |                                                     status -> "ready" (or "failed")
 3. QUERY      POST /query {question}
                     -> embed the question -> find the top-k most similar chunks
                     -> prompt = question + chunks -> LLM -> answer
                     <- {answer, citations: [{document_id, chunk, score, text}]}
```

Design points:

* Upload returns **202** quickly; processing (parsing PDFs, embeddings) is slow, so it runs in the background (BackgroundTasks for a
  small app, a task queue for a big one). The client polls `GET /documents/{id}` for the status.
* Validate uploads: allowed types, size limit (413), clean file names, de-duplicate by content hash.
* Answers include **citations** so users can check the source; say "I don't know" when retrieval finds nothing relevant.
* Readiness: not ready until the embedding model is loaded and the LLM key is configured.

Below is a tiny, fully runnable version. "Embeddings" are just sets of words and "similarity" is word overlap, so it runs without
any ML library; the API shape is the real one.

In [53]:
# A mini RAG API: upload -> background processing -> query with citations
import hashlib, re, uuid
from typing import Annotated
from fastapi import BackgroundTasks, FastAPI, File, HTTPException, UploadFile, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

MAX_UPLOAD_BYTES = 100_000
documents: dict[str, dict] = {}          # id -> {name, status, hash}
chunks: list[dict] = []                  # {document_id, index, text, terms}

def terms(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z]+", text.lower()) if len(w) > 3}

def process_document(doc_id: str, text: str) -> None:
    """Background task: chunk and 'embed' the text."""
    try:
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
        for i in range(0, len(sentences), 2):              # 2 sentences per chunk
            chunk_text = " ".join(sentences[i:i + 2])
            chunks.append({"document_id": doc_id, "index": i // 2, "text": chunk_text, "terms": terms(chunk_text)})
        documents[doc_id]["status"] = "ready"
    except Exception:
        documents[doc_id]["status"] = "failed"

class Citation(BaseModel):
    document_id: str
    chunk: int
    score: float
    text: str

class QueryIn(BaseModel):
    question: str = Field(min_length=3, max_length=500)
    top_k: int = Field(default=2, ge=1, le=5)

class QueryOut(BaseModel):
    answer: str
    citations: list[Citation]

rag_app = FastAPI(title="Mini RAG")

@rag_app.post("/v1/documents", status_code=status.HTTP_202_ACCEPTED)
async def upload(background: BackgroundTasks, file: Annotated[UploadFile, File()]):
    if file.content_type not in {"text/plain", "text/markdown"}:
        raise HTTPException(415, "Only .txt and .md files are supported")
    data = await file.read(MAX_UPLOAD_BYTES + 1)
    if len(data) > MAX_UPLOAD_BYTES:
        raise HTTPException(413, "File too large")
    digest = hashlib.sha256(data).hexdigest()
    for doc_id, doc in documents.items():
        if doc["hash"] == digest:
            return {"id": doc_id, "status": doc["status"], "duplicate": True}
    doc_id = uuid.uuid4().hex[:8]
    documents[doc_id] = {"name": file.filename, "status": "processing", "hash": digest}
    background.add_task(process_document, doc_id, data.decode("utf-8", errors="replace"))
    return {"id": doc_id, "status": "processing", "duplicate": False}

@rag_app.get("/v1/documents/{doc_id}")
def document_status(doc_id: str):
    if doc_id not in documents:
        raise HTTPException(404, "Document not found")
    return {"id": doc_id, **{k: v for k, v in documents[doc_id].items() if k != "hash"}}

@rag_app.post("/v1/query")
def query(body: QueryIn) -> QueryOut:
    q = terms(body.question)
    scored = [(len(q & c["terms"]) / (len(q) or 1), c) for c in chunks]
    top = [(s, c) for s, c in sorted(scored, key=lambda x: -x[0]) if s > 0][: body.top_k]
    if not top:
        return QueryOut(answer="I could not find this in the uploaded documents.", citations=[])
    context = " ".join(c["text"] for _, c in top)
    answer = f"Based on the documents: {context}"            # a real app sends question + context to an LLM here
    return QueryOut(answer=answer, citations=[Citation(document_id=c["document_id"], chunk=c["index"],
                                                       score=round(s, 2), text=c["text"]) for s, c in top])

policy = (b"Patients must fast for eight hours before surgery. Water is allowed until two hours before. "
          b"Visiting hours are from ten to six. Children under twelve must be accompanied by an adult.")
client = TestClient(rag_app)
up = client.post("/v1/documents", files={"file": ("policy.txt", policy, "text/plain")})
print(up.status_code, up.json()["status"])                 # 202 processing
doc_id = up.json()["id"]
print(client.get(f"/v1/documents/{doc_id}").json()["status"])   # ready  (TestClient ran the background task)
res = client.post("/v1/query", json={"question": "How many hours must patients fast before surgery?"}).json()
print(res["citations"][0]["chunk"], res["citations"][0]["score"])   # 0 0.86
print(res["answer"][:70])                                  # Based on the documents: Patients must fast for eight hours before surg
print(client.post("/v1/documents", files={"file": ("p2.txt", policy, "text/plain")}).json()["duplicate"])   # True
print(client.post("/v1/documents", files={"file": ("x.exe", b"MZ", "application/octet-stream")}).status_code)   # 415

202 processing
ready
0 0.86
Based on the documents: Patients must fast for eight hours before surg


True
415


### Your own project follows this pattern

Your Healthcare RAG project (`healthcare_rag_new`, a FastAPI app) uses the same architecture. Reading its `app/main.py` shows:

* An **app factory** `create_app(settings=None, services=None)`, so tests can inject settings and fake services.
* A **lifespan** that creates the `Services` object and **warms up the models in a background thread**, so `/api/health` answers
  immediately while models load.
* **Liveness vs readiness:** `/api/health` always returns `{"status": "ok"}`; `/api/ready` returns **503** until the models are loaded
  *and* the LLM API key is configured (Part 1.5).
* `POST /api/documents` returns **202 Accepted**, detects duplicates ("This file is already uploaded."), and processes the file with
  **BackgroundTasks** (`services.process_document`).
* `POST /api/query` answers with a `QueryResponse` model and turns LLM failures into **503** ("The language model is unavailable right now").
* **API-key** and **admin** dependencies (`require_api_key`, `require_admin`) on routes, **rate limits** with `slowapi` on upload and
  query, CORS limited to configured origins, a request-logging middleware with durations, a catch-all exception handler that logs
  and returns a generic 500, and a size check that returns **413** for large uploads.

So you can explain it in interviews with the vocabulary of this notebook: lifespan, readiness, 202 + background processing,
dependencies for auth, rate limiting, consistent error handling and timeouts.

> **Interview tip:** when you describe a RAG project, cover the full path: ingestion (parse, chunk, embed, store), retrieval
> (top-k, filters, re-ranking), generation (prompt with context, citations, "I don't know"), and operations (background processing,
> caching, rate limits, evaluation of answer quality, privacy of documents).

In [54]:
# A console client for the mini RAG: ask questions until "quit"
def rag_cli(client):
    while True:
        question = input("Ask a question (or 'quit'): ").strip()
        if question.lower() == "quit":
            break
        r = client.post("/v1/query", json={"question": question})
        if r.status_code != 200:
            print("  Error:", r.json()["detail"][0]["msg"])
            continue
        data = r.json()
        print("  Answer:", data["answer"][:90])
        for c in data["citations"]:
            print(f"  [source {c['document_id']} chunk {c['chunk']}, score {c['score']}]")

use_sample_input("Can children visit alone?", "Hi", "What are the parking rules?", "quit")
rag_cli(client)
# Ask a question (or 'quit'): Can children visit alone?
#   Answer: Based on the documents: Visiting hours are from ten to six. Children under twelve ...
#   [source <id> chunk 1, score 0.33]
# Ask a question (or 'quit'): Hi
#   Error: String should have at least 3 characters
# Ask a question (or 'quit'): What are the parking rules?
#   Answer: I could not find this in the uploaded documents.
# Ask a question (or 'quit'): quit

Ask a question (or 'quit'): Can children visit alone?
  Answer: Based on the documents: Visiting hours are from ten to six. Children under twelve must be 
  [source 9f9a5c26 chunk 1, score 0.33]
Ask a question (or 'quit'): Hi
  Error: String should have at least 3 characters
Ask a question (or 'quit'): What are the parking rules?
  Answer: I could not find this in the uploaded documents.
Ask a question (or 'quit'): quit


### Practice 6.5: RAG API

**A. Predict**

1. Why does the upload endpoint return 202 and not 201?
2. What does the client see from `GET /v1/documents/{id}` if `process_document` raises an exception?

**B. Write code**

3. Upload a second document about parking (`b"Parking is free for patients. Visitors pay at the gate."`), then check that the question
   "Is parking free for patients?" returns a citation from the **new** document id.
4. Add an endpoint `DELETE /v1/documents/{doc_id}` to `rag_app` that removes the document and all its chunks (204), or 404. Check that
   after deleting the parking document, the parking question returns no citations.

**C. Explain:** the processing runs in `BackgroundTasks`. What happens to a document that is "processing" when the server restarts?
How would you fix it?

### Solutions 6.5

1. The work is not finished: the document exists but is still being processed. 202 says "accepted for processing"; the client polls
   the status URL.
2. `"failed"`, because `process_document` catches the exception and sets the status.
3. and 4. See the cell below.

**C.** The task is lost and the status stays "processing" forever. Fixes: a durable task queue (Celery/RQ/Arq) that retries jobs, or a
startup job that re-queues documents stuck in "processing", plus a timeout that marks them failed.

In [55]:
# Solutions 6.5
from fastapi import HTTPException, Response
from fastapi.testclient import TestClient

parking = b"Parking is free for patients. Visitors pay at the gate."
new_id = client.post("/v1/documents", files={"file": ("parking.txt", parking, "text/plain")}).json()["id"]
res = client.post("/v1/query", json={"question": "Is parking free for patients?"}).json()
assert res["citations"][0]["document_id"] == new_id

@rag_app.delete("/v1/documents/{doc_id}", status_code=204)
def delete_document(doc_id: str) -> None:
    if documents.pop(doc_id, None) is None:
        raise HTTPException(404, "Document not found")
    chunks[:] = [c for c in chunks if c["document_id"] != doc_id]

assert client.delete(f"/v1/documents/{new_id}").status_code == 204
assert client.delete(f"/v1/documents/{new_id}").status_code == 404
res = client.post("/v1/query", json={"question": "Is parking free for patients?"}).json()
assert all(c["document_id"] != new_id for c in res["citations"])
print("Practice 6.5: all checks passed")

Practice 6.5: all checks passed


---

## How to debug ML/LLM services

| Symptom | Likely cause | Fix |
|---|---|---|
| First request is very slow, then fast | model loaded lazily on first call | load in the lifespan; gate traffic with readiness |
| Memory grows with workers | each worker holds a model copy | fewer workers, shared model server |
| All requests slow during inference | CPU work in `async def` | `def`, thread pool, process pool or model server |
| SSE arrives all at once at the end | proxy buffering, or a sync blocking generator | `EventSourceResponse` (sends `X-Accel-Buffering: no`), proxy config, async generator |
| Stream stops after ~60 s idle | proxy idle timeout | keep-alive pings (built in every 15 s), proxy timeouts |
| LLM errors become 500s | no timeout/retry/error mapping | timeouts, retries with backoff, 503/504 mapping |
| Big bill from the LLM provider | no caching or rate limits; generation continues after disconnect | cache answers, rate limit per key, stop on disconnect |
| RAG answers without sources | retrieval returns nothing relevant | similarity threshold, "I don't know" answer, evaluate retrieval |

## Summary

* Load models once in the lifespan; use dependencies; typed input/output; return the model version.
* Keep the event loop free: threads for blocking calls, processes or model servers for CPU work; micro-batch for throughput.
* Stream tokens with `fastapi.sse` (`EventSourceResponse`, `ServerSentEvent`), FastAPI 0.162.
* Call LLM APIs with timeouts, retries (backoff + jitter, only for temporary errors) and clear 503/504 errors.
* RAG API: 202 upload + background processing + status, then query with citations.

# Part 7: Project 1, a Production-Style Task Manager API

**What this Part covers**

A complete, portfolio-quality REST API that combines everything from Notebooks 1-3:

* Settings from the environment, SQLAlchemy 2.0 models on SQLite (Postgres in production).
* Users with **password hashing** (Argon2 via `pwdlib`) and **JWT login** (OAuth2 password flow).
* Tasks **owned by users**: every query is filtered by the owner, so users only see their own tasks.
* Pagination, filtering and sorting; a consistent **RFC 9457** error format; a **request-ID middleware**.
* An **app factory** (`create_app(settings)`) so tests get a fresh app with an in-memory database.
* A full test suite run inside the notebook, and an `input()`-driven **console client**.

**Before you start:** Notebook 2, Parts 1, 2, 4, 5 and 8 (DI, structure, SQLAlchemy, security, testing) and Part 5 of this notebook.

> **Note:** in a real repository each section below is its own file. The cells are in the same order, so you can copy them out.

## 7.1 Project layout and design

```text
task_manager/
  app/
    __init__.py
    config.py        # Settings (pydantic-settings)                       -> 7.2
    db.py            # engine, session dependency, Base                     -> 7.2
    models.py        # User, Task (SQLAlchemy 2.0)                          -> 7.2
    schemas.py       # Pydantic request/response models                    -> 7.3
    security.py      # hashing, JWT, get_current_user                       -> 7.4
    errors.py        # problem details handlers + request-ID middleware     -> 7.5
    routers/
      auth.py        # POST /v1/auth/register, POST /v1/auth/login          -> 7.6
      tasks.py       # /v1/tasks CRUD                                       -> 7.6
    main.py          # create_app(settings) factory                         -> 7.7
  tests/
    conftest.py      # fixtures: app with in-memory DB, client, auth headers
    test_auth.py
    test_tasks.py
  alembic/           # migrations (production)
  Dockerfile, docker-compose.yml, pyproject.toml, .env.example
```

**Endpoints**

| Method | Path | Auth | Success | Errors |
|---|---|---|---|---|
| POST | `/v1/auth/register` | no | 201 user | 409 username taken, 422 |
| POST | `/v1/auth/login` | no (form data) | 200 token | 401 wrong credentials |
| GET | `/v1/users/me` | Bearer | 200 user | 401 |
| POST | `/v1/tasks` | Bearer | 201 task + Location | 401, 422 |
| GET | `/v1/tasks?limit&offset&done&sort` | Bearer | 200 page | 401, 422 |
| GET | `/v1/tasks/{id}` | Bearer | 200 task | 401, 404 |
| PATCH | `/v1/tasks/{id}` | Bearer | 200 task | 401, 404, 422 |
| DELETE | `/v1/tasks/{id}` | Bearer | 204 | 401, 404 |
| GET | `/healthz`, `/readyz` | no | 200 | 503 |

**Authorization rule:** a task that belongs to another user returns **404**, not 403. Returning 403 would tell an attacker that the
task ID exists.

---

## 7.2 Settings, database and models

* `Settings` reads `TASKAPP_*` environment variables (Part 1.7); the defaults are for development only.
* The engine is created **once per app** in the factory and stored on `app.state`; `get_db` opens one session per request.
* `connect_args={"check_same_thread": False}` is needed for SQLite because `def` endpoints run in a thread pool.
* `owner_id` is indexed: every task query filters by it.

In [56]:
# app/config.py, app/db.py, app/models.py
from collections.abc import Iterator
from datetime import UTC, datetime
from pathlib import Path

from fastapi import Request
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict
from sqlalchemy import ForeignKey, String, create_engine
from sqlalchemy.engine import Engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship
from sqlalchemy.pool import StaticPool

Path("fastapi_lab_3").mkdir(exist_ok=True)

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="TASKAPP_")
    app_name: str = "Task Manager API"
    database_url: str = "sqlite:///fastapi_lab_3/tasks.db"
    jwt_secret: SecretStr = SecretStr("dev-only-change-me-in-production-0123456789")
    jwt_algorithm: str = "HS256"
    access_token_minutes: int = 30
    max_page_size: int = 50

def make_engine(url: str) -> Engine:
    if url.startswith("sqlite"):
        kwargs = {"connect_args": {"check_same_thread": False}}
        if url in ("sqlite://", "sqlite:///:memory:"):
            kwargs["poolclass"] = StaticPool          # one shared in-memory DB for all sessions (tests)
        return create_engine(url, **kwargs)
    return create_engine(url, pool_size=10, max_overflow=20, pool_pre_ping=True)

def get_db(request: Request) -> Iterator[Session]:
    with Session(request.app.state.engine) as session:
        yield session

def utcnow() -> datetime:
    return datetime.now(UTC)

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    username: Mapped[str] = mapped_column(String(30), unique=True, index=True)
    password_hash: Mapped[str] = mapped_column(String(255))
    created_at: Mapped[datetime] = mapped_column(default=utcnow)
    tasks: Mapped[list["Task"]] = relationship(back_populates="owner", cascade="all, delete-orphan")

class Task(Base):
    __tablename__ = "tasks"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    description: Mapped[str | None] = mapped_column(String(2000), default=None)
    done: Mapped[bool] = mapped_column(default=False)
    created_at: Mapped[datetime] = mapped_column(default=utcnow)
    owner_id: Mapped[int] = mapped_column(ForeignKey("users.id"), index=True)
    owner: Mapped[User] = relationship(back_populates="tasks")

print(Settings().database_url)       # sqlite:///fastapi_lab_3/tasks.db

sqlite:///fastapi_lab_3/tasks.db

---

## 7.3 Schemas

Separate models for input and output: the client never sends `id` or `owner_id`, and the API never returns `password_hash`.
`Page[T]` is a **generic** model, so every list endpoint returns the same envelope.

In [57]:
# app/schemas.py
from datetime import datetime
from typing import Generic, TypeVar

from pydantic import BaseModel, ConfigDict, Field

T = TypeVar("T")

class UserCreate(BaseModel):
    username: str = Field(min_length=3, max_length=30, pattern=r"^[a-z0-9_]+$")
    password: str = Field(min_length=8, max_length=128)

class UserOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)     # read from SQLAlchemy objects
    id: int
    username: str

class TokenOut(BaseModel):
    access_token: str
    token_type: str = "bearer"
    expires_in: int

class TaskCreate(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    description: str | None = Field(default=None, max_length=2000)

class TaskUpdate(BaseModel):
    title: str | None = Field(default=None, min_length=1, max_length=200)
    description: str | None = Field(default=None, max_length=2000)
    done: bool | None = None

class TaskOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    title: str
    description: str | None
    done: bool
    created_at: datetime

class Page(BaseModel, Generic[T]):
    items: list[T]
    total: int
    limit: int
    offset: int

print(Page[TaskOut].model_json_schema()["properties"].keys())   # dict_keys(['items', 'total', 'limit', 'offset'])

dict_keys(['items', 'total', 'limit', 'offset'])


---

## 7.4 Security: hashing, JWT and the current user

* **Hashing:** `pwdlib`'s recommended hasher is **Argon2** (slow on purpose, salted). Never store plain passwords.
  (Notebook 2, Part 5 explains why; `passlib`, used by many older tutorials, is no longer maintained.)
* **JWT:** signed with HS256 and the secret from settings, with `sub` (user ID as a string), `iat` and `exp`. On decode we pass
  `algorithms=[...]` explicitly and require `exp` and `sub`.
* **Login timing:** when the username does not exist we still verify against a dummy hash, so attackers cannot discover valid
  usernames by measuring response times.
* Every auth failure is a **401** with `WWW-Authenticate: Bearer`.

In [58]:
# app/security.py
from datetime import UTC, datetime, timedelta
from typing import Annotated

import jwt
from fastapi import Depends, HTTPException, Request, status
from fastapi.security import OAuth2PasswordBearer
from pwdlib import PasswordHash
from sqlalchemy import select
from sqlalchemy.orm import Session

password_hash = PasswordHash.recommended()
DUMMY_HASH = password_hash.hash("dummy-password-for-timing")
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/v1/auth/login")

def get_settings(request: Request) -> Settings:
    return request.app.state.settings

def create_access_token(user_id: int, settings: Settings, minutes: int | None = None) -> str:
    now = datetime.now(UTC)
    exp = now + timedelta(minutes=settings.access_token_minutes if minutes is None else minutes)
    payload = {"sub": str(user_id), "iat": now, "exp": exp}
    return jwt.encode(payload, settings.jwt_secret.get_secret_value(), algorithm=settings.jwt_algorithm)

def authenticate(db: Session, username: str, password: str) -> User | None:
    user = db.scalar(select(User).where(User.username == username))
    if user is None:
        password_hash.verify(password, DUMMY_HASH)      # same work as a real check
        return None
    return user if password_hash.verify(password, user.password_hash) else None

CREDENTIALS_ERROR = HTTPException(status.HTTP_401_UNAUTHORIZED, "Could not validate credentials",
                                  headers={"WWW-Authenticate": "Bearer"})

def get_current_user(token: Annotated[str, Depends(oauth2_scheme)],
                     db: Annotated[Session, Depends(get_db)],
                     settings: Annotated[Settings, Depends(get_settings)]) -> User:
    try:
        payload = jwt.decode(token, settings.jwt_secret.get_secret_value(),
                             algorithms=[settings.jwt_algorithm], options={"require": ["exp", "sub"]})
        user_id = int(payload["sub"])
    except (jwt.InvalidTokenError, ValueError):
        raise CREDENTIALS_ERROR
    user = db.get(User, user_id)
    if user is None:
        raise CREDENTIALS_ERROR
    return user

CurrentUser = Annotated[User, Depends(get_current_user)]
DB = Annotated[Session, Depends(get_db)]
print(password_hash.hash("secret123")[:9])     # $argon2id

$argon2id

---

## 7.5 Errors and request IDs

Every error, including 404 for unknown routes and 422 validation errors, uses the problem-details format from Part 5.5 and carries
the request ID. The request-ID middleware also catches unexpected exceptions itself, because a handler for `Exception` would run
outside the middleware (Part 4.2, Practice).

In [59]:
# app/errors.py
import logging
import uuid
from contextvars import ContextVar
from http import HTTPStatus

from fastapi import FastAPI, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from starlette.exceptions import HTTPException as StarletteHTTPException

task_log = logging.getLogger("taskapp")
request_id_ctx: ContextVar[str] = ContextVar("taskapp_request_id", default="-")

def problem_response(status: int, detail, request: Request, **extra) -> JSONResponse:
    body = {"type": "about:blank", "title": HTTPStatus(status).phrase, "status": status, "detail": detail,
            "instance": request.url.path, "request_id": request_id_ctx.get(), **extra}
    return JSONResponse(body, status_code=status, media_type="application/problem+json")

def install_error_handling(app: FastAPI) -> None:
    @app.middleware("http")
    async def request_id_middleware(request: Request, call_next):
        rid = request.headers.get("X-Request-ID") or uuid.uuid4().hex[:16]
        token = request_id_ctx.set(rid)
        try:
            response = await call_next(request)
        except Exception:
            task_log.exception("unhandled error")
            response = problem_response(500, "Internal server error", request)
        finally:
            request_id_ctx.reset(token)
        response.headers["X-Request-ID"] = rid
        return response

    @app.exception_handler(StarletteHTTPException)
    async def http_error(request: Request, exc: StarletteHTTPException):
        response = problem_response(exc.status_code, exc.detail, request)
        if exc.headers:
            response.headers.update(exc.headers)
        return response

    @app.exception_handler(RequestValidationError)
    async def validation_error(request: Request, exc: RequestValidationError):
        errors = [{"loc": list(e["loc"]), "msg": e["msg"]} for e in exc.errors()]
        return problem_response(422, "Validation failed", request, errors=errors)

print("error handling ready")      # error handling ready

error handling ready


---

## 7.6 Routers: auth and tasks

The important line in every task query is `Task.owner_id == user.id`. Forgetting it is a classic security bug called **IDOR**
(Insecure Direct Object Reference): any user could read any task by guessing IDs.

In [60]:
# app/routers/auth.py and app/routers/tasks.py
from typing import Annotated, Literal

from fastapi import APIRouter, Depends, HTTPException, Query, Response, status
from fastapi.security import OAuth2PasswordRequestForm
from sqlalchemy import func, select
from sqlalchemy.exc import IntegrityError

auth_router = APIRouter(prefix="/v1", tags=["auth"])

@auth_router.post("/auth/register", status_code=status.HTTP_201_CREATED)
def register(body: UserCreate, db: DB) -> UserOut:
    user = User(username=body.username, password_hash=password_hash.hash(body.password))
    db.add(user)
    try:
        db.commit()
    except IntegrityError:                     # the unique constraint is the real guard (no race condition)
        db.rollback()
        raise HTTPException(status.HTTP_409_CONFLICT, "Username is already taken")
    db.refresh(user)
    return user

@auth_router.post("/auth/login")
def login(form: Annotated[OAuth2PasswordRequestForm, Depends()], db: DB,
          settings: Annotated[Settings, Depends(get_settings)]) -> TokenOut:
    user = authenticate(db, form.username, form.password)
    if user is None:
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "Incorrect username or password",
                            headers={"WWW-Authenticate": "Bearer"})
    return TokenOut(access_token=create_access_token(user.id, settings),
                    expires_in=settings.access_token_minutes * 60)

@auth_router.get("/users/me")
def me(user: CurrentUser) -> UserOut:
    return user

tasks_router = APIRouter(prefix="/v1/tasks", tags=["tasks"])

def get_owned_task(task_id: int, user: CurrentUser, db: DB) -> Task:
    task = db.scalar(select(Task).where(Task.id == task_id, Task.owner_id == user.id))
    if task is None:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "Task not found")   # also for other users' tasks
    return task

OwnedTask = Annotated[Task, Depends(get_owned_task)]

@tasks_router.post("", status_code=status.HTTP_201_CREATED)
def create_task(body: TaskCreate, user: CurrentUser, db: DB, response: Response) -> TaskOut:
    task = Task(**body.model_dump(), owner_id=user.id)
    db.add(task)
    db.commit()
    db.refresh(task)
    response.headers["Location"] = f"/v1/tasks/{task.id}"
    return task

SORTS = {"created_at": (Task.created_at.asc(), Task.id.asc()), "-created_at": (Task.created_at.desc(), Task.id.desc()),
         "title": (Task.title.asc(), Task.id.asc())}

@tasks_router.get("")
def list_tasks(user: CurrentUser, db: DB, settings: Annotated[Settings, Depends(get_settings)],
               limit: Annotated[int, Query(ge=1)] = 10, offset: Annotated[int, Query(ge=0)] = 0,
               done: bool | None = None,
               sort: Literal["created_at", "-created_at", "title"] = "created_at") -> Page[TaskOut]:
    limit = min(limit, settings.max_page_size)
    conditions = [Task.owner_id == user.id]
    if done is not None:
        conditions.append(Task.done == done)
    total = db.scalar(select(func.count()).select_from(Task).where(*conditions))
    rows = db.scalars(select(Task).where(*conditions).order_by(*SORTS[sort]).limit(limit).offset(offset)).all()
    return Page[TaskOut](items=rows, total=total, limit=limit, offset=offset)

@tasks_router.get("/{task_id}")
def read_task(task: OwnedTask) -> TaskOut:
    return task

@tasks_router.patch("/{task_id}")
def update_task(body: TaskUpdate, task: OwnedTask, db: DB) -> TaskOut:
    for field, value in body.model_dump(exclude_unset=True).items():
        setattr(task, field, value)
    db.commit()
    db.refresh(task)
    return task

@tasks_router.delete("/{task_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_task(task: OwnedTask, db: DB) -> None:
    db.delete(task)
    db.commit()

print("routers ready")     # routers ready

routers ready


> **Note:** `get_owned_task` uses `user: CurrentUser` and `db: DB`, and the endpoint also uses `db: DB`. FastAPI **caches** a dependency
> within one request, so both get the *same* session (Notebook 2, Part 1). That is why `db.commit()` in the endpoint saves changes to the
> task loaded by the dependency.

---

## 7.7 The app factory

`create_app(settings)` builds a new, independent app. Production calls it once (`app = create_app()` in `main.py`); each test calls it
with an in-memory database. The lifespan creates the tables (a real project runs Alembic migrations instead) and disposes the engine
on shutdown, so the SQLite file is released.

In [61]:
# app/main.py
from contextlib import asynccontextmanager

from fastapi import FastAPI, Response
from sqlalchemy import text

def create_app(settings: Settings | None = None) -> FastAPI:
    settings = settings or Settings()

    @asynccontextmanager
    async def lifespan(app: FastAPI):
        Base.metadata.create_all(app.state.engine)     # demo only: use Alembic migrations in production
        yield
        app.state.engine.dispose()

    app = FastAPI(title=settings.app_name, version="1.0.0", lifespan=lifespan)
    app.state.settings = settings
    app.state.engine = make_engine(settings.database_url)
    install_error_handling(app)
    app.include_router(auth_router)
    app.include_router(tasks_router)

    @app.get("/healthz", tags=["ops"])
    def healthz():
        return {"status": "alive"}

    @app.get("/readyz", tags=["ops"])
    def readyz(response: Response):
        try:
            with app.state.engine.connect() as conn:
                conn.execute(text("SELECT 1"))
            return {"ready": True}
        except Exception:
            response.status_code = 503
            return {"ready": False}

    return app

print(sorted(create_app(Settings(database_url="sqlite://")).openapi()["paths"]))
# ['/healthz', '/readyz', '/v1/auth/login', '/v1/auth/register', '/v1/tasks', '/v1/tasks/{task_id}', '/v1/users/me']

['/healthz', '/readyz', '/v1/auth/login', '/v1/auth/register', '/v1/tasks', '/v1/tasks/{task_id}', '/v1/users/me']


---

## 7.8 Try it

We run the app against a real SQLite **file** in `fastapi_lab_3/` (deleted first, so "Run All" always starts clean).

In [62]:
# A full user journey with TestClient
from pathlib import Path
from fastapi.testclient import TestClient

db_file = Path("fastapi_lab_3/tasks.db")
db_file.unlink(missing_ok=True)
task_app = create_app()

with TestClient(task_app) as client:
    print(client.post("/v1/auth/register", json={"username": "asha", "password": "secret123"}).status_code)   # 201
    print(client.post("/v1/auth/register", json={"username": "asha", "password": "secret123"}).json()["title"])  # Conflict
    tok = client.post("/v1/auth/login", data={"username": "asha", "password": "secret123"}).json()["access_token"]
    auth = {"Authorization": f"Bearer {tok}"}

    r = client.post("/v1/tasks", json={"title": "Prepare for interview"}, headers=auth)
    print(r.status_code, r.headers["location"])                  # 201 /v1/tasks/1
    client.post("/v1/tasks", json={"title": "Revise FastAPI notes"}, headers=auth)
    client.patch("/v1/tasks/1", json={"done": True}, headers=auth)
    page = client.get("/v1/tasks", params={"done": False}, headers=auth).json()
    print(page["total"], [t["title"] for t in page["items"]])    # 1 ['Revise FastAPI notes']

    err = client.get("/v1/tasks/999", headers={**auth, "X-Request-ID": "demo-1"})
    print(err.status_code, err.headers["content-type"])          # 404 application/problem+json
    print(err.json())
    # {'type': 'about:blank', 'title': 'Not Found', 'status': 404, 'detail': 'Task not found', 'instance': '/v1/tasks/999', 'request_id': 'demo-1'}
    print(client.get("/v1/tasks").status_code)                   # 401 (no token)
print(db_file.exists())                                          # True (data is in the SQLite file)

201


Conflict


201 /v1/tasks/1
1 ['Revise FastAPI notes']
404 application/problem+json
{'type': 'about:blank', 'title': 'Not Found', 'status': 404, 'detail': 'Task not found', 'instance': '/v1/tasks/999', 'request_id': 'demo-1'}
401
True


---

## 7.9 The test suite

In a real project, `tests/conftest.py` would contain fixtures like this (pytest runs them for every test):

```python
# tests/conftest.py
import pytest
from fastapi.testclient import TestClient
from app.config import Settings
from app.main import create_app

@pytest.fixture
def client():
    app = create_app(Settings(database_url="sqlite://", jwt_secret="test-secret-0123456789-abcdefghij"))
    with TestClient(app) as c:          # runs the lifespan: tables are created
        yield c

@pytest.fixture
def auth_headers(client):
    def make(username="asha", password="secret123"):
        client.post("/v1/auth/register", json={"username": username, "password": password})
        r = client.post("/v1/auth/login", data={"username": username, "password": password})
        return {"Authorization": f"Bearer {r.json()['access_token']}"}
    return make
```

Each test gets a **fresh app and a fresh in-memory database**, so tests never interfere with each other (Part 11 shows what happens
when they do). Inside the notebook we imitate pytest with a tiny runner: the fixtures become helper functions.

In [63]:
# tests/test_auth.py + tests/test_tasks.py, run with a tiny pytest-like runner
import jwt
from contextlib import contextmanager
from fastapi.testclient import TestClient

TEST_SETTINGS = dict(database_url="sqlite://", jwt_secret="test-secret-0123456789-abcdefghij")

@contextmanager
def fresh_client():
    with TestClient(create_app(Settings(**TEST_SETTINGS))) as c:
        yield c

def login_headers(client, username="asha", password="secret123"):
    client.post("/v1/auth/register", json={"username": username, "password": password})
    r = client.post("/v1/auth/login", data={"username": username, "password": password})
    return {"Authorization": f"Bearer {r.json()['access_token']}"}

def test_register_returns_user_without_password():
    with fresh_client() as c:
        r = c.post("/v1/auth/register", json={"username": "asha", "password": "secret123"})
        assert r.status_code == 201 and r.json() == {"id": 1, "username": "asha"}

def test_register_validates_input():
    with fresh_client() as c:
        r = c.post("/v1/auth/register", json={"username": "A!", "password": "short"})
        assert r.status_code == 422
        assert {tuple(e["loc"]) for e in r.json()["errors"]} == {("body", "username"), ("body", "password")}

def test_duplicate_username_is_409():
    with fresh_client() as c:
        login_headers(c)
        r = c.post("/v1/auth/register", json={"username": "asha", "password": "another123"})
        assert r.status_code == 409

def test_login_wrong_password_is_401_with_header():
    with fresh_client() as c:
        login_headers(c)
        r = c.post("/v1/auth/login", data={"username": "asha", "password": "wrong-pass"})
        assert r.status_code == 401 and r.headers["www-authenticate"] == "Bearer"

def test_tampered_or_expired_token_is_401():
    with fresh_client() as c:
        headers = login_headers(c)
        token = headers["Authorization"].split()[1]
        assert c.get("/v1/users/me", headers={"Authorization": f"Bearer {token[:-2]}xx"}).status_code == 401
        expired = create_access_token(1, Settings(**TEST_SETTINGS), minutes=-1)
        assert c.get("/v1/users/me", headers={"Authorization": f"Bearer {expired}"}).status_code == 401
        forged = jwt.encode({"sub": "1", "exp": 9999999999}, "attacker-secret-0123456789-abcdef", algorithm="HS256")
        assert c.get("/v1/users/me", headers={"Authorization": f"Bearer {forged}"}).status_code == 401

def test_create_and_read_task():
    with fresh_client() as c:
        h = login_headers(c)
        r = c.post("/v1/tasks", json={"title": "Write tests"}, headers=h)
        assert r.status_code == 201 and r.headers["location"] == "/v1/tasks/1"
        assert c.get("/v1/tasks/1", headers=h).json()["title"] == "Write tests"

def test_users_cannot_see_each_others_tasks():
    with fresh_client() as c:
        asha, ravi = login_headers(c, "asha"), login_headers(c, "ravi")
        c.post("/v1/tasks", json={"title": "Asha's secret"}, headers=asha)
        assert c.get("/v1/tasks/1", headers=ravi).status_code == 404
        assert c.patch("/v1/tasks/1", json={"done": True}, headers=ravi).status_code == 404
        assert c.delete("/v1/tasks/1", headers=ravi).status_code == 404
        assert c.get("/v1/tasks", headers=ravi).json()["total"] == 0
        assert c.get("/v1/tasks/1", headers=asha).json()["done"] is False

def test_pagination_filter_and_sort():
    with fresh_client() as c:
        h = login_headers(c)
        for title in ["b", "d", "a", "c", "e"]:
            c.post("/v1/tasks", json={"title": title}, headers=h)
        c.patch("/v1/tasks/1", json={"done": True}, headers=h)
        page = c.get("/v1/tasks", params={"limit": 2, "offset": 2, "sort": "title"}, headers=h).json()
        assert [t["title"] for t in page["items"]] == ["c", "d"] and page["total"] == 5
        assert c.get("/v1/tasks", params={"done": True}, headers=h).json()["total"] == 1
        assert c.get("/v1/tasks", params={"limit": 1000}, headers=h).json()["limit"] == 50
        assert c.get("/v1/tasks", params={"sort": "owner_id"}, headers=h).status_code == 422

def test_patch_only_changes_sent_fields():
    with fresh_client() as c:
        h = login_headers(c)
        c.post("/v1/tasks", json={"title": "t", "description": "keep me"}, headers=h)
        r = c.patch("/v1/tasks/1", json={"done": True}, headers=h).json()
        assert r["description"] == "keep me" and r["done"] is True

def test_delete_then_404():
    with fresh_client() as c:
        h = login_headers(c)
        c.post("/v1/tasks", json={"title": "temp"}, headers=h)
        assert c.delete("/v1/tasks/1", headers=h).status_code == 204
        assert c.get("/v1/tasks/1", headers=h).status_code == 404

def test_errors_have_problem_format_and_request_id():
    with fresh_client() as c:
        r = c.get("/v1/nope", headers={"X-Request-ID": "t-42"})
        assert r.headers["content-type"] == "application/problem+json"
        assert r.json()["request_id"] == "t-42" and r.headers["x-request-id"] == "t-42"

def test_health_and_readiness():
    with fresh_client() as c:
        assert c.get("/healthz").status_code == 200 and c.get("/readyz").json() == {"ready": True}

def run_tests(namespace):
    tests = [(name, fn) for name, fn in namespace.items() if name.startswith("test_") and callable(fn)]
    failed = 0
    for name, fn in tests:
        try:
            fn()
            print(f"PASSED {name}")
        except AssertionError as e:
            failed += 1
            print(f"FAILED {name}: {e!r}")
    print(f"{len(tests) - failed} passed, {failed} failed")
    return failed

assert run_tests(dict(globals())) == 0
# PASSED test_register_returns_user_without_password
# ...
# 12 passed, 0 failed

PASSED test_register_returns_user_without_password


PASSED test_register_validates_input


PASSED test_duplicate_username_is_409


PASSED test_login_wrong_password_is_401_with_header


PASSED test_tampered_or_expired_token_is_401


PASSED test_create_and_read_task


PASSED test_users_cannot_see_each_others_tasks


PASSED test_pagination_filter_and_sort


PASSED test_patch_only_changes_sent_fields


PASSED test_delete_then_404
PASSED test_errors_have_problem_format_and_request_id
PASSED test_health_and_readiness
12 passed, 0 failed


---

## 7.10 A console client with `input()`

A small terminal program for people who want to use the API without a frontend. It shows a menu, asks for choices with `input()`, and
calls the API (here through TestClient; against a real server you would use `httpx.Client(base_url="http://localhost:8000")`, and the
rest of the code stays the same).

In [64]:
# Console client: register, login, add a task, list tasks
def task_console(client):
    token = None
    while True:
        print("\n1) Register  2) Login  3) Add task  4) List tasks  5) Mark done  0) Quit")
        choice = input("Choose: ").strip()
        if choice == "0":
            print("Goodbye!")
            return
        if choice in ("1", "2"):
            username = input("Username: ").strip()
            password = input("Password: ")             # a real CLI would use getpass.getpass() to hide it
            if choice == "1":
                r = client.post("/v1/auth/register", json={"username": username, "password": password})
                print("Registered." if r.status_code == 201 else f"Error: {r.json()['detail']}")
            else:
                r = client.post("/v1/auth/login", data={"username": username, "password": password})
                if r.status_code == 200:
                    token = r.json()["access_token"]
                    print("Logged in.")
                else:
                    print("Error:", r.json()["detail"])
            continue
        if token is None:
            print("Please log in first.")
            continue
        headers = {"Authorization": f"Bearer {token}"}
        if choice == "3":
            title = input("Task title: ").strip()
            r = client.post("/v1/tasks", json={"title": title}, headers=headers)
            print(f"Added task #{r.json()['id']}." if r.status_code == 201 else f"Error: {r.json()['detail']}")
        elif choice == "4":
            page = client.get("/v1/tasks", headers=headers).json()
            for t in page["items"]:
                print(f"  #{t['id']} [{'x' if t['done'] else ' '}] {t['title']}")
            print(f"  ({page['total']} tasks)")
        elif choice == "5":
            task_id = input("Task number: ").strip()
            r = client.patch(f"/v1/tasks/{task_id}", json={"done": True}, headers=headers)
            print("Done!" if r.status_code == 200 else f"Error: {r.json()['detail']}")
        else:
            print("Unknown choice.")

use_sample_input("3",                                   # try to add before logging in
                 "1", "meera", "meera-pass-1",          # register
                 "2", "meera", "wrong",                 # failed login
                 "2", "meera", "meera-pass-1",          # login
                 "3", "Book mock interview",
                 "3", "Update CV",
                 "5", "2",
                 "4",
                 "0")
with TestClient(create_app(Settings(**TEST_SETTINGS))) as console_client:
    task_console(console_client)


1) Register  2) Login  3) Add task  4) List tasks  5) Mark done  0) Quit
Choose: 3
Please log in first.

1) Register  2) Login  3) Add task  4) List tasks  5) Mark done  0) Quit
Choose: 1
Username: meera
Password: meera-pass-1
Registered.

1) Register  2) Login  3) Add task  4) List tasks  5) Mark done  0) Quit
Choose: 2
Username: meera
Password: wrong


Error: Incorrect username or password

1) Register  2) Login  3) Add task  4) List tasks  5) Mark done  0) Quit
Choose: 2
Username: meera
Password: meera-pass-1


Logged in.

1) Register  2) Login  3) Add task  4) List tasks  5) Mark done  0) Quit
Choose: 3
Task title: Book mock interview
Added task #1.

1) Register  2) Login  3) Add task  4) List tasks  5) Mark done  0) Quit
Choose: 3
Task title: Update CV
Added task #2.

1) Register  2) Login  3) Add task  4) List tasks  5) Mark done  0) Quit
Choose: 5
Task number: 2
Done!

1) Register  2) Login  3) Add task  4) List tasks  5) Mark done  0) Quit
Choose: 4
  #1 [ ] Book mock interview
  #2 [x] Update CV
  (2 tasks)

1) Register  2) Login  3) Add task  4) List tasks  5) Mark done  0) Quit
Choose: 0
Goodbye!


### Practice 7: extend the project

**A. Predict**

1. Ravi requests `GET /v1/tasks/1`, which belongs to Asha. What status does he get, and why not 403?
2. A token was issued with `access_token_minutes=30`. The server restarts with a **new** `TASKAPP_JWT_SECRET`. What happens to that token?

**B. Write code**

3. Add `GET /v1/tasks/stats` returning `{"total": n, "done": n, "open": n}` for the current user only. Hint: declare it on a new
   router `stats_router` with the prefix `/v1/tasks` and include it **before** `tasks_router` in a new factory `create_app_v2`, otherwise
   `/v1/tasks/stats` is matched by `/v1/tasks/{task_id}` and returns 422. Test it with two users.
4. Write a test that `GET /v1/tasks?limit=0` returns 422 with the problem-details `errors` list mentioning `limit`.

**C. Find and fix the bug**

5. ```python
   @tasks_router.get("/{task_id}")
   def read_task(task_id: int, user: CurrentUser, db: DB) -> TaskOut:
       task = db.get(Task, task_id)
       if task is None:
           raise HTTPException(404, "Task not found")
       return task
   ```

### Solutions 7

1. 404 "Task not found". The query filters by owner, so for Ravi the task does not exist. A 403 would confirm that task 1 exists.
2. It becomes invalid (401): its signature was made with the old secret. Rotating the secret logs everyone out. Real systems support
   key rotation with several keys (a `kid` header) or use asymmetric keys (RS256/EdDSA).
3. and 4. See the cell below.
5. It does not check the owner: any logged-in user can read any task (IDOR). Use `get_owned_task` (filter by `Task.owner_id == user.id`).

In [65]:
# Solutions 7
from contextlib import asynccontextmanager
from fastapi import APIRouter, FastAPI
from fastapi.testclient import TestClient
from sqlalchemy import func, select

stats_router = APIRouter(prefix="/v1/tasks", tags=["tasks"])

@stats_router.get("/stats")
def task_stats(user: CurrentUser, db: DB) -> dict[str, int]:
    total = db.scalar(select(func.count()).select_from(Task).where(Task.owner_id == user.id))
    done = db.scalar(select(func.count()).select_from(Task).where(Task.owner_id == user.id, Task.done.is_(True)))
    return {"total": total, "done": done, "open": total - done}

def create_app_v2(settings: Settings) -> FastAPI:
    @asynccontextmanager
    async def lifespan(app: FastAPI):
        Base.metadata.create_all(app.state.engine)
        yield
        app.state.engine.dispose()

    app = FastAPI(title=settings.app_name, lifespan=lifespan)
    app.state.settings = settings
    app.state.engine = make_engine(settings.database_url)
    install_error_handling(app)
    app.include_router(stats_router)        # before tasks_router: "/stats" must win over "/{task_id}"
    app.include_router(auth_router)
    app.include_router(tasks_router)
    return app

with TestClient(create_app_v2(Settings(**TEST_SETTINGS))) as c:
    asha, ravi = login_headers(c, "asha"), login_headers(c, "ravi")
    for title in ["a", "b", "c"]:
        c.post("/v1/tasks", json={"title": title}, headers=asha)
    c.patch("/v1/tasks/2", json={"done": True}, headers=asha)
    c.post("/v1/tasks", json={"title": "ravi's"}, headers=ravi)
    print(c.get("/v1/tasks/stats", headers=asha).json())     # {'total': 3, 'done': 1, 'open': 2}
    assert c.get("/v1/tasks/stats", headers=asha).json() == {"total": 3, "done": 1, "open": 2}
    assert c.get("/v1/tasks/stats", headers=ravi).json() == {"total": 1, "done": 0, "open": 1}

def test_limit_zero_is_422():
    with fresh_client() as c:
        r = c.get("/v1/tasks", params={"limit": 0}, headers=login_headers(c))
        assert r.status_code == 422
        assert any(e["loc"] == ["query", "limit"] for e in r.json()["errors"])

test_limit_zero_is_422()
print("Practice 7: all checks passed")

{'total': 3, 'done': 1, 'open': 2}


Practice 7: all checks passed


---

## How to debug this project

| Symptom | Likely cause | Fix |
|---|---|---|
| `sqlite3.ProgrammingError: ... created in a thread ...` | SQLite + thread pool | `check_same_thread=False` |
| `no such table: users` in tests | lifespan not run (no `with TestClient(...)`), or a new in-memory DB per connection | use `with`; `StaticPool` for `sqlite://` |
| Every request is 401 | wrong header format (`Bearer <token>`), secret mismatch, `algorithms` missing in decode | print the decode error in a debug session |
| `/v1/tasks/stats` returns 422 | matched by `/{task_id}` first | declare the static route first |
| Users see others' tasks | query not filtered by owner | owner filter in one shared dependency |
| Tests pass alone, fail together | shared DB/state between tests | fresh app + DB per test (fixture) |
| `PermissionError` deleting `tasks.db` on Windows | the engine still holds the file | `engine.dispose()` (done in the lifespan shutdown) |

## Summary

* Structure: settings, db, models, schemas, security, errors, routers, factory; tests use the factory with an in-memory DB.
* Security: Argon2 hashes, signed JWTs with expiry, 401 + `WWW-Authenticate`, owner filters everywhere (404 for others' data).
* API quality: 201 + `Location`, 204, PATCH with `exclude_unset`, pagination with a capped limit, allow-listed sorting.
* Operations: problem-details errors with request IDs, health and readiness endpoints, lifespan-managed engine.

# Part 8: Project 2, an AI/LLM Microservice

**What this Part covers**

A small but realistic "chat completion" microservice, the kind many companies put in front of an LLM provider:

* A **fake LLM behind a clean interface** (a `Protocol`), so the real provider can be swapped in without touching the API code.
* `POST /v1/chat` (JSON answer) and `POST /v1/chat/stream` (**SSE** token streaming with `fastapi.sse`).
* A **TTL response cache** (same question -> instant answer, no LLM cost).
* **API-key auth** and **per-key rate limiting** (token bucket) with `429` + `Retry-After`.
* **Timeouts** with clear `504` errors, **health/readiness** endpoints, and a **test suite**.
* A console **chat client** using `input()`.

**Before you start:** Part 6 (SSE, timeouts), Notebook 2, Part 5 (API keys) and Part 10 (rate limiting and caching basics).

## 8.1 Architecture

```text
 client --X-API-Key--> [auth dependency] --> [rate limit dependency] --> endpoint
                              | 401                  | 429 + Retry-After       |
                                                                               v
                                                     cache hit? --yes--> answer (cached: true)
                                                         | no
                                                         v
                                        LLMClient (Protocol)  <-- FakeLLM in dev/tests
                                          with timeout              OpenAILLM / GroqLLM in prod
                                                         | 504 on timeout
                                                         v
                                                  store in cache -> answer / SSE tokens
```

Why each piece:

| Piece | Problem it solves |
|---|---|
| `LLMClient` protocol | API code does not depend on one vendor; tests use a fake |
| TTL cache | repeated questions cost nothing and return instantly |
| API keys | know who is calling; revoke a leaked key |
| Rate limiting per key | one client cannot use the whole LLM budget |
| Timeouts | a slow provider does not hold connections forever |
| Readiness | no traffic until the model/client is warmed up |

> **Note:** the cache and rate limiter here live **in memory**, so they are per worker (Part 1.3). In production with several workers or
> containers, keep them in **Redis**. The interfaces below are written so that swapping in a Redis version only changes one class.

---

## 8.2 The LLM interface and a fake implementation

A **`Protocol`** (from `typing`) describes *what methods an object must have*, without inheritance. Any class with matching methods
"is" an `LLMClient`. This is called **structural typing** ("duck typing" that the type checker understands).

In [66]:
# llm.py: the interface and a fake implementation
import asyncio
from collections.abc import AsyncIterator
from typing import Protocol, runtime_checkable

@runtime_checkable
class LLMClient(Protocol):
    model: str
    async def warm_up(self) -> None: ...
    def stream(self, prompt: str, max_tokens: int) -> AsyncIterator[str]: ...

class FakeLLM:
    """Deterministic fake: answers from a small table, streams word by word."""
    model = "fake-llm-1"
    ANSWERS = {
        "asgi": "ASGI is the async interface between Python web servers and apps like FastAPI.",
        "workers": "Use about one worker per CPU core, then measure with a load test.",
        "cors": "CORS lets a browser call an API on another origin when the server allows it.",
    }

    def __init__(self, token_delay: float = 0.0, warmup_delay: float = 0.0):
        self.token_delay, self.warmup_delay = token_delay, warmup_delay
        self.calls = 0

    async def warm_up(self) -> None:
        await asyncio.sleep(self.warmup_delay)       # e.g. open connections, check the API key

    async def stream(self, prompt: str, max_tokens: int):
        self.calls += 1
        topic = next((k for k in self.ANSWERS if k in prompt.lower()), None)
        answer = self.ANSWERS.get(topic, "I am a small fake model. Ask me about ASGI, workers or CORS.")
        for word in answer.split()[:max_tokens]:
            await asyncio.sleep(self.token_delay)
            yield word + " "

    async def complete(self, prompt: str, max_tokens: int) -> str:
        return "".join([t async for t in self.stream(prompt, max_tokens)]).strip()

llm = FakeLLM()
print(isinstance(llm, LLMClient))                          # True (it has the right methods)
print(await llm.complete("What is ASGI?", max_tokens=5))   # ASGI is the async interface

True
ASGI is the async interface


A real implementation would wrap a provider SDK or `httpx` with the timeout/retry logic from Part 6.4, for example:

```python
class OpenAICompatibleLLM:
    def __init__(self, http: httpx.AsyncClient, model: str):
        self.http, self.model = http, model

    async def warm_up(self) -> None:
        (await self.http.get("/models")).raise_for_status()      # fail readiness if the key is wrong

    async def stream(self, prompt: str, max_tokens: int):
        body = {"model": self.model, "stream": True, "max_tokens": max_tokens,
                "messages": [{"role": "user", "content": prompt}]}
        async with self.http.stream("POST", "/chat/completions", json=body) as r:
            r.raise_for_status()
            async for line in r.aiter_lines():                  # the provider streams SSE too
                if line.startswith("data: ") and line != "data: [DONE]":
                    delta = json.loads(line[6:])["choices"][0]["delta"].get("content")
                    if delta:
                        yield delta
```

---

## 8.3 Cache, rate limiter and settings

Both classes take a `clock` function, so tests can move time forward without sleeping.

**Token bucket** rate limiting: each API key has a bucket of `capacity` tokens that refills at `rate` tokens per second. A request takes one
token; an empty bucket means **429**, and `Retry-After` says how long until one token is back. It allows short bursts but limits the average.

In [67]:
# cache.py, ratelimit.py, config.py
import hashlib, math, time
from collections import OrderedDict
from pydantic import BaseModel, Field

class TTLCache:
    """In-memory TTL cache with a size limit (oldest entries are evicted first)."""
    def __init__(self, ttl: float, maxsize: int = 1000, clock=time.monotonic):
        self.ttl, self.maxsize, self.clock = ttl, maxsize, clock
        self._data: OrderedDict[str, tuple[float, str]] = OrderedDict()
        self.hits = self.misses = 0

    def get(self, key: str) -> str | None:
        item = self._data.get(key)
        if item is None or item[0] <= self.clock():
            self._data.pop(key, None)
            self.misses += 1
            return None
        self.hits += 1
        return item[1]

    def set(self, key: str, value: str) -> None:
        self._data[key] = (self.clock() + self.ttl, value)
        self._data.move_to_end(key)
        while len(self._data) > self.maxsize:
            self._data.popitem(last=False)

class TokenBucketLimiter:
    def __init__(self, capacity: int, per_seconds: float, clock=time.monotonic):
        self.capacity, self.rate, self.clock = capacity, capacity / per_seconds, clock
        self._buckets: dict[str, tuple[float, float]] = {}       # key -> (tokens, last_time)

    def hit(self, key: str) -> tuple[bool, int, float]:
        """Return (allowed, remaining, retry_after_seconds)."""
        now = self.clock()
        tokens, last = self._buckets.get(key, (self.capacity, now))
        tokens = min(self.capacity, tokens + (now - last) * self.rate)
        if tokens >= 1:
            self._buckets[key] = (tokens - 1, now)
            return True, int(tokens - 1), 0.0
        self._buckets[key] = (tokens, now)
        return False, 0, (1 - tokens) / self.rate

class ServiceSettings(BaseModel):          # a BaseSettings in the real project (env prefix CHAT_)
    api_keys: dict[str, str] = Field(default_factory=lambda: {"key-asha-123": "asha", "key-ravi-456": "ravi"})
    rate_capacity: int = 5                  # 5 requests ...
    rate_per_seconds: float = 60            # ... per minute per key
    cache_ttl_seconds: float = 300
    llm_timeout_seconds: float = 10
    max_tokens_limit: int = 200

def cache_key(model: str, prompt: str, max_tokens: int) -> str:
    normalized = " ".join(prompt.lower().split())                # same question, different spacing/case -> same key
    return hashlib.sha256(f"{model}|{max_tokens}|{normalized}".encode()).hexdigest()

now = [0.0]
limiter = TokenBucketLimiter(capacity=2, per_seconds=10, clock=lambda: now[0])
print([limiter.hit("k")[0] for _ in range(3)])        # [True, True, False]
now[0] = 5.0                                          # 5 s later: one token refilled (rate 0.2/s)
print(limiter.hit("k"))                               # (True, 0, 0.0)
print(cache_key("m", "What is  ASGI?", 50) == cache_key("m", "what is asgi?", 50))   # True

[True, True, False]
(True, 0, 0.0)
True


---

## 8.4 The service

* `require_api_key` uses `APIKeyHeader(auto_error=False)` so **we** decide the error (401 with a clear message), and compares keys with
  `secrets.compare_digest` (constant time, so timing does not leak how many characters matched).
* `rate_limited` depends on `require_api_key`, so it knows *who* is calling, and returns 429 with `Retry-After` and `X-RateLimit-Remaining`.
* `/v1/chat` uses `asyncio.timeout(...)` (Python 3.11+) around the LLM call and maps a timeout to **504**.
* `/v1/chat/stream` streams `token` events, then a `done` event with usage. Once streaming has started the status code is already sent
  (200), so a timeout in the middle is reported as an `error` **event**, not as an HTTP status.

In [68]:
# main.py: the app factory
import asyncio, secrets
from contextlib import asynccontextmanager
from typing import Annotated
from fastapi import Depends, FastAPI, HTTPException, Request, Response, status
from fastapi.security import APIKeyHeader
from fastapi.sse import EventSourceResponse, ServerSentEvent
from pydantic import BaseModel, Field

class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=4000)
    max_tokens: int = Field(default=50, ge=1, le=200)

class ChatResponse(BaseModel):
    answer: str
    model: str
    cached: bool
    usage: dict[str, int]

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

def create_chat_app(settings: ServiceSettings | None = None, llm: LLMClient | None = None, clock=time.monotonic) -> FastAPI:
    settings = settings or ServiceSettings()
    llm = llm or FakeLLM()

    @asynccontextmanager
    async def lifespan(app: FastAPI):
        app.state.ready = False
        await llm.warm_up()
        app.state.ready = True
        yield
        app.state.ready = False

    app = FastAPI(title="Chat Service", version="1.0.0", lifespan=lifespan)
    app.state.ready = False
    app.state.llm = llm
    app.state.cache = TTLCache(settings.cache_ttl_seconds, clock=clock)
    app.state.limiter = TokenBucketLimiter(settings.rate_capacity, settings.rate_per_seconds, clock=clock)
    app.state.usage = {}

    def require_api_key(key: Annotated[str | None, Depends(api_key_header)]) -> str:
        if key:
            for known, client_name in settings.api_keys.items():
                if secrets.compare_digest(key, known):
                    return client_name
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "Missing or invalid API key",
                            headers={"WWW-Authenticate": "APIKey"})

    def rate_limited(client_name: Annotated[str, Depends(require_api_key)], response: Response) -> str:
        allowed, remaining, retry_after = app.state.limiter.hit(client_name)
        if not allowed:
            raise HTTPException(status.HTTP_429_TOO_MANY_REQUESTS, "Rate limit exceeded",
                                headers={"Retry-After": str(math.ceil(retry_after)), "X-RateLimit-Remaining": "0"})
        response.headers["X-RateLimit-Remaining"] = str(remaining)
        return client_name

    Caller = Annotated[str, Depends(rate_limited)]

    def record_usage(client_name: str, tokens: int) -> None:
        app.state.usage[client_name] = app.state.usage.get(client_name, 0) + tokens

    @app.post("/v1/chat")
    async def chat(body: ChatRequest, client_name: Caller) -> ChatResponse:
        key = cache_key(llm.model, body.message, body.max_tokens)
        if (hit := app.state.cache.get(key)) is not None:
            return ChatResponse(answer=hit, model=llm.model, cached=True, usage={"tokens": 0})
        try:
            async with asyncio.timeout(settings.llm_timeout_seconds):
                tokens = [t async for t in llm.stream(body.message, body.max_tokens)]
        except TimeoutError:
            raise HTTPException(status.HTTP_504_GATEWAY_TIMEOUT, "The model took too long to answer")
        answer = "".join(tokens).strip()
        app.state.cache.set(key, answer)
        record_usage(client_name, len(tokens))
        return ChatResponse(answer=answer, model=llm.model, cached=False, usage={"tokens": len(tokens)})

    @app.post("/v1/chat/stream", response_class=EventSourceResponse)
    async def chat_stream(body: ChatRequest, client_name: Caller):
        key = cache_key(llm.model, body.message, body.max_tokens)
        if (hit := app.state.cache.get(key)) is not None:
            yield ServerSentEvent(data={"text": hit}, event="token")
            yield ServerSentEvent(data={"tokens": 0, "cached": True}, event="done")
            return
        tokens = []
        try:
            async with asyncio.timeout(settings.llm_timeout_seconds):
                async for token in llm.stream(body.message, body.max_tokens):
                    tokens.append(token)
                    yield ServerSentEvent(data={"text": token}, event="token")
        except TimeoutError:
            yield ServerSentEvent(data={"detail": "The model took too long to answer"}, event="error")
            return
        app.state.cache.set(key, "".join(tokens).strip())
        record_usage(client_name, len(tokens))
        yield ServerSentEvent(data={"tokens": len(tokens), "cached": False}, event="done")

    @app.get("/v1/usage")
    def usage(client_name: Annotated[str, Depends(require_api_key)]) -> dict[str, int]:
        return {"tokens": app.state.usage.get(client_name, 0)}

    @app.get("/healthz")
    def healthz():
        return {"status": "alive"}

    @app.get("/readyz")
    def readyz(response: Response):
        if not app.state.ready:
            response.status_code = 503
        return {"ready": app.state.ready, "model": llm.model}

    return app

print(sorted(create_chat_app().openapi()["paths"]))
# ['/healthz', '/readyz', '/v1/chat', '/v1/chat/stream', '/v1/usage']

['/healthz', '/readyz', '/v1/chat', '/v1/chat/stream', '/v1/usage']


> **Note:** `asyncio.timeout()` around an `async for` that **yields** (in the streaming endpoint) is fine here because the whole block
> stays inside one generator. Do not hold a timeout open across a `yield` in normal (non-generator) code.

---

## 8.5 Try it

In [69]:
# A quick tour of the service
from fastapi.testclient import TestClient

chat_app = create_chat_app()
asha = {"X-API-Key": "key-asha-123"}

with TestClient(chat_app) as client:
    print(client.get("/readyz").json())                                    # {'ready': True, 'model': 'fake-llm-1'}
    print(client.post("/v1/chat", json={"message": "hi"}).status_code)     # 401
    r = client.post("/v1/chat", json={"message": "How many workers?"}, headers=asha)
    print(r.json()["answer"], r.json()["cached"], r.headers["x-ratelimit-remaining"])
    # Use about one worker per CPU core, then measure with a load test. False 4
    r = client.post("/v1/chat", json={"message": "how many   WORKERS?"}, headers=asha)
    print(r.json()["cached"], r.json()["usage"])                           # True {'tokens': 0}
    print(client.get("/v1/usage", headers=asha).json())                    # {'tokens': 13}

{'ready': True, 'model': 'fake-llm-1'}
401
Use about one worker per CPU core, then measure with a load test. False 4
True {'tokens': 0}
{'tokens': 13}


---

## 8.6 Tests

Each test builds its own app with a **fake clock** and its own `FakeLLM`, so tests are fast, deterministic and independent.

In [70]:
# tests for the chat service
import json
from fastapi.testclient import TestClient

def parse_sse(lines):
    """Turn SSE lines into dicts like {"event": "token", "data": "..."} (same helper as Part 6.3)."""
    event = {}
    for line in lines:
        if line == "":
            if event:
                yield event
            event = {}
        elif not line.startswith(":"):
            field, _, value = line.partition(":")
            event[field] = value.removeprefix(" ")
    if event:
        yield event

def run_suite(tests):
    """A tiny pytest-like runner (same idea as Part 7.9)."""
    failed = 0
    for test in tests:
        try:
            test()
            print("PASSED", test.__name__)
        except AssertionError as e:
            failed += 1
            print("FAILED", test.__name__, repr(e))
    print(f"{len(tests) - failed} passed, {failed} failed")
    return failed

ASHA = {"X-API-Key": "key-asha-123"}
RAVI = {"X-API-Key": "key-ravi-456"}

def make(llm=None, **overrides):
    clock = [0.0]
    app = create_chat_app(ServiceSettings(**overrides), llm or FakeLLM(), clock=lambda: clock[0])
    return app, clock

def sse_events(response):
    return list(parse_sse(response.iter_lines()))

def test_rejects_missing_and_wrong_keys():
    app, _ = make()
    with TestClient(app) as c:
        assert c.post("/v1/chat", json={"message": "x"}).status_code == 401
        r = c.post("/v1/chat", json={"message": "x"}, headers={"X-API-Key": "nope"})
        assert r.status_code == 401 and r.headers["www-authenticate"] == "APIKey"

def test_validation():
    app, _ = make()
    with TestClient(app) as c:
        assert c.post("/v1/chat", json={"message": ""}, headers=ASHA).status_code == 422
        assert c.post("/v1/chat", json={"message": "x", "max_tokens": 999}, headers=ASHA).status_code == 422

def test_cache_avoids_second_llm_call():
    llm = FakeLLM()
    app, clock = make(llm, cache_ttl_seconds=60)
    with TestClient(app) as c:
        a = c.post("/v1/chat", json={"message": "What is CORS?"}, headers=ASHA).json()
        b = c.post("/v1/chat", json={"message": "what is cors?"}, headers=RAVI).json()
        assert a["answer"] == b["answer"] and b["cached"] is True and llm.calls == 1
        clock[0] = 61                                     # TTL expired
        assert c.post("/v1/chat", json={"message": "What is CORS?"}, headers=ASHA).json()["cached"] is False
        assert llm.calls == 2

def test_rate_limit_per_key_with_retry_after():
    app, clock = make(rate_capacity=2, rate_per_seconds=10)
    with TestClient(app) as c:
        codes = [c.post("/v1/chat", json={"message": f"q{i}"}, headers=ASHA).status_code for i in range(3)]
        assert codes == [200, 200, 429]
        r = c.post("/v1/chat", json={"message": "q"}, headers=ASHA)
        assert r.status_code == 429 and r.headers["retry-after"] == "5"
        assert c.post("/v1/chat", json={"message": "q"}, headers=RAVI).status_code == 200   # other key unaffected
        clock[0] = 5                                      # one token refilled
        assert c.post("/v1/chat", json={"message": "again"}, headers=ASHA).status_code == 200

def test_timeout_returns_504():
    app, _ = make(FakeLLM(token_delay=0.05), llm_timeout_seconds=0.1)
    with TestClient(app) as c:
        r = c.post("/v1/chat", json={"message": "Tell me about ASGI"}, headers=ASHA)
        assert r.status_code == 504

def test_stream_sends_tokens_then_done_and_fills_cache():
    llm = FakeLLM()
    app, _ = make(llm)
    with TestClient(app) as c:
        with c.stream("POST", "/v1/chat/stream", json={"message": "ASGI?"}, headers=ASHA) as r:
            assert r.status_code == 200 and r.headers["content-type"].startswith("text/event-stream")
            events = sse_events(r)
        tokens = [json.loads(e["data"])["text"] for e in events if e["event"] == "token"]
        assert "".join(tokens).startswith("ASGI is") and events[-1]["event"] == "done"
        assert c.post("/v1/chat", json={"message": "ASGI?"}, headers=ASHA).json()["cached"] is True

def test_stream_timeout_is_an_error_event():
    app, _ = make(FakeLLM(token_delay=0.05), llm_timeout_seconds=0.1)
    with TestClient(app) as c:
        with c.stream("POST", "/v1/chat/stream", json={"message": "ASGI?"}, headers=ASHA) as r:
            events = sse_events(r)
        assert r.status_code == 200 and events[-1]["event"] == "error"

def test_readiness_follows_lifespan():
    app, _ = make()
    c = TestClient(app)
    assert c.get("/readyz").status_code == 503            # lifespan not started
    with TestClient(app) as c2:
        assert c2.get("/readyz").status_code == 200
        assert c2.get("/healthz").status_code == 200

assert run_suite([test_rejects_missing_and_wrong_keys, test_validation, test_cache_avoids_second_llm_call,
                  test_rate_limit_per_key_with_retry_after, test_timeout_returns_504,
                  test_stream_sends_tokens_then_done_and_fills_cache, test_stream_timeout_is_an_error_event,
                  test_readiness_follows_lifespan]) == 0
# PASSED test_rejects_missing_and_wrong_keys
# ...
# 8 passed, 0 failed

PASSED test_rejects_missing_and_wrong_keys
PASSED test_validation
PASSED test_cache_avoids_second_llm_call


PASSED test_rate_limit_per_key_with_retry_after


PASSED test_timeout_returns_504


PASSED test_stream_sends_tokens_then_done_and_fills_cache


PASSED test_stream_timeout_is_an_error_event
PASSED test_readiness_follows_lifespan
8 passed, 0 failed


---

## 8.7 A console chat client

The client asks for questions with `input()` until you type `exit`, prints the streamed tokens as they arrive, and shows when an answer
came from the cache or when you hit the rate limit.

In [71]:
# Console chat client using SSE streaming
import json

def chat_console(client, api_key: str):
    headers = {"X-API-Key": api_key}
    print("Chat with the model. Type 'exit' to quit.")
    while True:
        message = input("You: ").strip()
        if message.lower() == "exit":
            print("Bye!")
            break
        if not message:
            continue
        with client.stream("POST", "/v1/chat/stream", json={"message": message}, headers=headers) as r:
            if r.status_code != 200:
                r.read()
                print(f"[{r.status_code}] {r.json()['detail']}", end="")
                if "retry-after" in r.headers:
                    print(f" (try again in {r.headers['retry-after']} s)", end="")
                print()
                continue
            print("Bot: ", end="")
            for event in parse_sse(r.iter_lines()):
                data = json.loads(event["data"])
                if event["event"] == "token":
                    print(data["text"], end="", flush=True)
                elif event["event"] == "done":
                    print(" (cached)" if data["cached"] else f"({data['tokens']} tokens)")
                elif event["event"] == "error":
                    print("\n[error]", data["detail"])

use_sample_input("What is ASGI?", "how many workers should I use?", "What is ASGI?", "And CORS?", "exit")
console_app = create_chat_app(ServiceSettings(rate_capacity=3))
with TestClient(console_app) as client:
    chat_console(client, "key-ravi-456")
# Chat with the model. Type 'exit' to quit.
# You: What is ASGI?
# Bot: ASGI is the async interface between Python web servers and apps like FastAPI. (13 tokens)
# You: how many workers should I use?
# Bot: Use about one worker per CPU core, then measure with a load test. (13 tokens)
# You: What is ASGI?
# Bot: ASGI is the async interface between Python web servers and apps like FastAPI. (cached)
# You: And CORS?
# [429] Rate limit exceeded (try again in 20 s)
# You: exit
# Bye!

Chat with the model. Type 'exit' to quit.
You: What is ASGI?
Bot: ASGI 

is 

the 

async 

interface 

between 

... (18 outputs left out)

load 

test. 

(13 tokens)
You: What is ASGI?
Bot: ASGI is the async interface between Python web servers and apps like FastAPI.

 (cached)
You: And CORS?
[429] Rate limit exceeded (try again in 20 s)
You: exit
Bye!


### Practice 8: extend the service

**A. Predict**

1. The service runs with 4 workers and `rate_capacity=5` per minute (in memory). How many requests per minute can one key really make?
2. Two users ask the same question with different `max_tokens`. Should the second one get a cache hit?

**B. Write code**

3. Add **request-level metrics**: a new app from `create_chat_app()` plus a middleware that counts responses by status code into a dict.
   Make 3 good calls and 1 call without a key, and check the counts `{200: 3, 401: 1}`.
4. A teammate wrote `bad_cache_key(prompt) = prompt` (no model, no max_tokens, no normalization). Write a test that shows the bug:
   with `max_tokens=3` first, then `max_tokens=50`, the second answer is wrongly short. Then show that `cache_key` gives two different keys.
5. Write a `RecordingLLM` that implements `LLMClient` by wrapping another client and recording each prompt in a list. Use it in
   `create_chat_app` and check that a cached call does not add a prompt.

**C. Explain:** why does the streaming endpoint report a timeout as an `error` event instead of a 504?

### Solutions 8

1. Up to 20: each worker has its own bucket (4 x 5). Move the limiter to Redis to enforce a shared limit.
2. No. A different `max_tokens` can produce a different answer, so it is part of the cache key (as are the model name and any other
   generation parameters such as temperature).
3. to 5. See the cell below.

**C.** In SSE the HTTP status line (200) is sent before the first token. When the timeout happens later, the status cannot be changed, so
the only way to tell the client is an event in the stream.

In [72]:
# Solutions 8
from collections import Counter
from fastapi import Request
from fastapi.testclient import TestClient

# Q3
metrics_chat = create_chat_app()
status_counts = Counter()

@metrics_chat.middleware("http")
async def count_status(request: Request, call_next):
    response = await call_next(request)
    status_counts[response.status_code] += 1
    return response

with TestClient(metrics_chat) as c:
    for q in ["ASGI?", "CORS?", "workers?"]:
        c.post("/v1/chat", json={"message": q}, headers=ASHA)
    c.post("/v1/chat", json={"message": "no key"})
assert dict(status_counts) == {200: 3, 401: 1}

# Q4
def bad_cache_key(prompt):
    return prompt

bad_cache, llm4 = {}, FakeLLM()
async def answer_with_bad_cache(prompt, max_tokens):
    key = bad_cache_key(prompt)
    if key not in bad_cache:
        bad_cache[key] = await llm4.complete(prompt, max_tokens)
    return bad_cache[key]

short = await answer_with_bad_cache("What is ASGI?", 3)
long = await answer_with_bad_cache("What is ASGI?", 50)
print(repr(long))                           # 'ASGI is the'  <- wrong: truncated answer served from cache
assert long == short and len(long.split()) == 3
assert cache_key("m", "What is ASGI?", 3) != cache_key("m", "What is ASGI?", 50)

# Q5
class RecordingLLM:
    def __init__(self, inner):
        self.inner, self.model, self.prompts = inner, inner.model, []
    async def warm_up(self):
        await self.inner.warm_up()
    def stream(self, prompt, max_tokens):
        self.prompts.append(prompt)
        return self.inner.stream(prompt, max_tokens)

rec = RecordingLLM(FakeLLM())
assert isinstance(rec, LLMClient)
with TestClient(create_chat_app(llm=rec)) as c:
    c.post("/v1/chat", json={"message": "CORS?"}, headers=ASHA)
    c.post("/v1/chat", json={"message": "cors?"}, headers=ASHA)      # cache hit
assert rec.prompts == ["CORS?"]
print("Practice 8: all checks passed")

'ASGI is the'
Practice 8: all checks passed


---

## How to debug this service

| Symptom | Likely cause | Fix |
|---|---|---|
| Rate limit not enforced with several workers | in-memory limiter | Redis-based limiter (atomic `INCR`/Lua script) |
| Wrong answers from the cache | cache key misses a parameter | include model, all generation params, normalized prompt |
| Stream arrives all at once | proxy buffering, sync generator | `EventSourceResponse` + proxy config |
| 429 storms after an outage | clients retry without backoff | `Retry-After` header, clients with backoff + jitter |
| 401 in production only | key header renamed by proxy, wrong env var | log the presence (not the value) of the header |
| Readiness never becomes true | `warm_up` fails (bad provider key) | log the error in the lifespan; fail fast |
| Costs explode | no cache, no per-key limits, `max_tokens` unbounded | cache, rate limit, cap `max_tokens` |

## Summary

* Put the LLM behind a `Protocol`; use a deterministic fake in development and tests.
* Auth with API keys (`APIKeyHeader`, `compare_digest`); per-key token-bucket rate limits with `429` + `Retry-After`.
* TTL cache keyed by model + parameters + normalized prompt; Redis when you have several workers.
* `asyncio.timeout` + `504` for JSON; an `error` event for streams. Readiness reflects warm-up.
* Tests use a fake clock and a fake LLM, so they are fast and deterministic.

# Part 9: Interview Questions and Answers

**What this Part covers**

* About 100 FastAPI interview questions, from beginner to advanced, grouped by topic:
  basics, Pydantic, dependency injection, async, databases, security, testing, deployment, performance and API design.
* Each answer is short, **first person** and written to be **spoken**: 2-6 lines you can say in about 30 seconds.
* Code snippets where they help, a few runnable checks, and a flashcard quiz that uses `input()`.

**How to use it:** cover the answer, say yours out loud, then compare. Interviewers care less about exact words and more about
whether you explain the *why* and mention trade-offs. If you do not know something, say how you would find out.

Questions marked **[Hard]** are common in senior interviews.

## 9.1 Basics

**Q1. What is FastAPI?**
FastAPI is a modern Python web framework for building APIs. I declare endpoints as functions with type hints, and FastAPI uses those hints
to parse and validate requests, serialize responses and generate OpenAPI documentation automatically. It is built on Starlette (the web
layer) and Pydantic (validation), and it supports both sync and async code.

**Q2. What is ASGI, and how is it different from WSGI?**
WSGI is the old synchronous interface between Python servers and apps: one request per call, no native WebSockets. ASGI is the
asynchronous successor: the app is an async callable `app(scope, receive, send)`, so one process can handle many concurrent connections,
long-lived connections like WebSockets and SSE, and lifespan events. Uvicorn is an ASGI server; FastAPI is an ASGI app.

**Q3. FastAPI vs Flask vs Django?**
Django is a full-stack "batteries included" framework: ORM, admin, auth, templates. Great for classic web apps. Flask is a minimal WSGI
framework where you pick extensions. FastAPI is focused on APIs: type-hint validation, automatic docs and first-class async. I would choose
Django for an admin-heavy monolith, and FastAPI for JSON APIs, microservices and ML/LLM services. Each has trade-offs; FastAPI has no built-in
ORM or admin.

**Q4. What do Starlette and Pydantic do inside FastAPI?**
Starlette provides the ASGI toolkit: routing, requests and responses, middleware, WebSockets, background tasks and the test client.
Pydantic provides data validation and serialization from type hints. FastAPI glues them together with dependency injection and OpenAPI generation.

**Q5. What is a path operation?**
It is the combination of a path and an HTTP method, like `GET /items/{item_id}`, plus the function that handles it. In code it is a
function decorated with `@app.get("/items/{item_id}")`.

**Q6. How does FastAPI decide if a parameter is a path, query or body parameter?**
If the name appears in the path template, it is a path parameter. If it is a simple type (int, str, bool...) it is a query parameter.
If it is a Pydantic model, it is read from the JSON body. I can be explicit with `Annotated[int, Query()]`, `Path()`, `Body()`, `Header()` and so on.

**Q7. What does a 422 error mean in FastAPI?**
422 Unprocessable Content means the request was well-formed HTTP but failed validation: a missing field, a wrong type, a constraint like
`gt=0`. FastAPI returns it automatically with a `detail` list that shows the location (`body`, `query`, `path`), the message and the type of each error.
Debugging tip: read the `loc`, it tells you exactly which field failed.

**Q8. What does `response_model` (or the return type) do?**
It declares the output schema. FastAPI validates and **filters** the returned data through it, so extra fields like `password_hash` are never
sent, and it documents the response in OpenAPI. In current FastAPI, the return annotation `-> UserOut` does the same, and Pydantic serializes it
straight to JSON bytes, which is also faster.

**Q9. How is the documentation generated?**
FastAPI builds an OpenAPI schema from the routes, parameters and models, served at `/openapi.json`. Swagger UI at `/docs` and ReDoc at `/redoc`
render it. Because it comes from the code, it does not drift out of date, and clients can be generated from it.

**Q10. `raise HTTPException` vs `return JSONResponse(...)`?**
`HTTPException` stops the request from anywhere, even inside a dependency, and goes through the exception handlers, so the error format stays
consistent. Returning a `JSONResponse` works only in the endpoint and bypasses response-model validation. I raise for errors and return for success.

**Q11. How do I return 201 or 204?**
Set `status_code=201` (or `status.HTTP_201_CREATED`) in the decorator. For 204, set `status_code=204` and return nothing (`-> None`).
To decide the status at runtime, I accept a `response: Response` parameter and set `response.status_code`.

**Q12. How do you read headers, cookies, forms and files?**
With `Annotated[str | None, Header()]`, `Cookie()`, `Form()` and `File()` / `UploadFile`. Header names are converted from `user_agent` to
`User-Agent`. Forms and files need the `python-multipart` package (included in `fastapi[standard]`). `UploadFile` streams large files to disk
instead of keeping them in memory.

In [73]:
# Q8 in action: the return type filters out fields that must never leave the server
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

class UserInDB(BaseModel):
    id: int
    email: str
    password_hash: str

class UserOut(BaseModel):
    id: int
    email: str

q_app = FastAPI()

@q_app.get("/users/{user_id}")
def get_user(user_id: int) -> UserOut:
    return UserInDB(id=user_id, email="asha@example.com", password_hash="$argon2id$...")

print(TestClient(q_app).get("/users/1").json())     # {'id': 1, 'email': 'asha@example.com'}
print(TestClient(q_app).get("/users/x").json()["detail"][0]["loc"])   # ['path', 'user_id']  (Q7: a 422)

{'id': 1, 'email': 'asha@example.com'}
['path', 'user_id']


## 9.2 Pydantic

**Q13. What is Pydantic, and why is v2 faster?**
Pydantic validates and converts data using Python type hints. Version 2 moved the core validation into `pydantic-core`, written in Rust, so it is
much faster than v1. The API also changed: `model_dump()` instead of `dict()`, `field_validator` instead of `validator`, `model_config` instead of `class Config`.

**Q14. `BaseModel` vs `dataclass` vs `TypedDict`?**
`BaseModel` validates on creation and has serialization helpers; it is what FastAPI uses for bodies. A standard `dataclass` only stores data and does
not validate (FastAPI can still use it, and Pydantic has its own validating dataclass). `TypedDict` is just a typed dict shape for type checkers.
For API input and output I use `BaseModel`.

**Q15. `field_validator` vs `model_validator`?**
`field_validator` checks or transforms one field, for example normalizing an email. `model_validator(mode="after")` sees the whole model, so I use it
for cross-field rules like "end_date must be after start_date".
```python
@model_validator(mode="after")
def check_dates(self):
    if self.end <= self.start:
        raise ValueError("end must be after start")
    return self
```

**Q16. How do you add constraints to fields?**
With `Field(...)` or `Annotated` constraints: `Field(min_length=1, max_length=100)`, `gt=0`, `pattern=...`, or types like `EmailStr`. Constraints show up in the
OpenAPI schema too, so clients see them.

**Q17. What is the difference between "optional" and "has a default"?**
In Pydantic v2 a field is **required** unless it has a default. `name: str | None` is required but may be `null`. `name: str | None = None` is optional.
This is a common v1-to-v2 surprise.

**Q18. How do you implement PATCH with Pydantic?**
I make all fields optional in an update model, then use `model_dump(exclude_unset=True)` so only the fields the client actually sent are applied.
Without `exclude_unset`, every missing field would overwrite real data with `None`.

**Q19. Why separate input and output models?**
Security and clarity. The input model (`UserCreate`) has a password but no id; the output model (`UserOut`) has an id but never the password hash.
It also lets the API evolve each side separately.

**Q20. What is `from_attributes`?**
`model_config = ConfigDict(from_attributes=True)` lets Pydantic read data from object attributes, for example SQLAlchemy models, instead of only dicts.
In v1 this was called `orm_mode = True`.

**Q21. Does Pydantic convert types? What is strict mode?**
By default it is in "lax" mode and coerces sensible values: `"5"` becomes `5` for an `int` field, `"true"` becomes `True`. In strict mode
(`Field(strict=True)` or `ConfigDict(strict=True)`) it rejects them. I use strict mode where silent conversion could hide bugs, like money amounts.

**Q22. How do you manage configuration?**
With `pydantic-settings`: a `BaseSettings` class reads environment variables (and `.env` files), validates types, and fails at startup if something
required is missing. I expose it through a cached dependency so tests can override it, and use `SecretStr` for secrets so they are not printed.

## 9.3 Dependency injection

**Q23. How does dependency injection work in FastAPI?**
I declare what an endpoint needs with `Depends(some_function)`. For each request, FastAPI builds a graph of dependencies, calls each one (resolving
their own parameters like query params, headers or other dependencies), and passes the results to the endpoint. It is used for DB sessions, the
current user, settings and permissions. It keeps endpoints small and makes testing easy.
```python
def get_current_user(token: Annotated[str, Depends(oauth2_scheme)]) -> User: ...

@app.get("/me")
def me(user: Annotated[User, Depends(get_current_user)]): ...
```

**Q24. What does a dependency with `yield` do?**
The code before `yield` runs before the endpoint, the yielded value is injected, and the code after `yield` runs after the response is ready
(cleanup). It is perfect for opening and closing a DB session. If I wrap the `yield` in `try/finally`, cleanup also runs on errors.

**Q25. Are dependencies called once per request?**
Yes, by default. If several dependencies need `get_db`, FastAPI calls it once per request and shares the result. I can disable this with
`Depends(get_db, use_cache=False)`. They are not cached across requests; for app-wide singletons I use the lifespan or `lru_cache`.

**Q26. Can a class be a dependency?**
Yes. Anything callable works. A class used with `Depends(MyClass)` is instantiated per request with its `__init__` parameters resolved. A class
instance with `__call__` is useful for configurable dependencies, like `RateLimiter(times=10, seconds=60)`.

**Q27. What are router-level and app-level dependencies?**
`APIRouter(dependencies=[Depends(verify_api_key)])` runs the dependency for every route of the router, even though the endpoints do not use the
value. `FastAPI(dependencies=[...])` does it for the whole app. Good for auth or audit checks.

**Q28. Middleware vs dependency: when do you use which?**
Middleware wraps **every** request, including 404s and static files, and works with the raw request and response: timing, request IDs, CORS, GZip,
security headers. A dependency runs only for the routes that declare it, knows the route's parameters, can return values to the endpoint, and shows
up in OpenAPI: auth, DB sessions, permissions, pagination. If it needs per-route data or a return value, I use a dependency.

**Q29. How do you replace a dependency in tests?**
`app.dependency_overrides[get_db] = get_test_db`. FastAPI calls the override instead everywhere the original is used. I clear the overrides after
each test, usually in a fixture, so tests do not leak into each other.

**Q30. What is `Security()` and what are scopes?** [Hard]
`Security` is like `Depends` but also carries OAuth2 **scopes** (permissions like `items:read`). The dependency receives `SecurityScopes` and can
check that the token contains the required scopes. The scopes also appear in OpenAPI.

**Q31. Can dependencies depend on other dependencies?**
Yes, they form a tree: `get_current_active_user` depends on `get_current_user`, which depends on `oauth2_scheme` and `get_db`. FastAPI resolves the
whole tree and caches shared nodes within the request.

**Q32. Can dependencies be `async def` and `def`?**
Both work. `async def` dependencies run on the event loop; `def` dependencies run in the thread pool, like `def` endpoints. The same rule applies:
no blocking calls inside `async def`.

In [74]:
# Q25 and Q29 in action: dependency caching within a request, and overrides in tests
from typing import Annotated
from fastapi import Depends, FastAPI
from fastapi.testclient import TestClient

calls = []
def get_db():
    calls.append("open")
    yield "real-db"
    calls.append("close")

def get_repo(db: Annotated[str, Depends(get_db)]):
    return f"repo({db})"

di_app = FastAPI()

@di_app.get("/")
def endpoint(db: Annotated[str, Depends(get_db)], repo: Annotated[str, Depends(get_repo)]):
    return {"db": db, "repo": repo}

client = TestClient(di_app)
print(client.get("/").json(), calls)   # {'db': 'real-db', 'repo': 'repo(real-db)'} ['open', 'close']  (opened once)

di_app.dependency_overrides[get_db] = lambda: "test-db"
print(client.get("/").json())          # {'db': 'test-db', 'repo': 'repo(test-db)'}
di_app.dependency_overrides.clear()

{'db': 'real-db', 'repo': 'repo(real-db)'} ['open', 'close']
{'db': 'test-db', 'repo': 'repo(test-db)'}


## 9.4 Async and concurrency

**Q33. When do you use `def` and when `async def`?** (the classic)
I use `async def` when everything slow inside is **awaitable**: an async DB driver, `httpx.AsyncClient`, `asyncio.sleep`. I use plain `def` when I
call blocking libraries, like `requests` or a sync DB driver, because FastAPI runs `def` endpoints in a thread pool so they do not block the event loop.
The worst case is a blocking call inside `async def`: it freezes every request in that worker.

**Q34. What happens if I call `time.sleep(2)` inside an `async def` endpoint?**
It blocks the event loop for 2 seconds. During that time the worker cannot serve any other request, not even health checks. With 10 concurrent
requests the last one waits about 20 seconds. The fix is `await asyncio.sleep(2)`, or a `def` endpoint, or `run_in_threadpool` for blocking code.

**Q35. What is the event loop?**
It is the scheduler of asyncio. It runs one coroutine at a time; when a coroutine hits `await` on I/O, the loop switches to another ready coroutine.
That is how one thread handles thousands of connections: by never waiting idly. It only works if code yields control with `await`.

**Q36. How do you call blocking code from async code?**
`await run_in_threadpool(func, *args)` from `fastapi.concurrency`, or `await asyncio.to_thread(func, *args)` / `anyio.to_thread.run_sync`. The function
runs in a worker thread and the event loop stays free.

**Q37. How do you handle CPU-bound work?**
Threads do not help much for pure-Python CPU work because of the GIL. I use a `ProcessPoolExecutor`, a task queue like Celery for heavy jobs, or a separate
service. If the heavy lifting happens in C libraries that release the GIL (NumPy, PyTorch), threads can be fine.

**Q38. Concurrency vs parallelism?**
Concurrency is dealing with many things at once by switching between them (asyncio on one core). Parallelism is doing many things at the same instant
on multiple cores (multiple processes). FastAPI gets concurrency from async and parallelism from multiple workers.

**Q39. How do you call three services at the same time?**
With `asyncio.gather` (or a `TaskGroup`) on a shared `httpx.AsyncClient`. The total time is about the slowest call, not the sum. I add timeouts and
decide what happens if one fails: `return_exceptions=True` or fail the whole request.

**Q40. Is the thread pool unlimited?**
No. AnyIO's default limiter has 40 threads per worker. If 40 slow `def` endpoints are running, the 41st waits. I can raise the limit at startup, but usually
the better fix is to make the slow I/O async or add workers.

**Q41. BackgroundTasks vs Celery?** (classic)
`BackgroundTasks` runs a function **after the response is sent, in the same process**. It is simple and great for small things like sending an email or
writing an audit log. But there are no retries, no persistence, and if the process restarts, the task is lost. Celery, RQ or Arq use a broker (Redis,
RabbitMQ) and separate workers: retries, scheduling, scaling and visibility. For anything long, important or heavy, I use a real queue.

**Q42. lifespan vs `on_event`?** (classic)
`lifespan` is an async context manager passed to `FastAPI(lifespan=...)`: code before `yield` runs at startup, code after at shutdown, and both can share
local variables and `app.state`. `@app.on_event("startup")` and `"shutdown"` are the old style, deprecated in favour of lifespan, and you cannot use both
together. I load models, create DB engines and HTTP clients in lifespan.

**Q43. WebSockets vs SSE vs polling?**
Polling is simple but wasteful. SSE is one-way server-to-client streaming over normal HTTP; it auto-reconnects in browsers and is ideal for LLM tokens
and notifications. WebSockets are two-way and best for chat, games and collaborative editing, but need more infrastructure care (sticky connections,
proxy upgrade headers). FastAPI 0.162 has built-in SSE with `EventSourceResponse`.

**Q44. Is async always faster?** [Hard]
No. Async improves **throughput under concurrent I/O**. A single request is not faster, CPU work is not faster, and a blocking call in async code makes
things much slower. If my whole stack is synchronous (sync ORM, sync SDKs), well-written `def` endpoints are perfectly fine.

## 9.5 Databases

**Q45. How do you manage database sessions in FastAPI?**
One session per request through a `yield` dependency: open the session, yield it, close it after the response. The engine (and its connection pool) is
created once per process. Commits happen in the endpoint or service layer; on errors I roll back.
```python
def get_db():
    with Session(engine) as session:
        yield session
```

**Q46. What changed in SQLAlchemy 2.0 style?**
Queries use `select()` with `session.execute()` / `session.scalars()` instead of `session.query()`; models use typed `Mapped[...]` and
`mapped_column()`; there is first-class async support. It is more explicit and type-checker friendly.

**Q47. Sync or async SQLAlchemy with FastAPI?**
Both work. Sync sessions in `def` endpoints are simple and reliable. Async (`AsyncSession` with `asyncpg` or `aiosqlite`) fits `async def` endpoints and
high-concurrency services, but relationships must be loaded explicitly (lazy loading raises). I choose based on the team and the load; mixing them carelessly
causes blocking bugs.

**Q48. What is the N+1 problem and how do you fix it?**
Loading a list with one query and then running one more query per item to fetch related data: 1 + N queries. I fix it with eager loading:
`selectinload` (one extra `IN` query) or `joinedload` (a JOIN). I detect it by counting queries in tests or logs.

**Q49. How do you size the connection pool?** [Hard]
Each worker process has its own pool, so total connections = workers x (pool_size + max_overflow). That must stay below the database's
`max_connections`, leaving room for migrations and admin tools. With many workers or serverless, I put PgBouncer in front. I also enable `pool_pre_ping`.

**Q50. Why use Alembic instead of `create_all()`?**
`create_all` only creates missing tables; it never alters columns or keeps history. Alembic migrations are versioned scripts that upgrade (and downgrade)
the schema step by step, reviewed in code and run as a deployment step.

**Q51. How do you handle transactions?**
The session is a unit of work: changes are sent on `commit()`. If anything fails, I `rollback()`. For multi-step operations I use
`with session.begin():` so everything commits together or nothing does. I keep transactions short and never hold one open while calling external APIs.

**Q52. What is SQLModel?**
A library by the FastAPI author that combines SQLAlchemy and Pydantic: one class can be both the table and the schema. It reduces duplication in simple apps.
For complex apps I still prefer separate SQLAlchemy models and Pydantic schemas, because input, output and table shapes usually differ.

**Q53. What does "database is locked" mean in SQLite?**
SQLite allows only one writer at a time. Long transactions, many workers writing, or a session left open cause other writers to time out with this error.
Fixes: keep transactions short, close sessions, enable WAL mode and a busy timeout, or move to Postgres for concurrent writes. (Part 11 reproduces it.)

**Q54. Offset vs cursor pagination?**
Offset (`LIMIT/OFFSET`) is simple and supports page numbers, but gets slower for deep pages and can skip or repeat rows when data changes. Cursor/keyset
pagination (`WHERE id > last_id`) uses the index, stays fast and stable, but cannot jump to an arbitrary page. I use cursors for feeds and large tables.

## 9.6 Security

**Q55. 401 vs 403?** (classic)
401 Unauthorized actually means **not authenticated**: no credentials or invalid ones, "who are you?". It should include a `WWW-Authenticate` header.
403 Forbidden means **authenticated but not allowed**: "I know who you are, but you cannot do this". For resources the user must not even know exist,
I return 404 instead of 403.

**Q56. How does OAuth2 password flow with JWT work in FastAPI?**
The client posts username and password as form data to a token endpoint. The server verifies the password hash and returns a signed JWT access token.
The client sends `Authorization: Bearer <token>` on later requests. A dependency built on `OAuth2PasswordBearer` extracts the token, verifies the signature
and expiry, and loads the user.

**Q57. What are common JWT pitfalls?** (classic)
Not passing `algorithms=["HS256"]` when decoding (algorithm confusion, including `none`), no expiry, weak or committed secrets, putting sensitive data in
the payload (it is only base64, anyone can read it), long-lived tokens with no way to revoke them, and storing tokens where XSS can read them. I keep access
tokens short (15-30 minutes), use refresh tokens, validate `exp`/`iss`/`aud`, and rotate keys.

**Q58. Where should a browser app keep the token?**
Each option has trade-offs. `localStorage` is easy but readable by any XSS script. An `HttpOnly`, `Secure`, `SameSite` cookie cannot be read by JavaScript,
but then I need CSRF protection for state-changing requests. Many teams use a short-lived access token in memory and a refresh token in an HttpOnly cookie.

**Q59. How do you store passwords?**
Never in plain text and never with fast hashes like SHA-256. I use a slow, salted password hash: Argon2id (or bcrypt). In current FastAPI docs this is done
with `pwdlib` (`PasswordHash.recommended()`); older tutorials use `passlib`, which is no longer maintained.

**Q60. What is CORS and how do you configure it?** (classic)
CORS is a browser rule: JavaScript on `https://app.com` may call `https://api.com` only if the API's responses allow that origin. Browsers send a preflight
`OPTIONS` request for non-simple requests. In FastAPI I add `CORSMiddleware` with an explicit list of origins, methods and headers. I never combine
`allow_origins=["*"]` with `allow_credentials=True`: browsers forbid `*` with credentials, so Starlette echoes back the caller's origin, which in effect lets
every website send logged-in requests. CORS is not server-side security: curl ignores it.

**Q61. API keys vs JWT vs session cookies?**
API keys identify a **client application** (server-to-server, B2B); simple, revocable, but long-lived. JWTs identify a **user** session without a DB lookup per
request; stateless but hard to revoke. Server-side sessions with cookies are easy to revoke and good for traditional web apps, but need shared session storage.

**Q62. How do you implement authorization (not just authentication)?**
Authentication says who you are; authorization says what you may do. I check ownership in every query (`WHERE owner_id = current_user.id`) and roles or
scopes in dependencies like `require_role("admin")`. Forgetting the ownership filter causes IDOR bugs, so I put it in one shared dependency and test it.

**Q63. How do you rate limit a FastAPI app?**
Per client key (API key, user ID or IP) with an algorithm like fixed window, sliding window or token bucket, returning 429 with `Retry-After`. In one process
an in-memory dict works; with several workers it must be shared, usually Redis. Libraries like `slowapi` help; the proxy or API gateway can also do it.

**Q64. How do you manage secrets?**
Never in code or images. They come from environment variables injected by the platform, or a secret manager (Vault, AWS Secrets Manager, Kubernetes Secrets).
In code I load them with `pydantic-settings` as `SecretStr`, so they do not appear in logs or reprs.

**Q65. How does FastAPI work behind HTTPS and a proxy?**
The proxy ends TLS and forwards plain HTTP with `X-Forwarded-For` and `X-Forwarded-Proto`. I run Uvicorn with `--proxy-headers` and
`--forwarded-allow-ips` set to the proxy's IPs, so `request.client` and URLs are correct. If the app is under a path prefix, I set `root_path`.

**Q66. How do you prevent SQL injection?**
By never building SQL with string formatting from user input. The ORM and parameterized queries (`text("... WHERE id = :id")` with bound params) send values
separately from the SQL. For dynamic parts like sort columns, I use an allow-list.

**Q67. How do you protect upload endpoints?**
Limit size (reject with 413, and limit at the proxy too), check the content type and file signature, never trust the filename (path traversal), store outside
the web root with a generated name, and scan or process files in the background.

**Q68. How do refresh tokens and revocation work?** [Hard]
The access token is short-lived (minutes). A long-lived refresh token, stored securely and saved in the DB, is exchanged for new access tokens. To log out or
revoke, I delete or mark the refresh token, and rotate it on each use so a stolen one is detected. For instant revocation of access tokens, I keep a deny-list
of token IDs (`jti`) in Redis until they expire.

## 9.7 Testing

**Q69. How do you test a FastAPI app?**
With `TestClient` (built on httpx): it calls the app in memory, no server needed. I use pytest fixtures to create the app with test settings, override
dependencies like the DB, and assert on status codes and JSON bodies. I test happy paths, validation errors, auth failures and permissions.

**Q70. How do you test with a different database?** (classic)
I override the session dependency: `app.dependency_overrides[get_db] = override_get_db`, where the override yields a session to a test database (SQLite
in memory with `StaticPool`, or a Postgres test container). Each test gets a clean state: a new DB, or a transaction that is rolled back at the end.

**Q71. TestClient vs `httpx.AsyncClient` with `ASGITransport`?**
`TestClient` is synchronous and simple; it runs the app in a separate thread with its own event loop. When the test itself must be async, for example to
await an async DB fixture, I use `httpx.AsyncClient(transport=ASGITransport(app=app))` with `pytest-asyncio` or `anyio`. Note that `ASGITransport` does not run
the lifespan by itself.

**Q72. How do you test code that needs the lifespan?**
Use `with TestClient(app) as client:`. The `with` block triggers startup and shutdown. Without it, `app.state` values set in the lifespan do not exist.

**Q73. How do you test code that calls external APIs?**
Never call the real API in unit tests. I inject the HTTP client as a dependency and use `httpx.MockTransport` (or the `respx` library) to return canned
responses, including errors and timeouts, so I can test retries and error mapping.

**Q74. Your tests pass alone but fail when run together. Why?**
Shared state: a module-level dict, a shared DB file, a cache or rate limiter that is not reset, or `dependency_overrides` not cleared. I make each test build
its own app (an app factory) or reset state in fixtures, and I can find the culprit with `pytest -p no:randomly` vs random order.

**Q75. How do you test authentication?**
A fixture registers a user and logs in to get a token, then builds `Authorization` headers. I test: no token (401), invalid and expired tokens (401),
another user's resource (404/403), and the happy path. For speed I can override `get_current_user` in tests that are not about auth.

**Q76. How do you test WebSockets and SSE?**
WebSockets: `with client.websocket_connect("/ws") as ws: ws.send_text("hi"); assert ws.receive_text() == ...`. SSE: `with client.stream("GET", url) as r:` and
parse `r.iter_lines()` into events.

**Q77. What do you test in a FastAPI app?**
Mostly API-level tests through `TestClient` (they cover routing, validation, dependencies and serialization together), plus unit tests for pure business logic,
and a few end-to-end tests against a real database. I run them in CI with coverage, and I add a test for every bug I fix.

**Q78. How do you test the OpenAPI schema?**
`app.openapi()` returns the schema as a dict, so I can assert that routes, operation IDs and security schemes exist, or compare it with a committed snapshot to
catch accidental breaking changes.

## 9.8 Deployment

**Q79. How do you deploy a FastAPI app to production?**
I package it as a Docker image (slim base, non-root, dependencies cached), run it with `fastapi run` or Uvicorn (one or more workers), behind a reverse proxy or
load balancer that handles HTTPS. Configuration comes from environment variables, migrations run as a separate step, and the platform (Kubernetes, ECS, Cloud Run)
uses liveness and readiness probes. Logs go to stdout, metrics and traces to the monitoring stack.

**Q80. Uvicorn vs Gunicorn vs `fastapi run`?**
Uvicorn is the ASGI server. `fastapi run` is a CLI wrapper around Uvicorn with production defaults. Gunicorn is a process manager that can run Uvicorn worker
classes (now from the `uvicorn-worker` package) and offers more process-management options. Modern `uvicorn --workers` also restarts dead workers, so
Gunicorn is optional.

**Q81. How many workers do you run?**
A starting point is one worker per CPU core for async I/O-bound apps, more for blocking sync code, fewer when each worker loads a big model. In Kubernetes I often
run one worker per container and scale containers. Then I confirm with a load test, watching CPU, memory and p95 latency.

**Q82. Why can one worker be a bottleneck?** (classic)
One worker is one process and, because of the GIL, uses about one CPU core for Python code. If anything blocks its event loop or does heavy CPU work, every
request in that worker waits. More workers or containers use more cores, and shared state must move to Redis or the DB.

**Q83. What are your Docker best practices for FastAPI?**
Slim base image, copy and install dependencies before the code for layer caching, multi-stage builds, a non-root user, exec-form `CMD` so SIGTERM reaches the
server, `PYTHONUNBUFFERED=1`, a `.dockerignore`, a health check, and no secrets in the image.

**Q84. Liveness vs readiness probes?**
Liveness answers "is the process alive?" and failing it restarts the container, so it must be trivial. Readiness answers "can it serve traffic now?" (DB
reachable, model loaded) and failing it only removes the instance from the load balancer. Putting the DB check in liveness causes restart storms.

**Q85. How does graceful shutdown work?**
The platform sends SIGTERM. Uvicorn stops accepting new connections, waits for running requests (up to `--timeout-graceful-shutdown`), then runs the lifespan
shutdown code, where I close pools and flush data. The platform's grace period must be longer than that timeout.

**Q86. How do you handle configuration across environments?**
Twelve-factor: the same image everywhere, different environment variables. A `Settings` class validates them at startup. Secrets come from the platform's secret
store. No `if env == "prod"` scattered in the code; behaviour is driven by settings.

**Q87. Your app is served under `/api` behind Nginx and `/docs` is broken. Why?**
The docs page loads `/openapi.json` from the root, but the app lives under `/api`. Setting `root_path="/api"` (or `--root-path /api`) tells FastAPI about the
prefix so the docs use `/api/openapi.json`.

**Q88. How do you deploy without downtime when the schema changes?** [Hard]
Rolling deploys run old and new versions at the same time, so migrations must be backward compatible: expand then contract. First add new columns (nullable),
deploy code that writes both, backfill, switch reads, and only later remove old columns. Readiness probes make sure new pods get traffic only when ready.

## 9.9 Performance and observability

**Q89. How do you scale a FastAPI app?** (classic)
Vertically first: async I/O done right, efficient queries, caching. Then horizontally: more workers per machine and more instances behind a load balancer,
keeping the app stateless (sessions, caches and rate limits in Redis or the DB). The database is often the real limit, so I add read replicas, connection
pooling and caching. Heavy work moves to background workers.

**Q90. "The API is slow." What do you do?**
Measure first: which endpoints, p50 vs p99, since when. Then look at traces and logs to split DB time, external calls and app time. Common causes: N+1 queries,
missing indexes, blocking calls in async code, no pagination, slow external APIs without timeouts, and an exhausted pool. Fix the biggest one, then measure again.

**Q91. What caching strategies do you know?**
HTTP caching with `Cache-Control` and `ETag` (304), CDN caching for public content, an in-process TTL cache for small hot data, and Redis cache-aside for shared
data, with TTLs and invalidation on writes. I watch for stale data and stampedes when hot keys expire.

**Q92. Should I use `ORJSONResponse` for speed?** [Hard]
Not in current FastAPI. In 0.162 `ORJSONResponse` and `UJSONResponse` are deprecated: when an endpoint declares a return type or `response_model`, FastAPI
serializes directly to JSON bytes with Pydantic, which is fast without a custom response class. So my advice is: declare return types.

**Q93. How do you handle large file uploads and downloads?**
Uploads: `UploadFile` (spooled to disk), read in chunks, enforce a size limit, store to object storage, process in the background. Downloads: `FileResponse` or
`StreamingResponse` so the file is never fully in memory; for big files, give the client a pre-signed S3 URL instead of proxying.

**Q94. What do you monitor in production?**
The RED metrics per route (rate, errors, duration percentiles), saturation (CPU, memory, DB pool, queue length), structured logs with request IDs, distributed
traces with OpenTelemetry, and error tracking with Sentry. I alert on user-facing symptoms like error rate and p95 latency.

**Q95. How do you load test?**
With Locust, k6 or similar against a production-like environment, ramping up users while watching throughput, error rate and p95/p99. The point where latency
jumps is my capacity. I repeat after changes to catch regressions.

**Q96. Does FastAPI support compression?**
Yes, `GZipMiddleware(minimum_size=1000)` compresses large responses for clients that accept gzip. Often the reverse proxy or CDN does it instead, to save app CPU.

## 9.10 API design and system design

**Q97. What makes a good REST API?**
Resources as plural nouns (`/orders/42`), HTTP methods as verbs, correct status codes (201, 204, 404, 409, 422, 429), consistent JSON field naming,
pagination on lists, one error format, versioning for breaking changes, and good docs. Predictability matters more than cleverness.

**Q98. PUT vs PATCH?**
PUT replaces the whole resource and is idempotent. PATCH changes only the sent fields. In FastAPI, PATCH uses an all-optional model and
`model_dump(exclude_unset=True)`.

**Q99. What is idempotency and how do you make a POST idempotent?**
An operation is idempotent if doing it twice has the same effect as once. GET, PUT and DELETE are; POST is not. For payments or orders I accept an
`Idempotency-Key` header, store the key with the result, and on a retry with the same key return the stored response instead of processing again.

**Q100. How do you version an API?**
Usually with a URL prefix (`/v1`, `/v2`) using one `APIRouter` per version, because it is explicit and easy to route. I only create a new version for breaking
changes, mark old endpoints `deprecated=True`, send `Deprecation`/`Sunset` headers and give clients a migration window.

**Q101. How do you return errors consistently?**
One format for every error, ideally RFC 9457 problem details (`type`, `title`, `status`, `detail`, `instance`, `application/problem+json`), produced by
exception handlers for `HTTPException`, validation errors and unexpected errors, with a request ID for support.

**Q102. How do you structure a large FastAPI project?**
By feature or layer: `routers/` (thin HTTP layer), `schemas/` (Pydantic), `models/` (ORM), `services/` (business logic), `repositories/` or `crud/` (DB access),
`core/` (config, security), plus an app factory and `tests/`. Routers depend on services through dependencies, so logic is testable without HTTP.

**Q103. Design the API for a URL shortener.**
`POST /v1/links` with `{"url": ...}` returns 201 with the short code; `GET /{code}` returns 307/308 redirect or 404; `GET /v1/links/{code}/stats` for clicks.
I would generate codes with base62 IDs, cache hot codes in Redis, count clicks asynchronously, rate limit creation, and validate URLs to block abuse.

**Q104. How do you design an endpoint for a long-running job?**
`POST /v1/reports` returns **202 Accepted** with a job ID and a `Location` to `GET /v1/reports/{id}`, which shows `pending/running/done/failed` and the result
link. A worker (Celery, Arq) does the work. Optionally a webhook or SSE notifies the client.

**Q105. How do you serve an ML model with FastAPI?**
Load the model once in the lifespan, expose it through a dependency, validate input and output with Pydantic, return the model version, and keep inference off
the event loop (`def` endpoint, thread pool, or process pool / model server for heavy CPU or GPU work). Add batching, timeouts, readiness that waits for the model,
and monitoring of latency and prediction distribution.

**Q106. Why is FastAPI popular for LLM applications?**
LLM apps are mostly I/O: calling model APIs, vector databases and tools. Async lets one worker handle many slow calls; SSE streaming sends tokens as they arrive
(built in since recent versions); Pydantic validates structured outputs and tool calls; and automatic docs help frontend teams. I still add timeouts, retries,
caching and rate limits because LLM calls are slow and expensive.

---

## 9.11 Flashcard drill (uses `input()`)

The cell below is a tiny quiz program. It asks a question, waits for your short answer with `input()`, checks it for key words, and shows the model answer.
With "Run All" it uses sample answers; run it yourself with `use_real_input()` first to practise for real.

In [75]:
# Flashcards: type a short answer; the checker looks for key ideas
import random

FLASHCARDS = [
    ("Status code for 'authenticated but not allowed'?", {"403"}, "403 Forbidden (401 = not authenticated)."),
    ("Where does a def endpoint run?", {"thread"}, "In the thread pool, so blocking calls do not block the event loop."),
    ("What replaces @app.on_event('startup')?", {"lifespan"}, "The lifespan async context manager: FastAPI(lifespan=...)."),
    ("PATCH: which model_dump option keeps only sent fields?", {"exclude_unset"}, "model_dump(exclude_unset=True)."),
    ("Status code for 'too many requests'?", {"429"}, "429, with a Retry-After header."),
]

def run_flashcards(cards, shuffle=False):
    if shuffle:
        cards = random.sample(cards, len(cards))
    score = 0
    for question, keywords, model_answer in cards:
        answer = input(f"Q: {question}\n   Your answer: ").lower()
        correct = any(k in answer for k in keywords)
        score += correct
        print("   Correct!" if correct else "   Not quite.", "Model answer:", model_answer)
    print(f"Score: {score}/{len(cards)}")
    return score

use_sample_input("403", "on the event loop", "lifespan", "exclude_unset=True", "429 with retry-after")
score = run_flashcards(FLASHCARDS)
# Q: Status code for 'authenticated but not allowed'?
#    Your answer: 403
#    Correct! Model answer: 403 Forbidden (401 = not authenticated).
# ...
# Score: 4/5

Q: Status code for 'authenticated but not allowed'?
   Your answer: 403
   Correct! Model answer: 403 Forbidden (401 = not authenticated).
Q: Where does a def endpoint run?
   Your answer: on the event loop
   Not quite. Model answer: In the thread pool, so blocking calls do not block the event loop.
Q: What replaces @app.on_event('startup')?
   Your answer: lifespan
   Correct! Model answer: The lifespan async context manager: FastAPI(lifespan=...).
Q: PATCH: which model_dump option keeps only sent fields?
   Your answer: exclude_unset=True
   Correct! Model answer: model_dump(exclude_unset=True).
Q: Status code for 'too many requests'?
   Your answer: 429 with retry-after
   Correct! Model answer: 429, with a Retry-After header.
Score: 4/5


### Practice 9

1. Pick 10 questions at random (for example with `random.sample(range(1, 107), 10)`) and answer them out loud in under 45 seconds each.
2. Add three of your own flashcards about topics you find hard, and run the quiz with `use_real_input()`.
3. For Q33, Q41, Q55 and Q82, write your answer in your own words in a markdown cell, then compare.
4. Write a check that every flashcard has at least one keyword and a model answer that ends with a full stop.

In [76]:
# Solutions 9 (the checkable part)
assert score == 4                                   # the sample answer "on the event loop" is wrong for Q2
assert all(keywords and model.endswith(".") for _, keywords, model in FLASHCARDS)

MY_CARDS = FLASHCARDS + [
    ("Which header tells the client when to retry after 429/503?", {"retry-after"}, "Retry-After."),
    ("What does N+1 mean?", {"query", "queries"}, "One query for the list plus one query per item."),
]
use_sample_input("403", "thread pool", "lifespan", "exclude_unset", "429", "Retry-After", "one query per item")
assert run_flashcards(MY_CARDS) == len(MY_CARDS)
print("Practice 9: all checks passed")

Q: Status code for 'authenticated but not allowed'?
   Your answer: 403
   Correct! Model answer: 403 Forbidden (401 = not authenticated).
Q: Where does a def endpoint run?
   Your answer: thread pool
   Correct! Model answer: In the thread pool, so blocking calls do not block the event loop.
Q: What replaces @app.on_event('startup')?
   Your answer: lifespan
   Correct! Model answer: The lifespan async context manager: FastAPI(lifespan=...).
Q: PATCH: which model_dump option keeps only sent fields?
   Your answer: exclude_unset
   Correct! Model answer: model_dump(exclude_unset=True).
Q: Status code for 'too many requests'?
   Your answer: 429
   Correct! Model answer: 429, with a Retry-After header.
Q: Which header tells the client when to retry after 429/503?
   Your answer: Retry-After
   Correct! Model answer: Retry-After.
Q: What does N+1 mean?
   Your answer: one query per item
   Correct! Model answer: One query for the list plus one query per item.
Score: 7/7
Practice 9: all c

## Summary

* Answer in the first person, give the *why*, and mention trade-offs: interviewers listen for judgement, not memorised text.
* The classics to know perfectly: `def` vs `async def`, how DI works, ASGI, FastAPI vs Flask/Django, 401 vs 403, 422, scaling and why one worker is a
  bottleneck, BackgroundTasks vs Celery, middleware vs dependency, `response_model`, CORS, JWT pitfalls, testing with overrides, lifespan vs `on_event`.
* Mention current facts confidently but carefully: lifespan over `on_event`, Pydantic v2 names, `pwdlib` over `passlib`, built-in SSE, return types instead of
  `ORJSONResponse` in FastAPI 0.162.

# Part 10: Coding Round Questions

**What this Part covers**

18 "write this" tasks that interviewers give in live-coding rounds and take-home tests. Each has:

* the task as an interviewer would say it, with the requirements;
* a full solution that runs, followed by `assert` checks with TestClient;
* a short note on what the interviewer is looking for.

**How to practise:** read the task, close the solution (or scroll it out of view), write your own version in a new cell in 10-15 minutes, then run
the solution's asserts against your code. Talk while you code: interviewers grade your reasoning as much as the result.

**Before you start:** Notebooks 1 and 2. Each task is self-contained.

### Coding task 10.1: a paginated list endpoint

"Write `GET /products` with `limit` (default 10, 1-100) and `offset` (default 0, >= 0) query parameters. Return
`{"items": [...], "total": n, "limit": l, "offset": o}`. Invalid values must return 422."

*Interviewer looks for:* `Annotated[int, Query(ge=..., le=...)]`, a response model, no off-by-one errors.

In [77]:
# Solution 10.1
from typing import Annotated
from fastapi import FastAPI, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel

class Product(BaseModel):
    id: int
    name: str

class ProductPage(BaseModel):
    items: list[Product]
    total: int
    limit: int
    offset: int

PRODUCTS = [Product(id=i, name=f"Product {i}") for i in range(1, 26)]
app_10_1 = FastAPI()

@app_10_1.get("/products")
def list_products(limit: Annotated[int, Query(ge=1, le=100)] = 10,
                  offset: Annotated[int, Query(ge=0)] = 0) -> ProductPage:
    return ProductPage(items=PRODUCTS[offset:offset + limit], total=len(PRODUCTS), limit=limit, offset=offset)

c = TestClient(app_10_1)
page = c.get("/products", params={"limit": 5, "offset": 20}).json()
assert [p["id"] for p in page["items"]] == [21, 22, 23, 24, 25] and page["total"] == 25
assert c.get("/products").json()["limit"] == 10
assert c.get("/products", params={"offset": 30}).json()["items"] == []
assert c.get("/products", params={"limit": 0}).status_code == 422
assert c.get("/products", params={"limit": 101}).status_code == 422
assert c.get("/products", params={"offset": -1}).status_code == 422
print("Task 10.1: all checks passed")

Task 10.1: all checks passed

### Coding task 10.2: a dependency that checks an API key

"Protect `GET /reports` with an `X-API-Key` header. Valid keys come from a set. Missing or wrong key -> 401. The endpoint should receive the client
name linked to the key. Make it reusable for a whole router."

*Interviewer looks for:* a dependency (not copy-pasted checks), `APIKeyHeader` or `Header`, constant-time comparison, router-level `dependencies=[...]`.

In [78]:
# Solution 10.2
import secrets
from typing import Annotated
from fastapi import APIRouter, Depends, FastAPI, HTTPException, status
from fastapi.security import APIKeyHeader
from fastapi.testclient import TestClient

API_KEYS = {"k-live-111": "acme", "k-live-222": "globex"}
key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

def require_client(api_key: Annotated[str | None, Depends(key_header)]) -> str:
    for known, client in API_KEYS.items():
        if api_key is not None and secrets.compare_digest(api_key, known):
            return client
    raise HTTPException(status.HTTP_401_UNAUTHORIZED, "Invalid or missing API key")

router = APIRouter(prefix="/internal", dependencies=[Depends(require_client)])   # whole router protected

@router.get("/ping")
def ping():
    return {"pong": True}

app_10_2 = FastAPI()
app_10_2.include_router(router)

@app_10_2.get("/reports")
def reports(client: Annotated[str, Depends(require_client)]):
    return {"client": client, "reports": ["q1", "q2"]}

c = TestClient(app_10_2)
assert c.get("/reports").status_code == 401
assert c.get("/reports", headers={"X-API-Key": "wrong"}).status_code == 401
assert c.get("/reports", headers={"X-API-Key": "k-live-222"}).json()["client"] == "globex"
assert c.get("/internal/ping").status_code == 401
assert c.get("/internal/ping", headers={"X-API-Key": "k-live-111"}).json() == {"pong": True}
print("Task 10.2: all checks passed")

Task 10.2: all checks passed


### Coding task 10.3: a timing middleware

"Add a middleware that measures how long each request takes and returns it in an `X-Process-Time` header (milliseconds). It must also work for 404s."

*Interviewer looks for:* `@app.middleware("http")`, `time.perf_counter()` (not `time.time()`), `await call_next(request)`, setting the header on the response.

In [79]:
# Solution 10.3
import asyncio, time
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

app_10_3 = FastAPI()

@app_10_3.middleware("http")
async def add_process_time(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    response.headers["X-Process-Time"] = f"{(time.perf_counter() - start) * 1000:.1f}"
    return response

@app_10_3.get("/slow")
async def slow():
    await asyncio.sleep(0.05)
    return {"ok": True}

c = TestClient(app_10_3)
r = c.get("/slow")
print(r.headers["x-process-time"])          # about 50.x
assert float(r.headers["x-process-time"]) >= 50
assert "x-process-time" in c.get("/missing").headers      # 404 also timed
print("Task 10.3: all checks passed")

65.4
Task 10.3: all checks passed


### Coding task 10.4: a custom exception handler

"Our service layer raises `InsufficientStock(product_id, requested, available)`. Turn it into a **409** response:
`{"error": "insufficient_stock", "message": ..., "product_id": ..., "available": ...}` without try/except in every endpoint."

*Interviewer looks for:* a domain exception class, `@app.exception_handler(...)`, keeping HTTP concerns out of business logic.

In [80]:
# Solution 10.4
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel

class InsufficientStock(Exception):
    def __init__(self, product_id: int, requested: int, available: int):
        self.product_id, self.requested, self.available = product_id, requested, available
        super().__init__(f"Only {available} left of product {product_id}, {requested} requested")

STOCK = {1: 3}

def reserve(product_id: int, qty: int) -> None:            # business logic: no HTTP here
    available = STOCK.get(product_id, 0)
    if qty > available:
        raise InsufficientStock(product_id, qty, available)
    STOCK[product_id] = available - qty

app_10_4 = FastAPI()

@app_10_4.exception_handler(InsufficientStock)
async def stock_handler(request: Request, exc: InsufficientStock):
    return JSONResponse(status_code=409, content={"error": "insufficient_stock", "message": str(exc),
                                                  "product_id": exc.product_id, "available": exc.available})

class OrderIn(BaseModel):
    product_id: int
    qty: int

@app_10_4.post("/orders", status_code=201)
def create_order(order: OrderIn):
    reserve(order.product_id, order.qty)
    return {"status": "reserved"}

c = TestClient(app_10_4)
assert c.post("/orders", json={"product_id": 1, "qty": 2}).status_code == 201
r = c.post("/orders", json={"product_id": 1, "qty": 2})
print(r.status_code, r.json())   # 409 {'error': 'insufficient_stock', 'message': 'Only 1 left of product 1, 2 requested', 'product_id': 1, 'available': 1}
assert r.status_code == 409 and r.json()["available"] == 1
print("Task 10.4: all checks passed")

409 {'error': 'insufficient_stock', 'message': 'Only 1 left of product 1, 2 requested', 'product_id': 1, 'available': 1}
Task 10.4: all checks passed


### Coding task 10.5: a file upload with size and type checks

"Write `POST /avatars` that accepts an image upload. Only PNG or JPEG (check the content type **and** the first bytes). Max 1 MB -> otherwise 413.
Wrong type -> 415. Return the size and a safe generated file name."

*Interviewer looks for:* `UploadFile`, reading at most limit + 1 bytes, magic-number check, never trusting the client's file name.

In [81]:
# Solution 10.5
import uuid
from typing import Annotated
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.testclient import TestClient

MAX_BYTES = 1_000_000
SIGNATURES = {"image/png": b"\x89PNG\r\n\x1a\n", "image/jpeg": b"\xff\xd8\xff"}
app_10_5 = FastAPI()

@app_10_5.post("/avatars", status_code=201)
async def upload_avatar(file: Annotated[UploadFile, File()]):
    signature = SIGNATURES.get(file.content_type)
    if signature is None:
        raise HTTPException(415, "Only PNG or JPEG images are allowed")
    data = await file.read(MAX_BYTES + 1)             # never read an unbounded body into memory
    if len(data) > MAX_BYTES:
        raise HTTPException(413, "File is larger than 1 MB")
    if not data.startswith(signature):
        raise HTTPException(415, "File content does not match its type")
    ext = ".png" if file.content_type == "image/png" else ".jpg"
    return {"stored_as": f"{uuid.uuid4().hex}{ext}", "size": len(data)}

c = TestClient(app_10_5)
png = b"\x89PNG\r\n\x1a\n" + b"0" * 100
r = c.post("/avatars", files={"file": ("../../etc/passwd.png", png, "image/png")})
assert r.status_code == 201 and r.json()["size"] == 108 and "/" not in r.json()["stored_as"]
assert c.post("/avatars", files={"file": ("a.gif", b"GIF89a", "image/gif")}).status_code == 415
assert c.post("/avatars", files={"file": ("fake.png", b"MZ-not-png", "image/png")}).status_code == 415
assert c.post("/avatars", files={"file": ("big.png", b"\x89PNG\r\n\x1a\n" + b"0" * MAX_BYTES, "image/png")}).status_code == 413
print("Task 10.5: all checks passed")

Task 10.5: all checks passed


### Coding task 10.6: a rate limiter dependency

"Limit each client IP to N requests per window of W seconds. Over the limit -> 429 with a `Retry-After` header. Make the limiter a configurable
dependency: `Depends(RateLimiter(times=3, seconds=10))`. It must be testable without sleeping."

*Interviewer looks for:* a callable class as a dependency, per-key state, an injectable clock, correct 429 + `Retry-After`, awareness that in-memory
state is per worker (Redis in production).

In [82]:
# Solution 10.6
import math, time
from fastapi import Depends, FastAPI, HTTPException, Request
from fastapi.testclient import TestClient

class RateLimiter:
    """Fixed-window limiter per client IP."""
    def __init__(self, times: int, seconds: float, clock=time.monotonic):
        self.times, self.seconds, self.clock = times, seconds, clock
        self.windows: dict[str, tuple[float, int]] = {}      # ip -> (window_start, count)

    def __call__(self, request: Request) -> None:
        key = request.client.host if request.client else "unknown"
        now = self.clock()
        start, count = self.windows.get(key, (now, 0))
        if now - start >= self.seconds:
            start, count = now, 0                            # a new window
        if count >= self.times:
            retry = math.ceil(start + self.seconds - now)
            raise HTTPException(429, "Too many requests", headers={"Retry-After": str(retry)})
        self.windows[key] = (start, count + 1)

fake_now = [100.0]
limiter_10_6 = RateLimiter(times=3, seconds=10, clock=lambda: fake_now[0])
app_10_6 = FastAPI()

@app_10_6.get("/search", dependencies=[Depends(limiter_10_6)])
def search(q: str):
    return {"q": q}

c = TestClient(app_10_6)
assert [c.get("/search", params={"q": "x"}).status_code for _ in range(4)] == [200, 200, 200, 429]
fake_now[0] = 104.0
r = c.get("/search", params={"q": "x"})
assert r.status_code == 429 and r.headers["retry-after"] == "6"
fake_now[0] = 110.0                                           # window over
assert c.get("/search", params={"q": "x"}).status_code == 200
print("Task 10.6: all checks passed")

Task 10.6: all checks passed


### Coding task 10.7: a WebSocket echo

"Write a WebSocket endpoint `/ws/echo` that accepts a connection, sends back every text message as `echo: <message>`, and closes the connection when the
client sends `bye`. Handle client disconnects without errors."

*Interviewer looks for:* `await websocket.accept()`, a receive loop, `WebSocketDisconnect`, a clean close.

In [83]:
# Solution 10.7
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.testclient import TestClient

app_10_7 = FastAPI()
disconnects = []

@app_10_7.websocket("/ws/echo")
async def echo(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            text = await websocket.receive_text()
            if text == "bye":
                await websocket.send_text("goodbye")
                await websocket.close(code=1000)
                return
            await websocket.send_text(f"echo: {text}")
    except WebSocketDisconnect:
        disconnects.append("client left")              # cleanup would go here

c = TestClient(app_10_7)
with c.websocket_connect("/ws/echo") as ws:
    ws.send_text("hello")
    assert ws.receive_text() == "echo: hello"
    ws.send_text("bye")
    assert ws.receive_text() == "goodbye"
with c.websocket_connect("/ws/echo") as ws:
    ws.send_text("one")
    assert ws.receive_text() == "echo: one"
# leaving the block disconnects the client
print("Task 10.7: all checks passed")

Task 10.7: all checks passed


### Coding task 10.8: an idempotent POST

"Make `POST /transfers` safe to retry. Clients send an `Idempotency-Key` header. A retry with the same key and the same body must not create a second
transfer and must return the original response. The same key with a different body -> 409."

*Interviewer looks for:* storing key + request fingerprint + response, replaying, rejecting mismatches, mentioning TTL and concurrent duplicates.
(Part 5.4 explains the pattern in detail.)

In [84]:
# Solution 10.8
import hashlib, uuid
from typing import Annotated
from fastapi import FastAPI, Header, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class TransferIn(BaseModel):
    to_account: str
    amount: int = Field(gt=0)

transfers: list[dict] = []
seen: dict[str, tuple[str, dict]] = {}
app_10_8 = FastAPI()

@app_10_8.post("/transfers", status_code=201)
def create_transfer(body: TransferIn, idempotency_key: Annotated[str, Header()]) -> dict:
    fingerprint = hashlib.sha256(body.model_dump_json().encode()).hexdigest()
    if idempotency_key in seen:
        old_fp, old_response = seen[idempotency_key]
        if old_fp != fingerprint:
            raise HTTPException(409, "Idempotency-Key reused with a different body")
        return old_response
    transfer = {"id": str(uuid.uuid4()), **body.model_dump()}
    transfers.append(transfer)
    seen[idempotency_key] = (fingerprint, transfer)
    return transfer

c = TestClient(app_10_8)
h = {"Idempotency-Key": "abc-123"}
first = c.post("/transfers", json={"to_account": "ACC-9", "amount": 500}, headers=h)
retry = c.post("/transfers", json={"to_account": "ACC-9", "amount": 500}, headers=h)
assert first.json() == retry.json() and len(transfers) == 1
assert c.post("/transfers", json={"to_account": "ACC-9", "amount": 900}, headers=h).status_code == 409
assert c.post("/transfers", json={"to_account": "ACC-9", "amount": 500}).status_code == 422   # header required
print("Task 10.8: all checks passed")

Task 10.8: all checks passed


### Coding task 10.9: a JWT create and verify function

"Write `create_token(user_id, secret, minutes)` and `verify_token(token, secret) -> int` with PyJWT. Verification must reject expired tokens, tokens signed
with another secret, tokens with `alg: none`, and tokens without `sub`. Raise one `InvalidToken` exception for all failures."

*Interviewer looks for:* `algorithms=["HS256"]` in `decode`, required claims, `exp`, catching `jwt.InvalidTokenError` (the base class).

In [85]:
# Solution 10.9
import base64, json
from datetime import UTC, datetime, timedelta
import jwt

class InvalidToken(Exception):
    pass

def create_token(user_id: int, secret: str, minutes: int = 15) -> str:
    now = datetime.now(UTC)
    return jwt.encode({"sub": str(user_id), "iat": now, "exp": now + timedelta(minutes=minutes)}, secret, algorithm="HS256")

def verify_token(token: str, secret: str) -> int:
    try:
        payload = jwt.decode(token, secret, algorithms=["HS256"], options={"require": ["exp", "sub"]})
        return int(payload["sub"])
    except (jwt.InvalidTokenError, ValueError) as exc:
        raise InvalidToken(str(exc)) from exc

SECRET = "a-long-random-secret-from-the-environment-123456"
assert verify_token(create_token(42, SECRET), SECRET) == 42

def rejected(token, secret=SECRET):
    try:
        verify_token(token, secret)
        return False
    except InvalidToken:
        return True

assert rejected(create_token(42, SECRET, minutes=-1))                          # expired
assert rejected(create_token(42, "another-secret-from-an-attacker-12345678"))  # wrong signature
assert rejected(jwt.encode({"exp": 9999999999}, SECRET, algorithm="HS256"))    # no sub

def b64(d):
    return base64.urlsafe_b64encode(json.dumps(d).encode()).rstrip(b"=").decode()
none_token = f'{b64({"alg": "none", "typ": "JWT"})}.{b64({"sub": "1", "exp": 9999999999})}.'
assert rejected(none_token)                                                    # alg=none attack
assert rejected("not-a-jwt")
print("Task 10.9: all checks passed")

Task 10.9: all checks passed


### Coding task 10.10: CRUD with PUT and PATCH

"Build an in-memory notes API: create (201), get (404 if missing), list, PUT (full replace), PATCH (partial), DELETE (204). Use separate input/output models."

*Interviewer looks for:* correct statuses, `exclude_unset=True` for PATCH, no shared mutable defaults, a clean store.

In [86]:
# Solution 10.10
from fastapi import FastAPI, HTTPException, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class NoteIn(BaseModel):
    title: str = Field(min_length=1)
    body: str = ""
    pinned: bool = False

class NotePatch(BaseModel):
    title: str | None = Field(default=None, min_length=1)
    body: str | None = None
    pinned: bool | None = None

class NoteOut(NoteIn):
    id: int

class NoteStore:
    def __init__(self):
        self.items: dict[int, NoteOut] = {}
        self.next_id = 1

    def get(self, note_id: int) -> NoteOut:
        if note_id not in self.items:
            raise HTTPException(status.HTTP_404_NOT_FOUND, "Note not found")
        return self.items[note_id]

store = NoteStore()
app_10_10 = FastAPI()

@app_10_10.post("/notes", status_code=201)
def create_note(note: NoteIn) -> NoteOut:
    new = NoteOut(id=store.next_id, **note.model_dump())
    store.items[new.id] = new
    store.next_id += 1
    return new

@app_10_10.get("/notes")
def list_notes() -> list[NoteOut]:
    return list(store.items.values())

@app_10_10.get("/notes/{note_id}")
def get_note(note_id: int) -> NoteOut:
    return store.get(note_id)

@app_10_10.put("/notes/{note_id}")
def replace_note(note_id: int, note: NoteIn) -> NoteOut:
    store.get(note_id)
    store.items[note_id] = NoteOut(id=note_id, **note.model_dump())
    return store.items[note_id]

@app_10_10.patch("/notes/{note_id}")
def patch_note(note_id: int, patch: NotePatch) -> NoteOut:
    updated = store.get(note_id).model_copy(update=patch.model_dump(exclude_unset=True))
    store.items[note_id] = updated
    return updated

@app_10_10.delete("/notes/{note_id}", status_code=204)
def delete_note(note_id: int) -> None:
    store.get(note_id)
    del store.items[note_id]

c = TestClient(app_10_10)
assert c.post("/notes", json={"title": "a", "body": "text"}).status_code == 201
assert c.patch("/notes/1", json={"pinned": True}).json() == {"title": "a", "body": "text", "pinned": True, "id": 1}
assert c.put("/notes/1", json={"title": "b"}).json() == {"title": "b", "body": "", "pinned": False, "id": 1}
assert len(c.get("/notes").json()) == 1
assert c.delete("/notes/1").status_code == 204 and c.get("/notes/1").status_code == 404
assert c.put("/notes/9", json={"title": "x"}).status_code == 404
print("Task 10.10: all checks passed")

Task 10.10: all checks passed


### Coding task 10.11: list query parameters and enums

"`GET /jobs` filters by multiple `tag` query parameters (`?tag=python&tag=remote`), a `level` that must be one of `junior`, `mid`, `senior`, and an optional
`min_salary` >= 0. Return jobs that have **all** given tags."

*Interviewer looks for:* `list[str]` with `Query()`, an `Enum` or `Literal`, optional filters.

In [87]:
# Solution 10.11
from enum import Enum
from typing import Annotated
from fastapi import FastAPI, Query
from fastapi.testclient import TestClient

class Level(str, Enum):
    junior = "junior"
    mid = "mid"
    senior = "senior"

JOBS = [
    {"id": 1, "title": "Backend Dev", "tags": {"python", "remote"}, "level": "mid", "salary": 90},
    {"id": 2, "title": "ML Engineer", "tags": {"python", "ml"}, "level": "senior", "salary": 140},
    {"id": 3, "title": "Intern", "tags": {"python", "remote"}, "level": "junior", "salary": 20},
]
app_10_11 = FastAPI()

@app_10_11.get("/jobs")
def search_jobs(tag: Annotated[list[str], Query()] = [],
                level: Level | None = None,
                min_salary: Annotated[int | None, Query(ge=0)] = None):
    result = [j for j in JOBS
              if set(tag) <= j["tags"]
              and (level is None or j["level"] == level.value)
              and (min_salary is None or j["salary"] >= min_salary)]
    return [j["id"] for j in result]

c = TestClient(app_10_11)
assert c.get("/jobs", params=[("tag", "python"), ("tag", "remote")]).json() == [1, 3]
assert c.get("/jobs", params={"tag": "python", "level": "senior"}).json() == [2]
assert c.get("/jobs", params={"min_salary": 50}).json() == [1, 2]
assert c.get("/jobs", params={"level": "boss"}).status_code == 422
print("Task 10.11: all checks passed")

Task 10.11: all checks passed


> **Note:** a mutable default like `= []` is safe **for FastAPI parameters**, because FastAPI creates a new value for each request. In normal Python
> functions it is still a classic bug.

### Coding task 10.12: a background task for audit logging

"After `POST /users` succeeds, write an audit entry `"created user <name>"` **after** the response is sent. The client must not wait for it."

*Interviewer looks for:* `BackgroundTasks.add_task`, knowing it runs after the response in the same process, and when to use Celery instead.

In [88]:
# Solution 10.12
from fastapi import BackgroundTasks, FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

audit_log: list[str] = []

def write_audit(message: str) -> None:
    audit_log.append(message)                 # imagine a slow write to another system

class NewUser(BaseModel):
    name: str

app_10_12 = FastAPI()

@app_10_12.post("/users", status_code=201)
def create_user(user: NewUser, background: BackgroundTasks):
    background.add_task(write_audit, f"created user {user.name}")
    return {"name": user.name}

r = TestClient(app_10_12).post("/users", json={"name": "Asha"})
assert r.status_code == 201
assert audit_log == ["created user Asha"]     # TestClient waits for background tasks before returning
print("Task 10.12: all checks passed")

Task 10.12: all checks passed


### Coding task 10.13: a dependency with cleanup

"Write a `get_connection` dependency that opens a (fake) connection, gives it to the endpoint, and **always** closes it, also when the endpoint raises. Prove
the order of events."

*Interviewer looks for:* `yield` dependency with `try/finally`, understanding when the code after `yield` runs.

In [89]:
# Solution 10.13
from typing import Annotated
from fastapi import Depends, FastAPI, HTTPException
from fastapi.testclient import TestClient

events: list[str] = []

class FakeConnection:
    def query(self, x):
        events.append(f"query {x}")
        return x * 2

def get_connection():
    events.append("open")
    conn = FakeConnection()
    try:
        yield conn
    finally:
        events.append("close")

app_10_13 = FastAPI()

@app_10_13.get("/double/{x}")
def double(x: int, conn: Annotated[FakeConnection, Depends(get_connection)]):
    if x < 0:
        raise HTTPException(400, "x must be positive")
    return {"result": conn.query(x)}

c = TestClient(app_10_13)
assert c.get("/double/4").json() == {"result": 8}
assert events == ["open", "query 4", "close"]
events.clear()
assert c.get("/double/-1").status_code == 400
assert events == ["open", "close"]             # closed even on error
print("Task 10.13: all checks passed")

Task 10.13: all checks passed


### Coding task 10.14: role-based access

"Users have roles. Write a dependency factory `require_role("admin")` so that `DELETE /users/{id}` is only for admins (403 for others, 401 without a user).
Use a simple `X-User` header as the 'authentication' for this exercise."

*Interviewer looks for:* a function that returns a dependency (closure), 401 vs 403, reuse across routes.

In [90]:
# Solution 10.14
from typing import Annotated
from fastapi import Depends, FastAPI, Header, HTTPException
from fastapi.testclient import TestClient

USERS = {"asha": {"name": "asha", "roles": {"admin"}}, "ravi": {"name": "ravi", "roles": {"viewer"}}}

def current_user(x_user: Annotated[str | None, Header()] = None) -> dict:
    if x_user not in USERS:
        raise HTTPException(401, "Not authenticated", headers={"WWW-Authenticate": "Bearer"})
    return USERS[x_user]

def require_role(role: str):
    def checker(user: Annotated[dict, Depends(current_user)]) -> dict:
        if role not in user["roles"]:
            raise HTTPException(403, f"Requires role: {role}")
        return user
    return checker

app_10_14 = FastAPI()

@app_10_14.delete("/users/{user_id}", status_code=204)
def delete_user(user_id: int, admin: Annotated[dict, Depends(require_role("admin"))]) -> None:
    return None

@app_10_14.get("/reports")
def reports(user: Annotated[dict, Depends(require_role("viewer"))]):
    return {"for": user["name"]}

c = TestClient(app_10_14)
assert c.delete("/users/1").status_code == 401
assert c.delete("/users/1", headers={"X-User": "ravi"}).status_code == 403
assert c.delete("/users/1", headers={"X-User": "asha"}).status_code == 204
assert c.get("/reports", headers={"X-User": "ravi"}).json() == {"for": "ravi"}
print("Task 10.14: all checks passed")

Task 10.14: all checks passed


### Coding task 10.15: stream a large CSV export

"`GET /export.csv` must stream 100,000 rows as CSV without building the whole file in memory, with a download file name."

*Interviewer looks for:* a generator + `StreamingResponse`, the `Content-Disposition` header, the right media type.

In [91]:
# Solution 10.15
import csv, io
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient

def csv_rows(n: int):
    buffer = io.StringIO()
    writer = csv.writer(buffer)
    writer.writerow(["id", "name", "score"])
    for i in range(1, n + 1):
        writer.writerow([i, f"user{i}", i % 100])
        if i % 1000 == 0:                        # send chunks of ~1000 rows
            yield buffer.getvalue()
            buffer.seek(0); buffer.truncate(0)
    yield buffer.getvalue()

app_10_15 = FastAPI()

@app_10_15.get("/export.csv")
def export_csv(rows: int = 100_000):
    return StreamingResponse(csv_rows(rows), media_type="text/csv",
                             headers={"Content-Disposition": 'attachment; filename="export.csv"'})

c = TestClient(app_10_15)
with c.stream("GET", "/export.csv") as r:
    assert r.headers["content-type"].startswith("text/csv")
    assert "export.csv" in r.headers["content-disposition"]
    lines = sum(chunk.count("\n") for chunk in r.iter_text())
assert lines == 100_001                          # header + rows
print("Task 10.15: all checks passed")

Task 10.15: all checks passed


### Coding task 10.16: fan out to two services with a timeout

"`GET /dashboard/{user_id}` must call the profile service and the orders service **concurrently** and combine the results. If the orders service is slower
than 0.2 s, return the profile with `"orders": null` and `"partial": true` instead of failing."

*Interviewer looks for:* one shared `httpx.AsyncClient`, `asyncio.gather` or a `TaskGroup`, per-call timeouts, graceful degradation.

In [92]:
# Solution 10.16
import asyncio, time
import httpx
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from contextlib import asynccontextmanager

ORDERS_DELAY = {"value": 0.0}

async def downstream(request: httpx.Request) -> httpx.Response:     # fake profile + orders services
    if request.url.host == "profile":
        return httpx.Response(200, json={"name": "Asha"})
    await asyncio.sleep(ORDERS_DELAY["value"])
    return httpx.Response(200, json=[{"id": 1}, {"id": 2}])

@asynccontextmanager
async def lifespan(app: FastAPI):
    async with httpx.AsyncClient(transport=httpx.MockTransport(downstream)) as http:
        app.state.http = http
        yield

app_10_16 = FastAPI(lifespan=lifespan)

@app_10_16.get("/dashboard/{user_id}")
async def dashboard(user_id: int, request: Request):
    http: httpx.AsyncClient = request.app.state.http

    async def orders():
        try:
            async with asyncio.timeout(0.2):
                return (await http.get(f"http://orders/users/{user_id}/orders")).json()
        except TimeoutError:
            return None

    profile, user_orders = await asyncio.gather(http.get(f"http://profile/users/{user_id}"), orders())
    return {"profile": profile.json(), "orders": user_orders, "partial": user_orders is None}

with TestClient(app_10_16) as c:
    assert c.get("/dashboard/1").json() == {"profile": {"name": "Asha"}, "orders": [{"id": 1}, {"id": 2}], "partial": False}
    ORDERS_DELAY["value"] = 1.0
    t = time.perf_counter()
    r = c.get("/dashboard/1").json()
    assert r["partial"] is True and r["orders"] is None and time.perf_counter() - t < 0.8
print("Task 10.16: all checks passed")

Task 10.16: all checks passed


### Coding task 10.17: a console client that POSTs items (uses `input()`)

"Write a small client script: ask the user for an item name and a price with `input()`, POST it to `/items`, and print the created item. If the server
answers 422, print the validation messages and ask again. Stop when the name is empty."

*Interviewer looks for:* separation of client and server (the server never calls `input()`), handling 422 bodies, a simple loop.

In [93]:
# Solution 10.17
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class ItemIn(BaseModel):
    name: str = Field(min_length=1, max_length=50)
    price: float = Field(gt=0)

items_db: list[dict] = []
app_10_17 = FastAPI()

@app_10_17.post("/items", status_code=201)
def add_item(item: ItemIn):
    record = {"id": len(items_db) + 1, **item.model_dump()}
    items_db.append(record)
    return record

def item_client(client) -> int:
    created = 0
    while True:
        name = input("Item name (empty to stop): ").strip()
        if not name:
            return created
        price = input("Price: ").strip()
        r = client.post("/items", json={"name": name, "price": price})
        if r.status_code == 201:
            created += 1
            print("Created:", r.json())
        else:
            for err in r.json()["detail"]:
                print(f"  {err['loc'][-1]}: {err['msg']}")

use_sample_input("Notebook", "120", "Pen", "-5", "Pen", "15.5", "")
n = item_client(TestClient(app_10_17))
# Item name (empty to stop): Notebook
# Price: 120
# Created: {'id': 1, 'name': 'Notebook', 'price': 120.0}
# Item name (empty to stop): Pen
# Price: -5
#   price: Input should be greater than 0
# ...
assert n == 2 and [i["name"] for i in items_db] == ["Notebook", "Pen"]
print("Task 10.17: all checks passed")

Item name (empty to stop): Notebook
Price: 120
Created: {'id': 1, 'name': 'Notebook', 'price': 120.0}
Item name (empty to stop): Pen
Price: -5
  price: Input should be greater than 0
Item name (empty to stop): Pen
Price: 15.5
Created: {'id': 2, 'name': 'Pen', 'price': 15.5}
Item name (empty to stop): 
Task 10.17: all checks passed


### Coding task 10.18: verify a webhook signature

"A payment provider calls `POST /webhooks/payments` and signs the **raw body** with HMAC-SHA256 using a shared secret, sending
`X-Signature: sha256=<hex>`. Reject invalid or missing signatures with 401 and replayed events (same `event_id`) with 200 but `"duplicate": true`."

*Interviewer looks for:* reading the raw body (`await request.body()`) before parsing, `hmac.compare_digest`, idempotent processing of webhooks.

In [94]:
# Solution 10.18
import hashlib, hmac, json
from typing import Annotated
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.testclient import TestClient

WEBHOOK_SECRET = b"whsec_test_123"
processed_events: set[str] = set()
app_10_18 = FastAPI()

def sign(body: bytes) -> str:
    return "sha256=" + hmac.new(WEBHOOK_SECRET, body, hashlib.sha256).hexdigest()

@app_10_18.post("/webhooks/payments")
async def payment_webhook(request: Request, x_signature: Annotated[str | None, Header()] = None):
    body = await request.body()                     # the exact bytes that were signed
    if x_signature is None or not hmac.compare_digest(x_signature, sign(body)):
        raise HTTPException(401, "Invalid signature")
    event = json.loads(body)
    if event["event_id"] in processed_events:
        return {"received": True, "duplicate": True}
    processed_events.add(event["event_id"])
    return {"received": True, "duplicate": False}

c = TestClient(app_10_18)
body = json.dumps({"event_id": "evt_1", "type": "payment.succeeded"}).encode()
ok = c.post("/webhooks/payments", content=body, headers={"X-Signature": sign(body), "Content-Type": "application/json"})
assert ok.json() == {"received": True, "duplicate": False}
again = c.post("/webhooks/payments", content=body, headers={"X-Signature": sign(body)})
assert again.json()["duplicate"] is True
assert c.post("/webhooks/payments", content=body, headers={"X-Signature": "sha256=bad"}).status_code == 401
assert c.post("/webhooks/payments", content=body).status_code == 401
tampered = body.replace(b"succeeded", b"refunded")
assert c.post("/webhooks/payments", content=tampered, headers={"X-Signature": sign(body)}).status_code == 401
print("Task 10.18: all checks passed")

Task 10.18: all checks passed


## How to do well in a live-coding round

* **Clarify first:** inputs, outputs, status codes, edge cases. Write them as comments.
* **Start simple, then harden:** happy path, then validation, errors, auth, limits.
* **Test as you go** with TestClient; even 3 asserts show discipline.
* **Name trade-offs out loud:** "in memory for now; with several workers this goes to Redis".
* **Use current APIs:** `Annotated`, return types, lifespan, `model_dump`.

## Summary

You practised the patterns that cover most live-coding questions: pagination, auth dependencies, middleware, exception handlers, uploads, rate limiting,
WebSockets, idempotency, JWTs, CRUD with PUT/PATCH, list query parameters, background tasks, cleanup with `yield`, role checks, streaming, concurrent calls with
timeouts, a client with `input()`, and webhook signatures.

# Part 11: Debugging Scenarios and a Mock Interview

**What this Part covers**

* 10 realistic "it is broken in production" scenarios. Each has the **symptom** (what you see), a **reproduction** in code, the **cause** and the **fix**.
* A method for debugging you can say out loud in interviews.
* A complete **45-minute mock interview** plan with timings, questions and a scoring sheet.

Debugging questions are popular because they show experience: "Our API hangs under load. What do you check?"

**A debugging method to say out loud**

1. **Reproduce:** which endpoint, which input, how often, since when (a deploy? more traffic?).
2. **Read the evidence:** status code, error body, logs with the request ID, metrics, traces.
3. **Form one hypothesis** and test it with the smallest experiment.
4. **Fix the cause**, not the symptom, and **add a test** so it never comes back.

## 11.1 Everything hangs under load

**Symptom:** with a few users the API is fine. Under load, *every* endpoint becomes slow, even `/healthz`, and the container is restarted because
liveness probes time out. CPU is low.

**Hypothesis:** something blocks the event loop. Reproduce: one slow report request, and a health check sent at the same moment.

In [95]:
# Reproduce: a blocking call in async def delays an unrelated health check
import asyncio, time
import httpx
from fastapi import FastAPI

def generate_report_sync():
    time.sleep(0.5)                       # e.g. requests.get(...), a sync DB driver, pandas work
    return "report"

hang_app = FastAPI()

@hang_app.get("/report")
async def report():
    return {"report": generate_report_sync()}       # BUG: blocking call inside async def

@hang_app.get("/healthz")
async def healthz():
    return {"ok": True}

async def health_latency_during_report(app):
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://t") as client:
        start = time.perf_counter()
        async def timed_health():
            await asyncio.sleep(0.05)                   # the health check is due 0.05 s after the report started
            await client.get("/healthz")
            return time.perf_counter() - (start + 0.05) # how late it finished compared to when it was due
        _, health = await asyncio.gather(client.get("/report"), timed_health())
        return health

late = await health_latency_during_report(hang_app)
print(f"health check took {late:.2f}s")          # about 0.5s (it had to wait for the report)
assert late > 0.3

health check took 0.51s


In [96]:
# Fix: make the endpoint def (thread pool) or offload the blocking call
from fastapi.concurrency import run_in_threadpool

fixed_hang_app = FastAPI()

@fixed_hang_app.get("/report")
async def report_fixed():
    return {"report": await run_in_threadpool(generate_report_sync)}

@fixed_hang_app.get("/healthz")
async def healthz_fixed():
    return {"ok": True}

latency = await health_latency_during_report(fixed_hang_app)
print(f"health check took {latency:.3f}s")      # about 0.001s
assert latency < 0.2

health check took 0.014s


**Cause:** a blocking call in `async def` froze the event loop. **Fix:** `def` endpoint, `run_in_threadpool`, or an async library.
**How to find it in production:** `py-spy dump --pid <pid>` shows the stack where the loop is stuck; asyncio debug mode (`PYTHONASYNCIODEBUG=1`) logs slow callbacks.

---

## 11.2 "CORS blocked" in the browser

**Symptom:** the React app on `http://localhost:3000` shows *"has been blocked by CORS policy"*. The same request works in curl and in `/docs`.

**Cause candidates:** no `CORSMiddleware`; the origin is not in the list; a typo in the origin (scheme, port, **trailing slash**).
We reproduce the browser's **preflight** request (`OPTIONS` with `Origin` and `Access-Control-Request-Method`).

In [97]:
# Reproduce: preflight fails without middleware, and with a trailing-slash typo
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.testclient import TestClient

PREFLIGHT = {"Origin": "http://localhost:3000", "Access-Control-Request-Method": "POST",
             "Access-Control-Request-Headers": "content-type"}

def make_cors_app(origins=None):
    app = FastAPI()
    if origins is not None:
        app.add_middleware(CORSMiddleware, allow_origins=origins, allow_methods=["GET", "POST"],
                           allow_headers=["Content-Type", "Authorization"], allow_credentials=True)
    @app.post("/items")
    def create_item():
        return {"ok": True}
    return app

r = TestClient(make_cors_app()).options("/items", headers=PREFLIGHT)
print(r.status_code)                                   # 405  (no middleware: OPTIONS is not handled)
r = TestClient(make_cors_app(["http://localhost:3000/"])).options("/items", headers=PREFLIGHT)
print(r.status_code, r.text)                           # 400 Disallowed CORS origin   (the trailing slash!)
r = TestClient(make_cors_app(["http://localhost:3000"])).options("/items", headers=PREFLIGHT)
print(r.status_code, r.headers["access-control-allow-origin"])   # 200 http://localhost:3000
assert r.status_code == 200

405
400 Disallowed CORS origin
200 http://localhost:3000


**Fix:** add `CORSMiddleware` with the **exact** origins (scheme + host + port, no path, no trailing slash), the methods and headers the frontend uses.
Remember that error responses also need CORS headers; if the app crashes before the middleware, the browser reports CORS even though the real problem is a 500.
Check the server logs, not only the browser console.

---

## 11.3 422 on a request that "looks valid"

**Symptom:** the frontend sends `POST /users` with JSON `{"name": "Asha"}` and gets 422. "But I sent the name!"

**Cause:** read the `loc` in the error. A simple-type parameter like `name: str` is a **query** parameter, not a body field.

In [98]:
# Reproduce: a str parameter is read from the query string, not from the JSON body
from typing import Annotated
from fastapi import Body, FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

bug_422 = FastAPI()

@bug_422.post("/users")
def create_user(name: str):                           # BUG: expects ?name=..., not JSON
    return {"name": name}

r = TestClient(bug_422).post("/users", json={"name": "Asha"})
print(r.status_code, r.json()["detail"][0]["loc"], r.json()["detail"][0]["msg"])   # 422 ['query', 'name'] Field required

# Fix A: a Pydantic model (the normal way)
class UserIn(BaseModel):
    name: str

fix_422 = FastAPI()

@fix_422.post("/users")
def create_user_fixed(user: UserIn):
    return {"name": user.name}

# Fix B: a single body field with Body(embed=True)
@fix_422.post("/users-embed")
def create_user_embed(name: Annotated[str, Body(embed=True)]):
    return {"name": name}

c = TestClient(fix_422)
assert c.post("/users", json={"name": "Asha"}).json() == {"name": "Asha"}
assert c.post("/users-embed", json={"name": "Asha"}).json() == {"name": "Asha"}
print("fixed")                                        # fixed

422 ['query', 'name'] Field required
fixed


Other common 422 causes: sending form data to a JSON endpoint (or the reverse, for `OAuth2PasswordRequestForm` logins), `Content-Type` missing,
a field name in `camelCase` while the model uses `snake_case` (use `alias` / `alias_generator`), and a number sent as `"12abc"`.

---

## 11.4 SQLite: "database is locked"

**Symptom:** under concurrent writes, some requests fail with `sqlite3.OperationalError: database is locked`.

**Cause:** SQLite allows **one writer at a time**. A transaction that stays open (a long request, a forgotten commit, a session kept between requests) blocks
every other writer until their busy timeout runs out.

In [99]:
# Reproduce with two connections to the same SQLite file
import sqlite3
from pathlib import Path

lab = Path("fastapi_lab_3"); lab.mkdir(exist_ok=True)
db_path = lab / "locked_demo.db"
db_path.unlink(missing_ok=True)

setup = sqlite3.connect(db_path)
setup.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, item TEXT)")
setup.commit(); setup.close()

slow_request = sqlite3.connect(db_path, timeout=0.1)
slow_request.execute("INSERT INTO orders (item) VALUES ('pen')")      # opens a write transaction ... and does not commit yet

other_request = sqlite3.connect(db_path, timeout=0.1)                 # waits only 0.1 s for the lock
try:
    other_request.execute("INSERT INTO orders (item) VALUES ('book')")
except sqlite3.OperationalError as e:
    print("OperationalError:", e)                                     # OperationalError: database is locked

slow_request.commit()                                                 # FIX part 1: keep write transactions short
other_request.execute("INSERT INTO orders (item) VALUES ('book')")
other_request.commit()
print(other_request.execute("SELECT count(*) FROM orders").fetchone()[0])   # 2
slow_request.close(); other_request.close()
db_path.unlink()

OperationalError: database is locked
2


**Fixes:** commit quickly and close sessions (a `yield` dependency per request); never hold a transaction while calling external APIs; enable
**WAL mode** (`PRAGMA journal_mode=WAL`, readers no longer block the writer) and a longer busy timeout (`connect_args={"timeout": 15}`);
for real concurrent write traffic, move to **Postgres**.

---

## 11.5 JWT is always invalid

**Symptom:** login works and returns a token, but every protected request answers 401 "Could not validate credentials".

**Debug step:** call `jwt.decode` yourself and print the **exception type**. Two causes that are very common today:

In [100]:
# Cause 1: "sub" is an int. PyJWT 2.10+ validates that sub is a string.
import secrets
import jwt

SECRET = "a-long-random-secret-from-the-environment-123456"
token = jwt.encode({"sub": 42, "exp": 9999999999}, SECRET, algorithm="HS256")     # encoding works...
try:
    jwt.decode(token, SECRET, algorithms=["HS256"])
except jwt.InvalidTokenError as e:
    print(type(e).__name__, "-", e)            # InvalidSubjectError - Subject must be a string

# Cause 2: the secret is generated at import time, so each worker (and each restart) has a different one
def make_worker_secret():
    return secrets.token_hex(32)               # BUG in real code: SECRET_KEY = secrets.token_hex(32) at module level

worker_a, worker_b = make_worker_secret(), make_worker_secret()
token_from_a = jwt.encode({"sub": "42", "exp": 9999999999}, worker_a, algorithm="HS256")
try:
    jwt.decode(token_from_a, worker_b, algorithms=["HS256"])
except jwt.InvalidSignatureError as e:
    print(type(e).__name__, "-", e)            # InvalidSignatureError - Signature verification failed

# Fix: sub as a string, and one secret from the environment shared by all workers
good = jwt.encode({"sub": "42", "exp": 9999999999}, SECRET, algorithm="HS256")
assert jwt.decode(good, SECRET, algorithms=["HS256"])["sub"] == "42"

InvalidSubjectError - Subject must be a string
InvalidSignatureError - Signature verification failed


Other causes: the client sends `Authorization: <token>` without `Bearer `; the clock of one server is wrong (use `leeway=` for small skews);
`algorithms` not passed to `decode`; the token was copied with quotes around it.

---

## 11.6 Tests pass alone but fail together

**Symptom:** `pytest tests/test_b.py` passes; `pytest` (all tests) fails in `test_b`. Or the result depends on the test order.

**Cause:** global state shared between tests: a module-level dict, a shared DB file, a cache, or `dependency_overrides` left behind.

In [101]:
# Reproduce: two tests share a module-level "database"
from fastapi import FastAPI
from fastapi.testclient import TestClient

USERS_DB: dict[str, dict] = {}                     # BUG: global state
shared_app = FastAPI()

@shared_app.post("/users", status_code=201)
def create(name: str):
    if name in USERS_DB:
        return {"error": "exists"}
    USERS_DB[name] = {"name": name}
    return USERS_DB[name]

@shared_app.get("/users/count")
def count():
    return {"count": len(USERS_DB)}

def test_a_creates_user():
    TestClient(shared_app).post("/users", params={"name": "asha"})
    assert TestClient(shared_app).get("/users/count").json() == {"count": 1}

def test_b_starts_empty():
    assert TestClient(shared_app).get("/users/count").json() == {"count": 0}

for test in (test_b_starts_empty, test_a_creates_user, test_b_starts_empty):
    try:
        test(); print("PASSED", test.__name__)
    except AssertionError:
        print("FAILED", test.__name__)
# PASSED test_b_starts_empty
# PASSED test_a_creates_user
# FAILED test_b_starts_empty        <- same test, different result: order dependence

PASSED test_b_starts_empty
PASSED test_a_creates_user
FAILED test_b_starts_empty


In [102]:
# Fix: an app factory that owns its state (or a fixture that resets it)
def create_users_app() -> FastAPI:
    app = FastAPI()
    app.state.users = {}

    @app.post("/users", status_code=201)
    def create(name: str):
        app.state.users[name] = {"name": name}
        return app.state.users[name]

    @app.get("/users/count")
    def count():
        return {"count": len(app.state.users)}
    return app

def test_a_fixed():
    c = TestClient(create_users_app())
    c.post("/users", params={"name": "asha"})
    assert c.get("/users/count").json() == {"count": 1}

def test_b_fixed():
    assert TestClient(create_users_app()).get("/users/count").json() == {"count": 0}

for test in (test_b_fixed, test_a_fixed, test_b_fixed):
    test()
print("order no longer matters")       # order no longer matters

order no longer matters


Also: always clear `app.dependency_overrides` (a fixture with `yield` and `clear()` after), use a fresh in-memory DB per test, and install the
`pytest-randomly` plugin (it shuffles the test order on every run) to find hidden order dependence.

---

## 11.7 `AttributeError: 'State' object has no attribute 'model'` in tests

**Symptom:** the app works with `fastapi dev`, but tests crash with `AttributeError` on `app.state.something`.

**Cause:** the value is created in the **lifespan**, and the test used `TestClient(app)` without `with`, so the lifespan never ran.

In [103]:
# Reproduce and fix: lifespan only runs inside "with TestClient(...)"
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.model = {"version": "1.0"}
    yield

life_app = FastAPI(lifespan=lifespan)

@life_app.get("/version")
def version(request: Request):
    return request.app.state.model

r = TestClient(life_app, raise_server_exceptions=False).get("/version")
print(r.status_code)                     # 500  (AttributeError inside: lifespan did not run)
try:
    TestClient(life_app).get("/version")
except AttributeError as e:
    print("AttributeError:", e)          # AttributeError: 'State' object has no attribute 'model'

with TestClient(life_app) as client:     # fix
    print(client.get("/version").json()) # {'version': '1.0'}

500
AttributeError: 'State' object has no attribute 'model'
{'version': '1.0'}


---

## 11.8 `/users/me` returns 422

**Symptom:** `GET /users/me` returns 422 "Input should be a valid integer". `GET /users/5` works.

**Cause:** routes are matched **in order**. `/users/{user_id}` was declared first, so `me` is treated as a `user_id` and fails `int` validation.

In [104]:
# Reproduce and fix: static paths must be declared before parameterized ones
from fastapi import FastAPI
from fastapi.testclient import TestClient

order_bug = FastAPI()

@order_bug.get("/users/{user_id}")
def get_user(user_id: int):
    return {"id": user_id}

@order_bug.get("/users/me")
def get_me():
    return {"id": "me"}

print(TestClient(order_bug).get("/users/me").status_code)     # 422

order_fixed = FastAPI()

@order_fixed.get("/users/me")             # static path first
def get_me_fixed():
    return {"id": "me"}

@order_fixed.get("/users/{user_id}")
def get_user_fixed(user_id: int):
    return {"id": user_id}

assert TestClient(order_fixed).get("/users/me").json() == {"id": "me"}
assert TestClient(order_fixed).get("/users/5").json() == {"id": 5}
print("fixed")                            # fixed

422
fixed


---

## 11.9 A 500 error, but the endpoint code "has no bug"

**Symptom:** an endpoint returns 500 Internal Server Error. The logs show `ResponseValidationError`.

**Cause:** the data returned does not match the declared response model (a missing or `None` field). FastAPI validates **outgoing** data too; that is a bug in
the server, so it is a 500, not a 422.

In [105]:
# Reproduce: the return value does not match the declared output model
from fastapi import FastAPI
from fastapi.exceptions import ResponseValidationError
from fastapi.testclient import TestClient
from pydantic import BaseModel

class ProfileOut(BaseModel):
    id: int
    email: str

resp_bug = FastAPI()

@resp_bug.get("/profile/{pid}")
def profile(pid: int) -> ProfileOut:
    row = {"id": pid, "email": None}            # e.g. a nullable DB column
    return row

print(TestClient(resp_bug, raise_server_exceptions=False).get("/profile/1").status_code)   # 500
try:
    TestClient(resp_bug).get("/profile/1")
except ResponseValidationError as e:
    print(type(e).__name__, e.errors()[0]["loc"], e.errors()[0]["msg"])
    # ResponseValidationError ('response', 'email') Input should be a valid string

500
ResponseValidationError ('response', 'email') Input should be a valid string


**Fix:** make the model match reality (`email: str | None`), or fix the data. Add a test for the endpoint with realistic data (including `NULL`s).

---

## 11.10 POST becomes a 307 redirect (and then CORS or a lost body)

**Symptom:** the frontend calls `POST /items`; the browser shows a `307 Temporary Redirect` to `/items/`, sometimes followed by a CORS error, or a mobile HTTP
library turns the redirected POST into a GET.

**Cause:** the route was declared as `/items/` (trailing slash). Starlette redirects `/items` to `/items/` (`redirect_slashes=True`).

In [106]:
# Reproduce: a trailing slash in the route causes a redirect
from fastapi import FastAPI
from fastapi.testclient import TestClient

slash_app = FastAPI()

@slash_app.post("/items/")
def create_item():
    return {"ok": True}

r = TestClient(slash_app).post("/items", json={}, follow_redirects=False)
print(r.status_code, r.headers["location"])        # 307 http://testserver/items/

fixed_slash = FastAPI()

@fixed_slash.post("/items")                        # fix: no trailing slash, and call the exact URL
def create_item_fixed():
    return {"ok": True}

assert TestClient(fixed_slash).post("/items", json={}, follow_redirects=False).status_code == 200
print("fixed")                                     # fixed

307 http://testserver/items/


fixed


Behind a proxy, the redirect may even point to `http://` instead of `https://` if proxy headers are not trusted (Part 1.2), which browsers block as mixed content.
Pick one style (no trailing slashes is common) and use it everywhere.

### Practice 11: debugging

**A. Diagnose** (say the cause and the first thing you would check)

1. After a deploy, all requests take 30 s and then fail with 500. Logs: `QueuePool limit of size 5 overflow 10 reached, connection timed out`.
2. Only the first request after a deploy is slow (8 s); the next ones take 50 ms.
3. The frontend gets `401` from `/v1/tasks` only in production; locally it works. The proxy is new.

**B. Write code**

4. This app returns 422 for `POST /login` with JSON `{"username": "a", "password": "b"}`. Find out why by printing the error `loc`, then make a test that sends the
   data correctly (as form fields) and gets 200.
   ```python
   @app.post("/login")
   def login(form: Annotated[OAuth2PasswordRequestForm, Depends()]):
       return {"user": form.username}
   ```
5. A teammate's endpoint leaks DB sessions: `get_db` returns a session but never closes it. Rewrite `get_db` as a `yield` dependency with `try/finally` and prove
   with a counter that every opened session is closed, even when the endpoint raises.

### Solutions 11

1. Sessions are not being returned to the pool (no `close()`, a dependency without `yield`/`finally`, or a transaction held during a slow external call), or the pool
   is too small for the traffic. Check the `get_db` dependency and long-running requests first.
2. Something is loaded lazily on the first request (a model, a big config, a connection pool warm-up). Load it in the lifespan and keep readiness at 503 until it is ready.
3. The proxy may drop or rename the `Authorization` header, or redirect (losing headers), or the app reads a different secret in production. Check what reaches the app
   (log whether the header is present, never its value) and compare the configuration.
4. and 5. See the cell below. `OAuth2PasswordRequestForm` reads **form fields**, so a JSON body gives 422 with `loc` `['body', 'username']`.

In [107]:
# Solutions 11
from typing import Annotated
from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import OAuth2PasswordRequestForm
from fastapi.testclient import TestClient

login_app = FastAPI()

@login_app.post("/login")
def login(form: Annotated[OAuth2PasswordRequestForm, Depends()]):
    return {"user": form.username}

c = TestClient(login_app)
bad = c.post("/login", json={"username": "a", "password": "b"})
print(bad.status_code, [e["loc"] for e in bad.json()["detail"]])   # 422 [['body', 'username'], ['body', 'password']]
good = c.post("/login", data={"username": "a", "password": "b"})   # form fields
assert good.status_code == 200 and good.json() == {"user": "a"}

counter = {"open": 0, "closed": 0}

class FakeSession:
    def __init__(self):
        counter["open"] += 1
    def close(self):
        counter["closed"] += 1

def get_db():
    session = FakeSession()
    try:
        yield session
    finally:
        session.close()

leak_app = FastAPI()

@leak_app.get("/ok")
def ok(db: Annotated[FakeSession, Depends(get_db)]):
    return {"ok": True}

@leak_app.get("/boom")
def boom(db: Annotated[FakeSession, Depends(get_db)]):
    raise HTTPException(500, "boom")

lc = TestClient(leak_app, raise_server_exceptions=False)
for path in ["/ok", "/boom", "/ok", "/boom"]:
    lc.get(path)
assert counter == {"open": 4, "closed": 4}
print("Practice 11: all checks passed")

422 [['body', 'username'], ['body', 'password']]
Practice 11: all checks passed


---

## 11.11 A 45-minute mock interview

Run this with a friend (or alone, out loud, with a timer). The interviewer reads the prompts; the candidate answers without notes.

| Time | Section | What happens | Material |
|---|---|---|---|
| 0-5 | Introduction | "Tell me about a FastAPI project you built." Candidate gives a 2-minute story: problem, architecture, one hard bug, result | Your Task Manager (Part 7), AI service (Part 8) or Healthcare RAG project |
| 5-15 | Fundamentals rapid-fire | 8-10 short questions, about 1 minute each | Part 9: Q2, Q7, Q8, Q23, Q28, Q33, Q41, Q42, Q55, Q60 |
| 15-30 | Live coding | One task from Part 10, with a follow-up that adds a requirement | 10.1 then "add filtering"; or 10.6 then "make it work with 4 workers" (Redis) |
| 30-38 | Debugging | One scenario from this Part, told as a story; the candidate asks questions and narrows it down | 11.1, 11.4, 11.5 or 11.6 |
| 38-43 | Design | "Design the API for X": endpoints, status codes, auth, pagination, scaling | Part 5 and Q103-Q106 (URL shortener, long jobs, ML serving) |
| 43-45 | Candidate questions | The candidate asks 1-2 good questions | "How do you deploy and monitor your FastAPI services?", "What does on-call look like?" |

**Follow-up questions interviewers love** (prepare them):

* "What would break first if traffic went up 10 times?"
* "How do you know your API is healthy right now?"
* "How would you test that?"
* "What would you change if this had to run with 8 workers?"
* "What is a mistake you made with async, and how did you find it?"

**Scoring sheet** (1 = weak, 3 = solid, 5 = excellent):

| Area | What a 5 looks like |
|---|---|
| Fundamentals | Correct, concise, explains *why* (event loop, DI graph, status code meaning) |
| Coding | Working code, `Annotated` and return types, tests with asserts, handles edge cases |
| Debugging | Asks for evidence first, forms hypotheses, proposes a test for the fix |
| Design | Correct methods and codes, pagination, idempotency, versioning, security |
| Production sense | Workers, timeouts, health checks, observability, trade-offs without "X is the best" |
| Communication | Thinks out loud, structured answers, admits unknowns and explains how to find out |

> **Interview tip:** after the mock, write down the 3 questions you answered worst and practise only those the next day. Repetition on weak spots beats
> re-reading everything.

## Summary

* Common production bugs: blocking the event loop, CORS origin typos, 422 from wrong parameter sources, SQLite write locks, JWT `sub` type and secret mismatches,
  shared state in tests, lifespan not running in tests, route order, response validation 500s, and trailing-slash redirects.
* Always debug with evidence: status code, error `loc`, logs with request IDs, and a minimal reproduction; then add a test.
* Practise the 45-minute mock interview at least twice before a real interview.

# Part 12: Cheat Sheet and Final Checklist

**What this Part covers**

* The imports, decorators, parameter types and status codes you use every day.
* Copy-ready patterns: lifespan, DB session, current user, pagination, errors, SSE, tests.
* Facts that changed in recent versions (checked in FastAPI 0.162).
* A production checklist to go through before you ship, and before you describe a project in an interview.
* A cell that cleans up the lab folder of this notebook.

## 12.1 Imports

```python
from typing import Annotated, Literal
from contextlib import asynccontextmanager

from fastapi import (FastAPI, APIRouter, Depends, HTTPException, Request, Response, status,
                     Path, Query, Body, Header, Cookie, Form, File, UploadFile,
                     BackgroundTasks, WebSocket, WebSocketDisconnect, Security)
from fastapi.responses import JSONResponse, StreamingResponse, FileResponse, RedirectResponse, PlainTextResponse
from fastapi.sse import EventSourceResponse, ServerSentEvent          # built-in SSE (recent FastAPI)
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm, APIKeyHeader, HTTPBearer
from fastapi.middleware.cors import CORSMiddleware
from fastapi.middleware.gzip import GZipMiddleware
from fastapi.exceptions import RequestValidationError, ResponseValidationError
from fastapi.concurrency import run_in_threadpool
from fastapi.testclient import TestClient

from pydantic import BaseModel, Field, ConfigDict, field_validator, model_validator, SecretStr, EmailStr
from pydantic_settings import BaseSettings, SettingsConfigDict

from sqlalchemy import select, func, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session, selectinload
```

## 12.2 Decorators and parameter sources

| Write | Means |
|---|---|
| `@app.get / post / put / patch / delete(path, status_code=..., tags=[...], deprecated=True)` | a path operation |
| `@router.get(...)` + `app.include_router(router, prefix="/v1")` | group routes |
| `@app.websocket("/ws")` | a WebSocket endpoint |
| `@app.middleware("http")` / `app.add_middleware(Cls, ...)` | middleware |
| `@app.exception_handler(ExcClass)` | custom error response |
| `FastAPI(lifespan=lifespan)` | startup/shutdown code (not `on_event`) |

| Parameter declaration | Read from |
|---|---|
| name in the path: `item_id: int` | path |
| simple type: `q: str \| None = None` | query string |
| `list[str]` with `Query()` | repeated query params `?tag=a&tag=b` |
| Pydantic model: `item: ItemIn` | JSON body |
| `Annotated[str, Body(embed=True)]` | one field of the JSON body |
| `Annotated[str, Header()]` / `Cookie()` | headers / cookies |
| `Annotated[str, Form()]`, `UploadFile` | form fields / files (needs python-multipart) |
| `Annotated[T, Depends(fn)]` | result of a dependency |
| `request: Request`, `response: Response`, `background: BackgroundTasks` | injected objects |

## 12.3 Status codes

| Code | Name | Use it when |
|---|---|---|
| 200 | OK | read / update succeeded |
| 201 | Created | POST created a resource (+ `Location`) |
| 202 | Accepted | work started in the background |
| 204 | No Content | DELETE succeeded, nothing to return |
| 301/308, 302/307 | Redirects | moved permanently / temporarily (307/308 keep the method) |
| 304 | Not Modified | `ETag` matched |
| 400 | Bad Request | malformed request (bad cursor, bad JSON) |
| 401 | Unauthorized | not authenticated (+ `WWW-Authenticate`) |
| 403 | Forbidden | authenticated, not allowed |
| 404 | Not Found | no such resource (also: not *yours*) |
| 405 | Method Not Allowed | wrong method for the path |
| 409 | Conflict | duplicate, version conflict |
| 413 | Content Too Large | upload too big |
| 415 | Unsupported Media Type | wrong content type |
| 422 | Unprocessable Content | validation failed (FastAPI default) |
| 429 | Too Many Requests | rate limited (+ `Retry-After`) |
| 500 | Internal Server Error | a bug |
| 502 / 503 / 504 | Bad Gateway / Unavailable / Gateway Timeout | upstream broken / not ready / upstream too slow |

> **Note:** Starlette 1.x renamed two constants to the current RFC names: use `status.HTTP_413_CONTENT_TOO_LARGE` and
> `status.HTTP_422_UNPROCESSABLE_CONTENT`. The old names (`..._REQUEST_ENTITY_TOO_LARGE`, `..._UNPROCESSABLE_ENTITY`) are kept for old code.

## 12.4 Patterns you will write again and again

```python
# Lifespan: create shared resources once
@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.http = httpx.AsyncClient(timeout=httpx.Timeout(10, connect=2))
    app.state.engine = create_engine(settings.database_url, pool_pre_ping=True)
    yield
    await app.state.http.aclose()
    app.state.engine.dispose()

# One DB session per request
def get_db(request: Request):
    with Session(request.app.state.engine) as session:
        yield session
DB = Annotated[Session, Depends(get_db)]

# Current user from a Bearer JWT
oauth2 = OAuth2PasswordBearer(tokenUrl="/v1/auth/login")
def current_user(token: Annotated[str, Depends(oauth2)], db: DB) -> User:
    try:
        payload = jwt.decode(token, SECRET, algorithms=["HS256"], options={"require": ["exp", "sub"]})
    except jwt.InvalidTokenError:
        raise HTTPException(401, "Invalid token", headers={"WWW-Authenticate": "Bearer"})
    return db.get(User, int(payload["sub"])) or ...

# Pagination parameters
def paging(limit: Annotated[int, Query(ge=1, le=100)] = 20, offset: Annotated[int, Query(ge=0)] = 0):
    return {"limit": limit, "offset": offset}

# PATCH
changes = patch.model_dump(exclude_unset=True)

# SSE streaming (recent FastAPI)
@app.post("/chat/stream", response_class=EventSourceResponse)
async def stream(body: ChatIn):
    async for token in llm.stream(body.prompt):
        yield ServerSentEvent(data={"text": token}, event="token")

# Tests
def test_x():
    app.dependency_overrides[get_db] = get_test_db
    with TestClient(app) as client:            # "with" runs the lifespan
        assert client.get("/healthz").status_code == 200
    app.dependency_overrides.clear()
```

## 12.5 Commands

```bash
fastapi dev main.py                                   # development, auto-reload
fastapi run main.py --workers 4                       # production
uvicorn app.main:app --host 0.0.0.0 --port 8000 --workers 4 --proxy-headers --forwarded-allow-ips 10.0.0.0/8
gunicorn app.main:app -k uvicorn_worker.UvicornWorker -w 4 -b 0.0.0.0:8000
pytest -q                                             # tests
alembic revision --autogenerate -m "add tasks" && alembic upgrade head
docker build -t api:1.0 . && docker run -p 8000:8000 --env-file .env api:1.0
docker compose up --build
locust -f locustfile.py --host http://localhost:8000
```

## 12.6 Facts that changed (checked in FastAPI 0.162, Starlette 1.10, Pydantic 2.14)

| Old (tutorials, interviews) | Current |
|---|---|
| `@app.on_event("startup")` | `FastAPI(lifespan=...)` (on_event is deprecated) |
| `param: str = Query(None)` | `param: Annotated[str \| None, Query()] = None` |
| Pydantic v1 `.dict()`, `@validator`, `orm_mode` | `.model_dump()`, `@field_validator`, `from_attributes=True` |
| `session.query(User)` | `session.scalars(select(User))` |
| `passlib` for hashing | `pwdlib` (Argon2) in the current FastAPI docs |
| `default_response_class=ORJSONResponse` for speed | declare return types; `ORJSONResponse`/`UJSONResponse` are deprecated in 0.162 |
| `sse-starlette` / hand-made `StreamingResponse` for SSE | `fastapi.sse.EventSourceResponse` + `ServerSentEvent` |
| `-k uvicorn.workers.UvicornWorker` | `-k uvicorn_worker.UvicornWorker` (separate package) |
| `opentelemetry-instrumentation-fastapi` only | also native `FastAPI(telemetry={...})`, active when an OpenTelemetry provider is configured |
| routes copied at `include_router` time; `app.routes` lists every route | included routers stay live; `app.routes` holds an included-router entry, so use `app.openapi()["paths"]` to list paths |
| `HTTP_422_UNPROCESSABLE_ENTITY` | `HTTP_422_UNPROCESSABLE_CONTENT` |
| JWT `sub` as an int | PyJWT 2.10+ requires `sub` to be a string |

In [108]:
# Smoke test: the cheat-sheet patterns really work together
from contextlib import asynccontextmanager
from typing import Annotated
from fastapi import Depends, FastAPI, Query, status
from fastapi.sse import EventSourceResponse, ServerSentEvent
from fastapi.testclient import TestClient
from pydantic import BaseModel

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.items = {1: "pen", 2: "book", 3: "bag"}
    yield

def paging(limit: Annotated[int, Query(ge=1, le=100)] = 2, offset: Annotated[int, Query(ge=0)] = 0):
    return {"limit": limit, "offset": offset}

class Item(BaseModel):
    id: int
    name: str

sheet_app = FastAPI(lifespan=lifespan)

@sheet_app.get("/items")
def list_items(page: Annotated[dict, Depends(paging)]) -> list[Item]:
    rows = sorted(sheet_app.state.items.items())[page["offset"]:page["offset"] + page["limit"]]
    return [Item(id=i, name=n) for i, n in rows]

@sheet_app.get("/ticks", response_class=EventSourceResponse)
async def ticks():
    for i in range(2):
        yield ServerSentEvent(data=i, event="tick")

with TestClient(sheet_app) as client:
    assert [i["name"] for i in client.get("/items").json()] == ["pen", "book"]
    assert client.get("/items", params={"limit": 0}).status_code == status.HTTP_422_UNPROCESSABLE_CONTENT
    assert client.get("/ticks").text.count("event: tick") == 2
print("Cheat sheet: all snippets work")

Cheat sheet: all snippets work


## 12.7 Production checklist

Go through this before a release, and use it to describe your projects in interviews ("we had readiness probes, structured logs with request IDs...").

**Correctness and API design**
- [ ] Input and output Pydantic models; no internal fields leak (`password_hash`, internal IDs)
- [ ] Correct status codes (201 + `Location`, 204, 404 vs 403, 409, 422, 429)
- [ ] Pagination with a capped `limit` on every list; allow-listed sort fields
- [ ] One error format (problem details) with a request ID
- [ ] Idempotency for payment-like POSTs; versioned routes (`/v1`)

**Security**
- [ ] Passwords hashed with Argon2/bcrypt; JWTs short-lived, `algorithms` pinned, `exp` required, secrets from the environment
- [ ] Ownership checks in every query (no IDOR); role checks in dependencies
- [ ] CORS with explicit origins; HTTPS at the proxy; HSTS
- [ ] Rate limits per key/user/IP (shared store if several workers); upload size and type limits
- [ ] No secrets in code, images or logs; dependencies scanned (`pip-audit`)

**Performance**
- [ ] No blocking calls in `async def`; CPU work off the event loop
- [ ] One engine / HTTP client per process; pool sizes fit the DB limit
- [ ] No N+1 queries; indexes on filtered and sorted columns
- [ ] Return types declared; responses small; caching with TTLs where it helps
- [ ] Load-tested; p95/p99 targets known

**Reliability and operations**
- [ ] `/healthz` (trivial) and `/readyz` (dependencies) with correct status codes
- [ ] Graceful shutdown: lifespan cleanup, graceful timeout, platform grace period
- [ ] Timeouts and retries (with backoff + jitter) on every external call; clear 503/504
- [ ] Background work that must not be lost goes to a durable queue
- [ ] Alembic migrations run as a separate, backward-compatible step

**Observability**
- [ ] JSON logs to stdout with request IDs; no personal data in logs
- [ ] RED metrics per route with bounded labels; dashboards
- [ ] Traces with OpenTelemetry (native `FastAPI(telemetry=...)` or the instrumentation package); error tracking (Sentry)
- [ ] Alerts on error rate and latency (SLO-based), not on every CPU spike

**Testing and delivery**
- [ ] Tests for happy paths, validation, auth (401/403/404) and edge cases; fresh state per test
- [ ] External APIs mocked (`MockTransport`); dependency overrides cleared
- [ ] CI runs lint, type checks and tests; Docker image is slim, non-root, with a health check
- [ ] Configuration via environment (12-factor); same image in every environment

In [109]:
# A quick self-assessment: mark what your current project has (edit the values)
checklist = {
    "typed input/output models": True,
    "consistent error format": True,
    "password hashing + short JWTs": True,
    "ownership checks": True,
    "health + readiness": True,
    "timeouts on external calls": True,
    "structured logs with request IDs": True,
    "metrics and alerts": False,
    "load test done": False,
    "tests with fresh state": True,
}
done = sum(checklist.values())
print(f"{done}/{len(checklist)} items done")          # 8/10 items done
for item, ok in checklist.items():
    if not ok:
        print("  next:", item)
# next: metrics and alerts
# next: load test done

8/10 items done
  next: metrics and alerts
  next: load test done


## 12.8 Clean up

The next cell deletes the `fastapi_lab_3/` folder that this notebook created (the real-server demo app and the SQLite files).
Run it when you are done. Running the whole notebook again creates everything from scratch.

In [110]:
# Remove the lab folder created by this notebook
import shutil
from pathlib import Path

lab = Path("fastapi_lab_3")
try:
    task_app.state.engine.dispose()        # release the SQLite file from Part 7 (needed on Windows)
except NameError:
    pass                                   # Part 7 was not run in this session
shutil.rmtree(lab, ignore_errors=True)
print("lab folder removed:", not lab.exists())     # lab folder removed: True

lab folder removed: True


## Summary of Notebook 3

* **Production:** `fastapi run`/Uvicorn workers behind a proxy, health and readiness, graceful shutdown, 12-factor config, Docker and compose.
* **Performance and observability:** keep the event loop free, pool connections, fix N+1, paginate, cache; JSON logs, metrics, traces, alerts.
* **Design:** resources and status codes, PATCH vs PUT, cursor pagination, idempotency keys, problem details, versioning, generated clients.
* **AI services:** models loaded in lifespan, batching, SSE token streaming with `fastapi.sse`, resilient LLM calls, RAG with citations.
* **Projects:** a Task Manager API and an LLM microservice, both tested, both ready to discuss in interviews.
* **Interviews:** about 100 spoken answers, 18 coding tasks, 10 debugging scenarios and a 45-minute mock interview.

You have finished all three FastAPI notebooks. Build one of the projects in a real folder, deploy it (even to a free tier), and talk about what you learned. Good luck with your interviews!